# 🧬 **pdb2reaction**: End-to-End Reaction-Path Elucidation from PDB Structures Using Machine-Learning Interatomic Potentials

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-0hmura/pdb2reaction/blob/main/examples/pdb2reaction_colab.ipynb)

Build enzymatic reaction paths from PDB/mmCIF or small-molecule XYZ/GJF
structures with one guided interface: **Setup → Launch GUI**, then
**Input → Workflow → Select in 3D → Validate → Run → Results**.

- Put multiple structures in reaction order (**reactant → intermediates → product**), then choose the workflow. Its card states the required inputs, relevant selectors/options, and expected outputs.
- A 3D click selects one exact chain/residue/insertion-code context; the residue is framed in amber and the atom stays visible as an orange sphere with a dark halo. Selecting a residue *name* intentionally selects every matching copy.
- The editable command is exactly what Run executes. Validate checks that command against the current input files; changing either invalidates the marker.
- Results explains how to interpret the selected workflow, keeps unavailable energies as unknown, and limits previews/downloads to the current run.

> Use a **GPU** runtime (Runtime ▸ Change runtime type ▸ GPU). The first **Setup** run installs pdb2reaction and one MLIP backend.
Backend defaults to **MACE** (no login). UMA is Hugging-Face-gated — switch in Setup after accepting its license. Repo: <https://github.com/t-0hmura/pdb2reaction>


In [ ]:
#@title  ⚙️ Setup — install pdb2reaction + a backend  (first run ~5 min) { display-mode: "form" }
#@markdown Only the **selected backend** is installed (MACE and UMA cannot coexist because their `e3nn` requirements conflict). To switch, restart the runtime and run Setup again.
#@markdown **MACE / ORB = no login; UMA = gated Hugging Face model.** ORB/MACE run fp64 and UMA runs fp32 by default. The release ref defaults to the notebook's matching version.
backend = "mace"  #@param ["mace", "uma", "orb"]
pdb2reaction_ref = "v0.4.12"  #@param {type:"string"}
#@markdown Tick **install_dft** to add the optional DFT extra (PySCF + GPU4PySCF), needed only by the `dft` subcommand — a few extra minutes.
install_dft = False  #@param {type:"boolean"}
import os, sys, subprocess, time
from pathlib import Path
# pysisyphus reads this optional user config during import.  An empty file keeps
# a fresh Colab runtime quiet while preserving the package defaults.
Path.home().joinpath('.pysisyphusrc').touch(exist_ok=True)
_setup_started = time.monotonic()
def _phase(number, label):
    print('[%d/5] %s …' % (number, label), flush=True)
def _phase_done(label):
    print('      ✓ %s (%.1f min elapsed)' % (label, (time.monotonic() - _setup_started) / 60), flush=True)
if globals().get('BACKEND') not in (None, backend):
    raise RuntimeError('Backend switch requested. Restart the Colab runtime first, then rerun Setup.')
try: _gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError: _gpu = ''   # nvidia-smi absent on CPU runtimes -> don't crash Setup
print(_gpu or '⚠️ No GPU detected — for real runs pick a GPU runtime (Runtime ▸ Change runtime type ▸ GPU). The GUI still loads.')
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
# Install the pinned release from PyPI — the same path a normal user takes.
# The wheel carries the package only, so the bundled example structures are
# fetched from the matching git tag on demand (see the GUI's Load example).
REPO_DIR = 'pdb2reaction-src'          # only present in a source/debug checkout
_phase(1, 'pdb2reaction %s from PyPI' % pdb2reaction_ref)
pip('pdb2reaction' + ('[dft]' if install_dft else '') + '==' + pdb2reaction_ref.lstrip('v'))
_phase_done('tool package installed')
_phase(2, '%s MLIP backend' % backend.upper())
if backend == 'mace':
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','fairchem-core'], check=False)
    pip('mace-torch>=0.3.8'); print('MACE installed (no login needed).')
elif backend == 'orb':
    # ORB coexists with the base install (the e3nn clash is MACE vs fairchem-core only).
    pip('orb-models'); print('ORB installed (no login needed; runs fp64 by default).')
else:
    # UMA is gated: accept the FAIR Chemistry License at https://huggingface.co/facebook/UMA
    from huggingface_hub import login, notebook_login
    if os.environ.get('HF_TOKEN'): login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    else: notebook_login()
    print('UMA installed (uses fairchem-core; needs HF license + login).')
_phase_done('backend installed')
_phase(3, 'notebook widgets and molecular viewer')
pip('py3Dmol','ipywidgets','matplotlib')
_phase_done('GUI dependencies installed')
# Plotly static-image export (scan maps, energy diagrams) needs a kaleido/choreographer
# combo with a working Chrome bridge. Unpinned, pip resolves kaleido 1.3 / choreographer 1.3,
# which raise 'Kaleido requires Google Chrome' on Colab and lose every PNG in the Results tab.
_phase(4, 'plot export and Chromium bridge')
pip('plotly==6.6.0','kaleido==1.2.0','choreographer==1.2.1')
_chrome = subprocess.run(['plotly_get_chrome','-y'], capture_output=True, text=True)
if _chrome.returncode != 0:
    _chrome = subprocess.run(['choreo_get_chrome'], capture_output=True, text=True)
print('PNG export:', 'chromium ready' if _chrome.returncode == 0 else
      'chromium unavailable - PNG export will be skipped (interactive HTML plots still work)')
_phase_done('plot export checked')
_phase(5, 'installed-version verification')
INSTALL_DFT = install_dft
BACKEND = backend; TOOL = 'pdb2reaction'
from importlib.metadata import version
installed_version = version('pdb2reaction')
if pdb2reaction_ref.startswith('v') and installed_version != pdb2reaction_ref[1:]:
    raise RuntimeError('Installed version %s does not match requested ref %s.' % (installed_version, pdb2reaction_ref))
print('pdb2reaction', installed_version, '| ref', pdb2reaction_ref, '| from PyPI')
_phase_done('version verified')
print('\n✅ Setup done in %.1f min. backend = %s — now run "Launch GUI".' %
      ((time.monotonic() - _setup_started) / 60, BACKEND))


In [ ]:
#@title 🖥️ Launch GUI  (run once, after Setup) { display-mode: "form" }
import os, glob, json, shlex, math, shutil, subprocess, time, zipfile, hashlib, importlib.util, signal, html, tempfile, base64, csv, io
from pathlib import Path
import ipywidgets as W
from IPython.display import display, clear_output, Image, HTML
import py3Dmol
import click
try: TOOL
except NameError: TOOL = 'pdb2reaction'
try: BACKEND
except NameError: BACKEND = 'mace'
try: REPO_DIR
except NameError: REPO_DIR = 'pdb2reaction-src'
IS_CLUSTER = True
_RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='%s-colab-' % TOOL.replace('_', '-')))

def _runtime_path(*parts):
    """Allocate notebook-owned files away from uploads and run outputs."""
    path = _RUNTIME_DIR.joinpath(*parts)
    path.parent.mkdir(parents=True, exist_ok=True)
    return str(path)

def _unique_path(path):
    """Return path, or a numbered sibling, without replacing an existing file."""
    candidate = Path(path)
    if not candidate.exists(): return str(candidate)
    for number in range(2, 10000):
        numbered = candidate.with_name('%s_%d%s' % (candidate.stem, number, candidate.suffix))
        if not numbered.exists(): return str(numbered)
    raise RuntimeError('Could not allocate a unique path for %s.' % candidate.name)
# Colab needs the custom widget manager for ipywidgets uploads to render.
try:
    from google.colab import output as _cwm; _cwm.enable_custom_widget_manager()
except Exception:
    _cwm = None            # custom-widget-manager is Colab-only; skip elsewhere

ACCENT = '#114b8a'
display(HTML("""<style>
.rxapp { width:100%; max-width:100%; box-sizing:border-box; padding:0 8px 8px; }
.rxtabs { margin:5px 0 8px !important; column-gap:6px !important; row-gap:5px !important; }
.rxtabs button { min-width:132px; }
.rxfold { margin:4px 0 2px; }
.rxapp .rxcard { padding:6px 8px !important; margin:0 !important; }
.rxworkspace { row-gap:8px !important; }
.rxapp .widget-vbox > .widget-html:empty { display:none; }
.rxapp .widget-html-content > div { margin:0; }
.rxinspector, .rxviewer { min-width:0; }
.rxfold > button { background:#f8fafc !important; border:1px solid #e2e8f0 !important;
  color:#334155 !important; font-weight:600 !important; text-align:left !important; }
.rxapp, .rxapp .widget-label, .rxapp .widget-html-content, .rxapp .widget-readout {
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif !important;
  color:#1f2937; }
.rxapp .widget-html-content code { background:#f1f5f9; padding:1px 5px; border-radius:5px;
  font-family:ui-monospace,SFMono-Regular,Menlo,monospace; font-size:12px; color:#334155;
  overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
.rxapp .widget-button { border-radius:9px !important; font-weight:500 !important;
  box-shadow:none !important; min-height:34px; }
/* Button labels are complete; hide Font Awesome glyphs that become tofu boxes
   when a Colab/static frontend does not provide the icon font. */
.rxapp .widget-button .fa, .rxapp .widget-upload .fa { display:none !important; }
.rxapp button.jupyter-button, .rxapp .widget-upload > button { min-height:34px !important; }
.rxapp .widget-dropdown select, .rxapp .widget-text input, .rxapp .widget-textarea textarea,
.rxapp input[type="number"] { border-radius:9px !important; border:1px solid #e2e8f0 !important; }
.rxapp .widget-button:focus-visible, .rxapp select:focus-visible,
.rxapp input:focus-visible, .rxapp textarea:focus-visible {
  outline:3px solid rgba(37,99,235,.28) !important; outline-offset:2px !important; }
.rxapp .widget-tab > .rxapp-tabs, .rxapp .p-TabBar-tab { font-weight:500; }
.rxcard { border:1px solid #e8ecf2; border-radius:11px; padding:7px 9px; background:#ffffff;
  box-shadow:0 1px 2px rgba(16,24,40,0.05); box-sizing:border-box; flex:0 0 auto; min-width:0; }
.rxapp .rxworkspace { align-items:stretch !important; column-gap:8px; row-gap:7px !important;
  margin:0 0 7px !important; flex-wrap:nowrap !important; }
/* the lower panel row reuses the workspace column tracks so the frame edges line up */
.rxworkspace > .rxviewer > .rxcard { margin:0; }
.rxviewer { flex:2 1 520px !important; min-width:300px; }
.rxinspector { flex:1 1 300px !important; min-width:260px; }
.rxinspector > .rxcard { width:100%; }
.rxviewer .jupyter-widgets-output-area, .rxviewer iframe { width:100% !important; max-width:100% !important; }
.rxresults { width:100% !important; max-width:100% !important; min-width:0 !important; }
.rxresults .jupyter-widgets-output-area, .rxresults iframe,
.rxresults canvas { width:100% !important; max-width:100% !important; box-sizing:border-box !important; }
.rxapp img, .rxapp svg, .rxapp iframe { max-width:100%; }
.rxcmd textarea { font-family:ui-monospace,SFMono-Regular,Menlo,monospace !important;
  font-size:13px !important; background:#0f172a !important; color:#7ee787 !important;
  border-radius:11px !important; line-height:1.5 !important; border:1px solid #1e293b !important;
  padding:9px 11px !important; }
.rxdrop { border:2px dashed #cbd5e1; border-radius:14px; padding:16px; background:#f8fafc;
  transition:all .15s ease; }
.rxdrop:hover { border-color:#94a3b8; background:#f1f5f9; }
.rxapp .rxfile { width:100%; flex-wrap:nowrap !important; align-items:center !important;
  border:1px solid #e2e8f0; border-radius:10px; padding:5px 7px; background:#fff;
  box-sizing:border-box; }
.rxapp .rxfile > .widget-html { flex:1 1 auto !important; min-width:0 !important; }
.rxapp .rxfile > .widget-hbox { flex:0 0 auto !important; flex-wrap:nowrap !important; }
.rxapp .rxfile button { min-width:34px !important; padding:0 8px !important; }
.rxhelp-row, .rxflagrow { position:relative; overflow:visible !important; }
.rxhelp-trigger { position:relative; display:inline-block; margin-left:5px; color:#64748b;
  cursor:help; line-height:1; outline:none; }
.rxhelp-body { visibility:hidden; opacity:0; pointer-events:none; position:absolute; z-index:10000;
  right:0; top:calc(100% + 7px); width:min(340px,72vw); padding:8px 10px; border-radius:9px;
  background:#0f172a; color:#f8fafc; font-size:12px; font-weight:400; line-height:1.35;
  white-space:normal; text-align:left; box-shadow:0 6px 20px rgba(15,23,42,.28); }
.rxhelp-row:hover .rxhelp-body, .rxhelp-row:focus-within .rxhelp-body,
.rxflagrow:hover .rxhelp-body, .rxflagrow:focus-within .rxhelp-body {
  visibility:visible; opacity:1; }
.rxflagrow .rxhelp-body { top:auto; bottom:calc(100% + 7px); }
.rxflagrow:first-child .rxhelp-body { top:calc(100% + 7px); bottom:auto; }
.rxapp .rxflagrow { flex-wrap:nowrap !important; align-items:center !important; }
.rxflagrow > .widget-html { flex:0 0 22px !important; overflow:visible !important; }
.rxchip button { border-radius:999px !important; font-size:12px !important; padding:1px 10px !important; }
.rxrun button { font-weight:600 !important; }
.rxapp .widget-hbox { flex-wrap:wrap !important; row-gap:6px; }
.rxapp hr { border:none; border-top:1px solid #eef0f4; }
@media (max-width: 820px) {
  .rxapp .rxworkspace { flex-wrap:wrap !important; }
  .rxviewer, .rxinspector { flex:1 1 100% !important; min-width:0 !important; }
}
@media (max-width: 600px) {
  .rxapp .widget-dropdown, .rxapp .widget-text,
  .rxapp .widget-select-multiple { max-width:100% !important; }
  .rxapp .lm-TabBar-tab, .rxapp .p-TabBar-tab { min-width:0 !important; }
  .rxapp .lm-TabBar-tabLabel, .rxapp .p-TabBar-tabLabel {
    overflow:hidden; text-overflow:ellipsis; }
}
@media (prefers-reduced-motion: reduce) {
  .rxapp * { transition:none !important; animation:none !important; }
}
</style>"""))

CLI = 'pdb2reaction'
from pdb2reaction.cli import cli as PRODUCT_CLI
from pdb2reaction.domain.residue_data import AMINO_ACIDS as _AA_DATA, ION as _ION_CHARGES, WATER_RES as _WATER_DATA
_INITIAL_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2',
                  'orb': 'orb_v3_conservative_omol'}.get(BACKEND, 'MACE-OMOL-0')
DFT_READY = (importlib.util.find_spec('pyscf') is not None and
             importlib.util.find_spec('gpu4pyscf') is not None)
DMF_READY = (importlib.util.find_spec('dmf') is not None and
             importlib.util.find_spec('cyipopt') is not None)
S = {'tool': TOOL, 'backend': BACKEND, 'model': _INITIAL_MODEL,
     'mode': None, 'inputs': [], 'subcmd': 'all',
     'advanced_overrides': {},
     'parm': None, 'model_pdb': None, 'center': [], 'center_ids': [], 'lcharge': {},
     '_pre_extract': None,
     'scan_atoms': [None, None], 'scan_target': 1.6, 'scan_preset': '', 'scan_stages': [], 'scan_axes': [],
     'freeze_buf': [None, None], 'freeze_pairs': [], 'freeze_atoms': [], 'charge': 0,
     'charge_explicit': False,
     'tsopt': False, 'thermo': False, 'out_dir': 'result',
     '_last_out_dir': None, '_last_subcmd': None, '_last_argv': [], '_last_files': [], '_last_manifest': {}, '_last_log': '',
     '_pdb_path': None, '_pdb_text': '', '_hetero': [], '_atoms': {}, '_atom_meta': [],
     '_last_pick': None, '_last_pick_message': '', '_last_pick_tone': 'ok', '_viewer_view': None,
     '_zoompick': False, '_view_input_index': 0, '_view_mapping_ok': True,
     '_primary_atom_signatures': [], '_primary_atom_meta': [],
     '_installed_backend': BACKEND,
     'show_water': False, 'surface': False, 'spin': False,
     'measure_atoms': [], 'rep': 'cartoon', 'color': 'spectrum', 'viewer_height': 320}

_AA = set(_AA_DATA)
_WATER = set(_WATER_DATA) | {'T3P','TIP3P','TIP4P','TIP5P','SPC','SPCE','OPC','OPC3'}

def _pdb_coordinate_rows(text):
    rows = []
    for line in text.splitlines():
        if line[0:6].strip() not in ('ATOM', 'HETATM'): continue
        try:
            rows.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
        except ValueError:
            raise ValueError('Viewer bridge contains an invalid PDB coordinate row.')
    return rows

# Link/cap hydrogens are appended by `extract` as HL atoms in residue LKH.
# They are neither a ligand (no -l charge) nor a sensible extraction center.
_CAP = {'LKH'}


_EXAMPLE_URL = 'https://raw.githubusercontent.com/t-0hmura/pdb2reaction/%s/examples/%s'


def _example_file(relpath):
    """Return a local path to a bundled example structure.

    A source checkout (debug builds) has them on disk; a PyPI install does not,
    because the wheel ships the package only. In that case fetch the file from
    the git tag matching the installed release, so example and code agree.
    """
    local = os.path.join(REPO_DIR, 'examples', relpath)
    if os.path.exists(local):
        return local
    dest = _runtime_path('examples', relpath)
    if not os.path.exists(dest):
        import urllib.request
        urllib.request.urlretrieve(_EXAMPLE_URL % (pdb2reaction_ref, relpath), dest)
    return dest

def parse_residues(text, metadata=None):
    allr, het = set(), set()
    if metadata:
        for atom in metadata:
            r = str(atom.get('resname') or '').strip().upper()
            if not r or r in _CAP: continue
            allr.add(r)
            if r not in _WATER and r not in _AA: het.add(r)
        return sorted(allr), sorted(het)
    for ln in text.splitlines():
        if ln[0:6].strip() not in ('ATOM', 'HETATM'): continue
        r = ln[17:20].strip()
        if r in _CAP: continue
        allr.add(r)
        if r not in _WATER and r not in _AA: het.add(r)
    return sorted(allr), sorted(het)

def parse_atoms(text, metadata=None):
    if metadata:
        coords = _pdb_coordinate_rows(text)
        if len(coords) != len(metadata):
            raise ValueError('Viewer PDB atom count does not match retained structure metadata.')
        out = {}
        for index, (meta, xyz) in enumerate(zip(metadata, coords)):
            meta['xyz'] = xyz
            meta['index'] = index
            key = (str(meta.get('chain') or ''), str(meta.get('resname') or ''),
                   str(meta.get('resseq')), str(meta.get('name') or ''))
            out[key] = xyz
        return out
    d = {}
    for ln in text.splitlines():
        if ln[0:6].strip() in ('ATOM', 'HETATM'):
            try:
                d[(ln[21:22].strip(), ln[17:20].strip(), ln[22:26].strip(), ln[12:16].strip())] = \
                    (float(ln[30:38]), float(ln[38:46]), float(ln[46:54]))
            except ValueError:
                continue
    return d

def _load_view_structure(path):
    """Return a safe viewer PDB plus original/auth atom metadata."""
    from pdb2reaction.core.utils import prepare_input_structure, load_pdb_atom_metadata
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        text = source.read_text(encoding='utf-8', errors='replace')
        metadata = load_pdb_atom_metadata(source)
        viewer_path = Path(_runtime_path('viewer_input.pdb'))
        viewer_path.write_text(text, encoding='utf-8')
    parse_atoms(text, metadata)       # attach stable 0-based index + coordinates
    return text, metadata, str(viewer_path)

def _aspec(x):
    chain = str(x.get('chain') or '').strip()
    if chain:
        return '%s:%s:%s:%s' % (chain, x['resn'], x['resi'], x['atom'])
    return '%s %s %s' % (x['resn'], x['resi'], x['atom'])

def scan_literals():
    """One literal per stage (multiple = staged); tuples within a stage = concerted."""
    if S['scan_stages']:
        return ['[' + ','.join('(\"%s\",\"%s\",%g)' % (_aspec(bd['a']), _aspec(bd['b']), bd['t'])
                               for bd in stage) + ']' for stage in S['scan_stages']]
    if S['scan_preset'] and not all(S['scan_atoms']): return [S['scan_preset']]
    a, b = S['scan_atoms']
    if a and b: return ['[(\"%s\",\"%s\",%g)]' % (_aspec(a), _aspec(b), S['scan_target'])]
    return []

def scan2d_literal():
    """scan2d/scan3d need ONE -s with per-axis quadruples: [(a,b,low,high),...]."""
    ax = S.get('scan_axes', [])
    if not ax: return ''
    return '[' + ','.join('(\"%s\",\"%s\",%g,%g)' % (_aspec(a['a']), _aspec(a['b']), a['lo'], a['hi'])
                          for a in ax) + ']'

def freeze_pair_lit():
    if not S['freeze_pairs']: return ''
    parts = []
    for p in S['freeze_pairs']:
        if p.get('t') is not None:
            parts.append('(\"%s\",\"%s\",%g)' % (_aspec(p['a']), _aspec(p['b']), p['t']))
        else:
            parts.append('(\"%s\",\"%s\")' % (_aspec(p['a']), _aspec(p['b'])))
    return '[' + ','.join(parts) + ']'

def scan_distance():
    a, b = S['scan_atoms']
    if a and b and a.get('xyz') and b.get('xyz'):
        return math.dist(a['xyz'], b['xyz'])
    return None

def _xyz(d):
    if not d: return None
    if d.get('xyz') is not None: return d['xyz']
    return S['_atoms'].get((str(d.get('chain') or ''), d['resn'], str(d['resi']), d['atom']))

def _angle(a, b, c):                                   # angle at vertex b (degrees)
    u = [a[i] - b[i] for i in range(3)]; v = [c[i] - b[i] for i in range(3)]
    du = math.sqrt(sum(x * x for x in u)); dv = math.sqrt(sum(x * x for x in v))
    if not du or not dv: return None
    cos = max(-1.0, min(1.0, sum(u[i] * v[i] for i in range(3)) / (du * dv)))
    return math.degrees(math.acos(cos))

def _dihedral(p0, p1, p2, p3):                          # signed dihedral (degrees)
    b0 = [p0[i] - p1[i] for i in range(3)]; b1 = [p2[i] - p1[i] for i in range(3)]
    b2 = [p3[i] - p2[i] for i in range(3)]
    def cross(a, b): return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]
    def dot(a, b): return sum(a[i] * b[i] for i in range(3))
    n = math.sqrt(dot(b1, b1)) or 1.0
    b1n = [x / n for x in b1]
    v = [b0[i] - dot(b0, b1n) * b1n[i] for i in range(3)]
    w = [b2[i] - dot(b2, b1n) * b1n[i] for i in range(3)]
    return math.degrees(math.atan2(dot(cross(b1n, v), w), dot(v, w)))

CLUSTER_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
            'path-opt', 'path-search', 'extract', 'fix-altloc', 'add-elem-info',
            'energy-diagram', 'bond-summary', 'trj2fig']
MLMM_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
             'path-opt', 'path-search', 'extract', 'define-layer', 'mm-parm', 'oniom-export',
             'oniom-import', 'fix-altloc', 'add-elem-info', 'energy-diagram', 'bond-summary', 'trj2fig']
SUBS = CLUSTER_SUBS if IS_CLUSTER else MLMM_SUBS
COMPUTE = {'all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
           'path-opt', 'path-search'}
MLIP_COMPUTE = COMPUTE - {'dft'}
# Subcommands a non-advanced user can run entirely from the GUI (no CLI typing):
BASIC_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
              'path-opt', 'path-search']
if not DFT_READY:
    SUBS = [name for name in SUBS if name != 'dft']
    BASIC_SUBS = [name for name in BASIC_SUBS if name != 'dft']
# ── SPEC: what every subcommand needs, produces and accepts ────────────────
# One declarative table drives every branch: the input requirement hint, which
# Select panels apply, which Workflow flags are shown, and what the Results tab
# looks for. Verified against the CLI itself (click introspection) and docs/.
#   n_in   : (min, max) input files; max None = unbounded
#   panels : Select-tab panels that apply ('center' | 'scan' | 'freeze')
#   out    : the deliverables worth pointing the user at
SPEC = {
    'all':      dict(n_in=(1, None), panels=('center', 'scan'),
                     req='R + P structures (MEP) · or 1 file + scan-lists · or 1 TS + TS-only mode',
                     out=('summary.log', 'mep.pdb', 'energy_diagram_MEP.png', 'segments/seg_NN/')),
    'extract':  dict(n_in=(1, None), panels=('center',),
                     req='a complex PDB/mmCIF + center residues (-c, required)',
                     out=('the extracted cluster model (-o)',)),
    'opt':      dict(n_in=(1, 1), panels=('freeze',), req='one structure to optimize',
                     out=('final_geometry.{xyz,pdb,cif}', 'optimization_trj.xyz')),
    'sp':       dict(n_in=(1, 1), panels=(), req='one structure (single-point E/F)',
                     out=('stdout energy', 'forces.npy', 'hessian.npy (--hess)')),
    'tsopt':    dict(n_in=(1, 1), panels=('freeze',), req='one TS-candidate structure',
                     out=('final_geometry.{xyz,pdb,cif}', 'vib/imag_*cm-1 (expect exactly one)')),
    'freq':     dict(n_in=(1, 1), panels=('freeze',), req='one optimized structure',
                     out=('frequencies_cm-1.txt', 'mode_*', 'thermoanalysis.yaml (--dump/--thermo)')),
    'irc':      dict(n_in=(1, 1), panels=('freeze',), req='one TS structure',
                     out=('*finished_irc_trj.xyz', 'forward / backward branches')),
    'dft':      dict(n_in=(1, 1), panels=(), req='one structure (DFT single-point)',
                     out=('result.yaml', 'result.json (--out-json)')),
    'scan':     dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='1 file + scan-lists (pick atoms A & B in the Select tab)',
                     out=('stage_XX/result.*', 'scan_trj.xyz')),
    'scan2d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 2 scan axes (pick bonds + low/high in the Select tab)',
                     out=('scan grid outputs',)),
    'scan3d':   dict(n_in=(1, 1), panels=('scan', 'freeze'),
                     req='one structure + 3 scan axes (pick bonds + low/high in the Select tab)',
                     out=('scan grid outputs',)),
    'path-opt': dict(n_in=(2, 2), panels=('freeze',), req='exactly two structures (segment endpoints; -i takes both)',
                     out=('final_geometries_trj.xyz', 'hei.xyz (highest-energy image)')),
    'path-search': dict(n_in=(2, None), panels=('freeze',), req='two or more structures (reactant … product)',
                        out=('per-segment path outputs',)),
}
SPEC.update({
    'fix-altloc': dict(n_in=(1, 1), panels=(), req='one PDB (or directory of PDB files) with alternate locations',
                       out=('cleaned structure (-o)',)),
    'add-elem-info': dict(n_in=(1, 1), panels=(), req='one PDB with missing/incorrect element columns',
                          out=('element-repaired PDB (-o)',)),
    'energy-diagram': dict(n_in=(0, None), panels=(), req='label/energy values supplied with repeated -i',
                           out=('energy diagram image (-o)',)),
    'bond-summary': dict(n_in=(2, None), panels=(), req='two or more structures to compare',
                         out=('bond table or JSON on stdout (--json)',)),
    'trj2fig': dict(n_in=(1, 1), panels=(), req='one XYZ trajectory with per-frame energies',
                    out=('energy profile image / HTML / CSV',)),
})
SUBREQ = {k: v['req'] for k, v in SPEC.items()}
GUIDE = {
    'all': ('End-to-end mechanism workflow',
            'Read summary.log first. Treat an MEP-band barrier as screening; a TS is validated only when the refined structure has exactly one meaningful imaginary mode and IRC reaches the intended endpoints.'),
    'extract': ('Build an active-site model', 'Inspect the extracted boundary, cap hydrogens, residue completeness, and inferred charge before computing.'),
    'opt': ('Relax a structure to a minimum', 'Require a converged terminal status; use freq to confirm zero meaningful imaginary modes.'),
    'sp': ('Evaluate one geometry', 'Read the reported electronic energy and requested force/Hessian arrays; this does not establish a minimum or TS.'),
    'tsopt': ('Refine a TS candidate', 'Require convergence and exactly one meaningful imaginary mode; then run IRC to establish connectivity.'),
    'freq': ('Classify a stationary point', 'A minimum has zero meaningful imaginary modes; a TS has exactly one whose displacement follows the reaction coordinate.'),
    'irc': ('Validate TS connectivity', 'Inspect both branches and confirm that their optimized endpoints match the intended reactant and product.'),
    'dft': ('Single-point DFT correction', 'Compare energies only for consistently prepared geometries/settings; this command does not optimize or validate a stationary point.'),
    'scan': ('Drive one or more distances', 'Use the profile to locate candidates; a scan maximum is not a certified TS until tsopt + freq + IRC validation.'),
    'scan2d': ('Map a two-coordinate surface', 'Inspect the interactive surface for valleys/ridges and use candidate geometries as inputs to path or TS refinement.'),
    'scan3d': ('Map a three-coordinate surface', 'Inspect slices/density views and refine promising candidates; grid extrema alone are not stationary-point proofs.'),
    'path-opt': ('Optimize one endpoint-to-endpoint path', 'Inspect path continuity and the highest-energy image; refine that image with tsopt and validate with freq + IRC.'),
    'path-search': ('Discover a multistep path', 'Inspect every segment and bridge. Refine and validate each chemically relevant highest-energy image separately.'),
    'fix-altloc': ('Resolve alternate conformers', 'Verify residue identity and atom count in the cleaned structure.'),
    'add-elem-info': ('Repair PDB element fields', 'Spot-check two-letter elements and ions before downstream charge/topology work.'),
    'energy-diagram': ('Render labeled energy levels', 'Verify units, reference state, labels, and method consistency.'),
    'bond-summary': ('Compare bond connectivity', 'Review formed/broken bonds against the intended atom mapping.'),
    'trj2fig': ('Plot trajectory energies', 'Confirm every frame has a real energy and that the chosen reference/unit matches the intended comparison.'),
    'define-layer': ('Assign ML/MM layers', 'Inspect B-factors/layer counts and the covalent boundary before calculation.'),
    'mm-parm': ('Build Amber topology', 'Read LEaP diagnostics and verify atom order, protonation, residue templates, charge, and parm7/PDB consistency.'),
    'oniom-export': ('Export an external ONIOM job', 'Verify charge/multiplicity, layer labels, link atoms, method route, and atom order before submission.'),
    'oniom-import': ('Import external ONIOM data', 'Verify atom order and layer assignment before reusing the generated coordinates/PDB.'),
}
# Keep the basic GUI chemically scoped: bare MACE "small/medium/large" aliases are
# materials checkpoints, not MACE-OMOL-0 variants. Advanced users can still type a
# different --backend-model in the editable command line.
MODELS = {'mace': ['MACE-OMOL-0'],
          'uma': ['uma-s-1p2', 'uma-s-1p1', 'uma-m-1p1'],
          'orb': ['orb_v3_conservative_omol']}
DEFAULT_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2', 'orb': 'orb_v3_conservative_omol'}
# Which subcommand accepts which surfaced flag (verified against the CLI, not
# guessed). Widget name -> accepting subcommands; a flag is hidden elsewhere.
FLAG_SUBS = {
    'adv_mep':     {'all', 'path-opt', 'path-search'},
    'adv_dmf':     {'all', 'path-opt', 'path-search'},
    'adv_refine':  {'all'},                                   # --refine-path: path-opt -> path-search
    'adv_flatten': {'all', 'opt', 'tsopt'},
    'adv_thresh':  {'all', 'opt', 'tsopt', 'scan', 'scan2d', 'scan3d', 'path-opt', 'path-search'},
    'adv_maxcyc':  {'all', 'opt', 'tsopt', 'irc', 'path-opt', 'path-search'},
    'adv_radius':  {'all', 'extract'},
    'adv_dft':     {'all'},
    'adv_prec':    MLIP_COMPUTE,
    'adv_det':     MLIP_COMPUTE,
    'adv_mult':    COMPUTE,
}
TOOL_CAPABILITIES = {                    # derived view kept for existing consumers
    'mep_mode': FLAG_SUBS['adv_mep'],
    'threshold': FLAG_SUBS['adv_thresh'],
}
OUT_JSON_SUBS = {'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan',
                 'scan2d', 'scan3d', 'path-opt', 'extract'}
AUTOFILL_UTILS = {'fix-altloc', 'add-elem-info', 'bond-summary', 'trj2fig'}
_PREP_SUBS = set(COMPUTE)

def _wv(name, default=None):
    """Safely read a widget's .value by global name (widget may not exist yet)."""
    w = globals().get(name)
    return getattr(w, 'value', default) if w is not None else default

def set_subcmd(v):
    S['subcmd'] = v
    dd = globals().get('dd_subcmd')
    if dd is not None and dd.value != v: dd.value = v

def _residue_id_selector(meta):
    """Return one selector in the CLI's chain/sequence-number dialect."""
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    chain = str(meta.get('chain') or '').strip()
    return ('%s:%s' % (chain, resi)) if chain else resi

def _rich_residue_selector(meta):
    chain = str(meta.get('chain') or '').strip()
    resname = str(meta.get('resname') or '').strip().upper()
    resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
    return '%s:%s:%s' % (chain, resname, resi) if chain else resi

def _center_cli_selectors(center_names=None, exact_ids=None, metadata=None):
    """Render -c without mixing the CLI's name and residue-ID grammars.

    Exact picks are kept as rich CHAIN:RESNAME:RESSEQ labels in the GUI.  Once
    one is present, broad residue-name selections are expanded through the
    primary input metadata so the emitted value uses one coherent CLI grammar.
    """
    names = [str(value).strip().upper() for value in
             (S.get('center', []) if center_names is None else center_names) if str(value).strip()]
    exact = [str(value).strip() for value in
             (S.get('center_ids', []) if exact_ids is None else exact_ids) if str(value).strip()]
    if not exact:
        return list(dict.fromkeys(names))
    metadata = S.get('_primary_atom_meta', []) if metadata is None else metadata
    residues = []; seen_residues = set()
    for meta in metadata:
        key = (str(meta.get('chain') or ''), str(meta.get('resname') or '').strip().upper(),
               str(meta.get('resseq')), str(meta.get('icode') or ''))
        if key not in seen_residues:
            seen_residues.add(key); residues.append((key, meta))
    selected = set(); matched_names = set()
    for key, meta in residues:
        if key[1] in names: selected.add(key); matched_names.add(key[1])
    missing = [name for name in names if name not in matched_names]
    if missing:
        raise ValueError('Return to the primary input to resolve center name(s): %s.' % ', '.join(missing))
    for value in exact:
        parts = value.split(':'); matches = []
        for key, meta in residues:
            resi = key[2] + key[3]
            if ((len(parts) == 3 and key[0] == parts[0] and key[1] == parts[1].upper() and resi == parts[2]) or
                (len(parts) == 2 and key[0] == parts[0] and resi == parts[1]) or
                (len(parts) == 1 and resi == parts[0])):
                matches.append(key)
        if not matches:
            raise ValueError('Exact center %s is not present in the primary input.' % value)
        selected.update(matches)
    chosen = [(key, meta) for key, meta in residues if key in selected]
    use_rich = bool(chosen) and all(key[0] for key, _ in chosen)
    # Both CLI selector grammars treat an omitted insertion code as “any code”.
    # Reject a token if that would silently include a residue outside the GUI set.
    for key, _ in chosen:
        if key[3]: continue
        broadened = {other for other, _meta in residues
                     if other[2] == key[2] and
                     ((other[0] == key[0] and other[1] == key[1]) if use_rich else
                      (other[0] == key[0] if key[0] else True))}
        if not broadened.issubset(selected):
            raise ValueError('The CLI cannot express exact center %s without also selecting an insertion-code sibling.' %
                             _rich_residue_selector(dict(chain=key[0], resname=key[1], resseq=key[2], icode=key[3])))
    selectors = [(_rich_residue_selector(meta) if use_rich else _residue_id_selector(meta))
                 for key, meta in chosen]
    return list(dict.fromkeys(selectors))

def _input_count_error(sub, count):
    bounds = SPEC.get(sub, {}).get('n_in')
    if bounds is None: return ''
    lo, hi = bounds
    if count >= lo and (hi is None or count <= hi): return ''
    limit = str(lo) if lo == hi else ('%d or more' % lo if hi is None else '%d-%d' % (lo, hi))
    return '%s needs %s input file(s).' % (sub, limit)

_ADV_OWNED_SUBS = {
    'backend_model': set(MLIP_COMPUTE), 'deterministic': set(MLIP_COMPUTE),
    'backend': set(MLIP_COMPUTE), 'charge': set(COMPUTE), 'charge_override': {'all'},
    'center_spec': {'all'}, 'substrate_pdb': {'extract'},
    'dist_freeze_raw': {'opt'}, 'do_dft': {'all'}, 'do_thermo': {'all'}, 'do_tsopt': {'all'},
    'dft_func_basis': {'all'}, 'func_basis': {'dft'},
    'dmf_backend': set(FLAG_SUBS['adv_dmf']), 'flatten': set(FLAG_SUBS['adv_flatten']),
    'freeze_atoms_text': {s for s, spec in SPEC.items() if 'freeze' in spec.get('panels', ())},
    'ligand_charge': set(COMPUTE) | {'extract'},
    'max_cycles': set(FLAG_SUBS['adv_maxcyc']), 'mep_mode': set(FLAG_SUBS['adv_mep']),
    'precision': set(MLIP_COMPUTE), 'radius': {'all', 'extract'},
    'refine_path': set(FLAG_SUBS['adv_refine']), 'spin': set(COMPUTE),
    'scan_lists_raw': {'all', 'scan'}, 'scan_list_raw': {'scan2d', 'scan3d'},
    'thresh': set(FLAG_SUBS['adv_thresh']),
}
_ADV_GENERATED = {'dry_run'}
_ADV_BLOCKED = {'help_advanced', 'tr_projection', 'verbose'}
_ADV_AUTO_IO_SUBS = set(COMPUTE) | {'extract'} | set(AUTOFILL_UTILS)
_ADV_IO_FLAGS = {'-i', '--input', '-o', '--out', '--output', '--out-dir', '--out-prefix'}
_PATH_ADVANCED = {'align','climb','conv_tol','endopt','fix_ends','max_cycles','max_nodes',
                  'max_step_size','preopt','preopt_max_cycles','reference_mode_path',
                  'relax_max_cycles','thresh'}
_POST_ENDPOINT_ADVANCED = {'irc_step_size','irc_never_stop','opt_mode_post',
                           'reject_uphill','thresh_post'}
_HESSIAN_STAGE_ADVANCED = {'hessian_calc_mode'}
_TSOPT_STAGE_ADVANCED = set() if IS_CLUSTER else {'skip_final_freq'}

def _advanced_command(sub):
    try: return PRODUCT_CLI.get_command(click.Context(PRODUCT_CLI), sub)
    except Exception: return None

def _advanced_hidden_options(sub):
    command = _advanced_command(sub)
    return tuple(getattr(command, '_advanced_hidden_options', ()) or ()) if command else ()

def _advanced_options(sub):
    """Every Click option, including the normally visible and hidden pages."""
    command = _advanced_command(sub)
    return tuple(param for param in (getattr(command, 'params', ()) or ())
                 if isinstance(param, click.Option)) if command else ()

def _advanced_status(sub, param):
    name = param.name
    if name in _ADV_BLOCKED: return 'blocked'
    if name == 'out_json' and sub in OUT_JSON_SUBS: return 'generated'
    if name in _ADV_GENERATED: return 'generated'
    if sub in _ADV_AUTO_IO_SUBS and set(param.opts + param.secondary_opts) & _ADV_IO_FLAGS: return 'owned'
    if sub in _ADV_OWNED_SUBS.get(name, set()): return 'owned'
    return 'rendered'

def _advanced_semantic_applicable(sub, name):
    if sub != 'all': return True
    mode = _wv('all_mode', 'mep')
    ts_enabled = mode == 'tsonly' or bool(_wv('w_ts', S.get('tsopt')))
    thermo_enabled = bool(_wv('w_th', S.get('thermo')))
    dft_enabled = bool(_wv('adv_dft', False))
    # Cluster workflows run IRC/endpoint optimization only with --tsopt;
    # ML/MM workflows share those post stages across TS, thermo, and DFT depth.
    post_enabled = ts_enabled if IS_CLUSTER else (ts_enabled or thermo_enabled or dft_enabled)
    if name.startswith('scan_'): return mode == 'scan'
    if mode == 'tsonly' and name in _PATH_ADVANCED: return False
    if name in _POST_ENDPOINT_ADVANCED: return post_enabled
    if name in _HESSIAN_STAGE_ADVANCED: return ts_enabled or thermo_enabled
    if name.startswith('tsopt_') or name in _TSOPT_STAGE_ADVANCED: return ts_enabled
    if name.startswith('freq_'): return thermo_enabled
    if name.startswith('dft_'): return dft_enabled
    return True

def _named_flag_applies(widget_name, sub):
    if sub not in FLAG_SUBS.get(widget_name, set()): return False
    if sub != 'all': return True
    mode = _wv('all_mode', 'mep')
    if widget_name in {'adv_mep','adv_dmf','adv_refine','adv_thresh','adv_maxcyc'}:
        return mode != 'tsonly'
    if widget_name == 'adv_flatten':
        return mode == 'tsonly' or bool(_wv('w_ts', S.get('tsopt')))
    return True

def _advanced_flag(param):
    return next((opt for opt in param.opts if opt.startswith('--')), param.opts[0])

def _advanced_argv(sub):
    saved = S.get('advanced_overrides', {}).get(sub, {})
    if not saved: return []
    command = _advanced_command(sub)
    try:
        bool_values, bool_toggles, negative_aliases, bool_single = PRODUCT_CLI._resolve_bool_options(
            click.Context(PRODUCT_CLI), sub)
    except Exception:
        bool_values, bool_toggles, negative_aliases, bool_single = set(), set(), {}, set()
    argv = []
    for param in _advanced_options(sub):
        if (_advanced_status(sub, param) != 'rendered' or param.name not in saved or
                not _advanced_semantic_applicable(sub, param.name)):
            continue
        value = saved[param.name]; flag = _advanced_flag(param)
        is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
        if is_bool:
            if value is True:
                argv += [flag] if flag in (set(bool_toggles) | set(bool_single)) or param.is_bool_flag else [flag, 'true']
            elif value is False:
                negative = (next((opt for opt in param.secondary_opts if opt.startswith('--')), None)
                            or negative_aliases.get(flag))
                argv += [negative] if negative else [flag, 'false']
        elif param.multiple:
            values = shlex.split(value) if isinstance(value, str) else list(value)
            for item in values: argv += [flag, str(item)]
        elif value not in (None, ''):
            argv += [flag, str(value)]
    return argv

def _advanced_coverage(sub):
    return {param.name: _advanced_status(sub, param) for param in _advanced_options(sub)}

def build_cmd():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    sub = _wv('dd_subcmd', S['subcmd']) or 'all'
    S['subcmd'] = sub
    count_error = _input_count_error(sub, len(S.get('inputs', [])))
    if count_error:
        raise ValueError(count_error + (' Put structures in reaction order.' if sub in COMPUTE else ''))
    if sub in COMPUTE or sub == 'extract' or sub in AUTOFILL_UTILS:
        missing = [path for path in S['inputs'] if not os.path.isfile(path)]
        if missing:
            raise ValueError('Re-upload missing input file(s): %s.' % ', '.join(os.path.basename(p) for p in missing))
        if S.get('model_pdb') and not os.path.isfile(S['model_pdb']):
            raise ValueError('Re-upload the missing prepared model: %s.' % os.path.basename(S['model_pdb']))
    bk = S.get('backend', BACKEND)
    if sub in MLIP_COMPUTE and bk != S.get('_installed_backend'):
        raise ValueError('Backend %s is not installed in this runtime; rerun Setup with backend=%s.' % (bk, bk))
    if sub == 'path-search' and len(S['inputs']) < 2:
        raise ValueError('path-search needs two or more structures in reaction order.')
    if sub == 'path-opt' and len(S['inputs']) != 2:
        raise ValueError('path-opt needs exactly two endpoint structures.')
    if sub == 'all':
        all_kind = _wv('all_mode', 'mep')
        if all_kind == 'mep' and len(S['inputs']) < 2:
            raise ValueError('all MEP mode needs two or more structures.')
        if all_kind == 'scan' and (len(S['inputs']) != 1 or not scan_literals()):
            raise ValueError('all scan mode needs one structure and a picked scan bond.')
        if all_kind == 'tsonly' and len(S['inputs']) != 1:
            raise ValueError('all TS-only mode needs exactly one TS candidate.')
    if sub == 'scan' and not scan_literals():
        raise ValueError('scan needs a picked atom pair and target distance.')
    if sub in ('scan2d', 'scan3d'):
        expected = 2 if sub == 'scan2d' else 3
        if len(S.get('scan_axes', [])) != expected:
            raise ValueError('%s needs exactly %d scan axes.' % (sub, expected))
    if not IS_CLUSTER and sub in COMPUTE and not S.get('parm'):
        raise ValueError('ML/MM compute commands require a matching Amber parm7 upload in Colab.')
    if (sub == 'dft' or (sub == 'all' and _wv('adv_dft', False))) and not DFT_READY:
        raise ValueError('DFT support is not installed; rerun Setup with install_dft enabled.')
    center_all = _center_cli_selectors()
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    gjf_supplies_charge = bool(S['inputs']) and all(Path(path).suffix.lower() == '.gjf' for path in S['inputs'])
    needs_system_charge = (sub in COMPUTE and not lc and
                           (sub != 'all' or S['mode'] != 'pdb' or not center_all) and
                           not gjf_supplies_charge)
    if needs_system_charge and not S.get('charge_explicit'):
        raise ValueError('Verify the system charge (-q) on the Workflow page before running this command.')
    extra = _advanced_argv(sub)
    if sub not in COMPUTE and sub != 'extract':
        cmd = [CLI, sub]
        inputs = list(S.get('inputs', []))
        if sub == 'fix-altloc':
            src = Path(inputs[0]); cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_altloc_fixed.pdb'))]
        elif sub == 'add-elem-info':
            src = Path(inputs[0]); cmd += ['-i', str(src)]
            # p2r's --overwrite is an explicit in-place request. Keep the safe
            # generated output unless the user selected that advanced flag.
            if not bool(S.get('advanced_overrides', {}).get(sub, {}).get('overwrite')):
                cmd += ['-o', str(src.with_name(src.stem + '_elements.pdb'))]
        elif sub == 'bond-summary':
            for path in inputs: cmd += ['-i', path]
        elif sub == 'trj2fig':
            src = Path(inputs[0])
            if src.suffix.lower() != '.xyz': raise ValueError('trj2fig needs an uploaded XYZ trajectory.')
            cmd += ['-i', str(src), '-o', str(src.with_name(src.stem + '_energy.png'))]
        return cmd + extra
    cmd = [CLI, sub, '-i', *S['inputs']]
    if sub in MLIP_COMPUTE: cmd += ['-b', bk]
    mdl = S.get('model')
    if sub in MLIP_COMPUTE and mdl and mdl != DEFAULT_MODEL.get(bk):
        cmd += ['--backend-model', mdl]
    output_arg = S['out_dir']
    if sub == 'extract' and Path(output_arg).suffix.lower() not in ('.pdb', '.ent', '.cif', '.mmcif'):
        output_arg = os.path.join(output_arg, 'cluster.pdb')
    cmd += ['-o', output_arg]
    if sub in ('all', 'extract') and center_all: cmd += ['-c', ','.join(center_all)]
    if lc: cmd += ['-l', lc]
    if not IS_CLUSTER and sub in COMPUTE: cmd += ['--parm', S['parm']]
    if sub in OUT_JSON_SUBS: cmd += ['--out-json']
    if sub in ('scan2d', 'scan3d'):
        if scan2d_literal(): cmd += ['-s', scan2d_literal()]  # one -s, per-axis (i,j,low,high) quadruples
    elif (sub == 'scan') or (sub == 'all' and all_kind == 'scan'):
        for lit in scan_literals(): cmd += ['-s', lit]        # multiple -s = staged stages
    if (sub in COMPUTE and not lc and
            (sub != 'all' or S['mode'] != 'pdb' or not center_all) and
            (not gjf_supplies_charge or S.get('charge_explicit'))):
        cmd += ['-q', str(S['charge'])]
    m = _wv('adv_mult', 1)
    if sub in COMPUTE and m not in (None, 1): cmd += ['-m', str(int(m))]
    precision = _wv('adv_prec', 'auto')
    if sub in MLIP_COMPUTE and precision in ('fp32', 'fp64'): cmd += ['--precision', precision]
    if sub in MLIP_COMPUTE and _wv('adv_det', False): cmd += ['--deterministic']
    r = _wv('adv_radius', 0.0)
    # -r/--radius is an extraction parameter: only all and extract accept it.
    if sub in ('all', 'extract') and r and r > 0: cmd += ['-r', str(r)]
    th = _wv('adv_thresh', '(default)')
    if _named_flag_applies('adv_thresh', sub) and th and th != '(default)': cmd += ['--thresh', th]
    mep = _wv('adv_mep', '(default)')
    if _named_flag_applies('adv_mep', sub) and mep and mep != '(default)':
        if mep == 'dmf' and not DMF_READY:
            raise ValueError('DMF needs pydmf + cyipopt, which are not installed by base Setup.')
        cmd += ['--mep-mode', mep]
        dmf_backend = _wv('adv_dmf', '(default)')
        if mep == 'dmf' and dmf_backend != '(default)':
            cmd += ['--dmf-backend', dmf_backend]
    if _named_flag_applies('adv_flatten', sub) and _wv('adv_flatten', False): cmd += ['--flatten']
    if _named_flag_applies('adv_refine', sub) and _wv('adv_refine', False): cmd += ['--refine-path']
    mc = _wv('adv_maxcyc', 0)
    if _named_flag_applies('adv_maxcyc', sub) and mc and int(mc) > 0: cmd += ['--max-cycles', str(int(mc))]
    if sub == 'opt' and freeze_pair_lit(): cmd += ['--dist-freeze', freeze_pair_lit()]
    if 'freeze' in SPEC.get(sub, {}).get('panels', ()) and S['freeze_atoms']:
        cmd += ['--freeze-atoms', ','.join(str(i) for i in S['freeze_atoms'])]
    if sub == 'all':
        if S['tsopt']: cmd.append('--tsopt')
        if S['thermo']: cmd.append('--thermo')
        if _wv('adv_dft', False):
            cmd.append('--dft')
            fb = _wv('adv_dftfb', '')
            if fb: cmd += ['--dft-func-basis', fb]
    elif sub == 'dft':
        fb = _wv('adv_dftfb', '')
        if fb: cmd += ['--func-basis', fb]
    cmd += extra
    return cmd

# -------- PyMOL-style editable command line + readiness chip ------------------
ready_chip = W.HTML()
toast = W.HTML()
run_status = W.HTML(value='<div role="status" aria-live="polite" aria-atomic="true" style="min-height:24px"></div>')
_RUN_STATE = {'validated_fingerprint': None, 'validation_log': '', 'kind': ''}
_RUN_TONES = {'info': '#2563eb', 'ok': '#1f7a3d', 'warn': '#7c5c00', 'error': '#a00', 'muted': '#64748b'}

def _set_run_status(text='', tone='muted', kind=''):
    """Update one consistently announced status region."""
    _RUN_STATE['kind'] = kind
    badge = (('<span style="background:%s;color:#fff;padding:2px 9px;border-radius:11px;">%s</span>' %
              (_RUN_TONES.get(tone, _RUN_TONES['muted']), html.escape(str(text)))) if text else '')
    run_status.value = ('<div role="status" aria-live="polite" aria-atomic="true" '
                        'style="min-height:24px">%s</div>' % badge)

def _command_fingerprint(text):
    return hashlib.sha256(str(text or '').encode('utf-8')).hexdigest()

def _invalidate_last_run(reason='Inputs changed; run again to populate Results.'):
    """Detach every Results widget from an obsolete compute identity."""
    had_run = bool(S.get('_last_manifest') or S.get('_last_files') or S.get('_last_log'))
    S.update(_last_out_dir=None, _last_subcmd=None, _last_argv=[], _last_files=[],
             _last_manifest={}, _last_log='', _results_notice=(reason if had_run else ''))
    _RUN_STATE['validated_fingerprint'] = None
    _RUN_STATE['validation_log'] = ''
    if had_run: _set_run_status('inputs changed · validate and run again', 'warn', 'changed')
    guard = globals().get('_result_pick_guard')
    if guard is not None: guard['active'] = True
    try:
        for name in ('artifact_choice', 'traj_choice'):
            widget = globals().get(name)
            if widget is not None:
                widget.options = []; widget.disabled = True
        slider = globals().get('frame_slider')
        if slider is not None:
            slider.max = 0; slider.value = 0; slider.disabled = True
    finally:
        if guard is not None: guard['active'] = False
    traj = globals().get('_TRAJ')
    if traj is not None: traj.update(frames=[], energies=[], path=None, semantics={})
    for name in ('res_out', 'artifact_out', 'traj_out', 'plot_out'):
        output = globals().get(name)
        if output is not None:
            with output: clear_output()
    for name in ('artifact_box', 'trajectory_box'):
        box = globals().get(name)
        if box is not None: box.layout.display = 'none'
    empty = globals().get('results_empty')
    if empty is not None:
        empty.layout.display = ''
        empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;'
                       'border-radius:10px;">%s</div>' % html.escape(reason))
    context = globals().get('result_context')
    if context is not None: context.value = ''
    for name in ('traj_label', 'frame_state', 'trajectory_intro'):
        widget = globals().get(name)
        if widget is not None: widget.value = ''
    button = globals().get('dl_btn')
    if button is not None: button.disabled = True
    button = globals().get('res_btn')
    if button is not None: button.description = 'Show results'
cmd_box = W.Textarea(value='', placeholder='the command builds here — edit it freely, then Run (like a PyMOL command line)',
                     layout=W.Layout(width='99%', height='48px'))
cmd_box.add_class('rxcmd')
_OK = '<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">● ready to run</span>'
_NO = '<span role="status" style="background:#64748b;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">○ %s</span>'
_MANUAL = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ add utility arguments below</span>'
_MANUAL_EDIT = '<span role="status" style="background:#7c5c00;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">◆ manual command · GUI changes are detached</span>'
_auto = {'on': True, 'guard': False}
def _set_cmd(txt):
    _auto['guard'] = True; cmd_box.value = txt; _auto['guard'] = False
def refresh(_=None):
    try:
        built = build_cmd()
        if _auto['on']: _set_cmd(' '.join(shlex.quote(c) for c in built))
        is_utility = len(built) > 1 and built[1] not in COMPUTE and built[1] != 'extract'
        manual_utility = is_utility and built[1] not in AUTOFILL_UTILS
        ready_chip.value = (_MANUAL if manual_utility and _auto['on'] else
                            (_OK if _auto['on'] else _MANUAL_EDIT))
    except Exception as e:
        if _auto['on']: _set_cmd('# ' + str(e))
        ready_chip.value = _NO % html.escape(str(e))
    _rs = globals().get('_render_summary')
    if _rs is not None: _rs()
    _rc = globals().get('_render_chips')
    if _rc is not None: _rc()
    _ro = globals().get('_render_output_note')
    if _ro is not None: _ro()
    valid = _RUN_STATE.get('validated_fingerprint')
    valid_command = valid[0] if isinstance(valid, tuple) else valid
    if valid and valid_command != _command_fingerprint(cmd_box.value):
        _RUN_STATE['validated_fingerprint'] = None
        _RUN_STATE['validation_log'] = ''
        _set_run_status('command changed · validate again', 'warn', 'changed')
def _on_cmd_edit(_):
    if not _auto['guard']:
        _auto['on'] = False   # user took manual control of the line
        if _RUN_STATE.get('validated_fingerprint'):
            _RUN_STATE['validated_fingerprint'] = None
            _RUN_STATE['validation_log'] = ''
            _set_run_status('command changed · validate again', 'warn', 'changed')
cmd_box.observe(_on_cmd_edit, names='value')

# ============================================================== INPUT tab
input_msg = W.HTML()
def _goto(i): _tab_go(i)

def _clear_structure_bound_state():
    """Clear every atom/residue selection before the input identity can change."""
    global center_widget, charge_rows
    _invalidate_last_run('Input identity changed; validate and run the new system.')
    center_widget = None; charge_rows = None
    S.update(center=[], center_ids=[], lcharge={}, model_pdb=None, _pre_extract=None,
             scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             freeze_buf=[None, None], freeze_pairs=[], freeze_atoms=[], measure_atoms=[],
             charge=0, charge_explicit=False, _pdb_path=None, _pdb_text='',
             _hetero=[], _atoms={}, _atom_meta=[], _last_out_dir=None,
             _last_subcmd=None, _last_argv=[], _last_files=[], _last_manifest={}, _last_log='',
             _last_pick=None, _last_pick_message='', _last_pick_tone='ok', _viewer_view=None,
             _zoompick=False, _view_input_index=0, _view_mapping_ok=True,
             _primary_atom_signatures=[], _primary_atom_meta=[])
    charge_widget = globals().get('w_q')
    if charge_widget is not None: charge_widget.value = 0
    charge_ok = globals().get('w_charge_ok')
    if charge_ok is not None: charge_ok.value = False
    S['charge_explicit'] = False
    revert = globals().get('b_revert')
    if revert is not None: revert.layout.display = 'none'
    for name, message in (('center_panel', 'Load a primary PDB/mmCIF to choose center residues.'),
                          ('charge_panel', 'Ligand charges appear after the primary structure loads.')):
        panel = globals().get(name)
        if panel is not None: panel.children = [W.HTML('<small>%s</small>' % message)]

def load_pdb(paths, parm=None, center=None, lcharge=None, scan_preset='', mode='pdb',
             append=False, keep_subcmd=False):
    previous_subcmd = S.get('subcmd', 'all')
    target_subcmd = previous_subcmd if keep_subcmd else 'all'
    paths = [p for p in paths if p]
    if append:
        _invalidate_last_run('Input files changed; validate and run the updated system.')
        paths = list(S.get('inputs', [])) + [p for p in paths if p not in S.get('inputs', [])]
        S.update(inputs=paths, parm=(parm or S.get('parm')), mode=mode, subcmd=target_subcmd)
        if center is not None: S['center'] = list(center)
        if lcharge is not None: S['lcharge'] = dict(lcharge)
        if scan_preset: S['scan_preset'] = scan_preset
    else:
        _clear_structure_bound_state()
        S.update(inputs=paths, parm=parm, mode=mode, subcmd=target_subcmd,
                 center=list(center or []), center_ids=[], lcharge=dict(lcharge or {}),
                 scan_atoms=[None, None], scan_preset=scan_preset)
    set_subcmd(target_subcmd)
    am = globals().get('all_mode')
    if am is not None and not keep_subcmd:
        am.value = 'mep' if len(paths) >= 2 else ('scan' if scan_preset else am.value)
    _riq = globals().get('_render_input_queue')
    if _riq is not None: _riq()
    build_selection()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()   # stay on Input: more files may still be added

_acc = '.pdb,.ent,.cif,.mmcif,.xyz,.gjf' if IS_CLUSTER else '.pdb,.ent,.cif,.mmcif,.parm7'
upl = W.FileUpload(accept=_acc, multiple=True, description='Drop files', icon='upload',
                   layout=W.Layout(width='260px'))
def _safe_upload_path(name):
    """Choose a local basename without replacing an existing runtime file."""
    clean = os.path.basename(str(name)) or 'upload.dat'
    if not os.path.exists(clean): return clean
    path = Path(clean)
    for number in range(2, 10000):
        candidate = '%s_%d%s' % (path.stem, number, path.suffix)
        if not os.path.exists(candidate): return candidate
    raise RuntimeError('Could not allocate a unique name for %s.' % clean)

def _save_upload(name, content):
    target = _safe_upload_path(name)
    with open(target, 'wb') as fh: fh.write(bytes(content))
    return target

def _ingest_saved_files(loaded, source='upload'):
    """Attach one saved browser/drop batch without replacing earlier batches."""
    loaded = [str(path) for path in loaded if path]
    structures = [p for p in loaded if p.lower().endswith(('.pdb', '.ent', '.cif', '.mmcif'))]
    parm_files = [p for p in loaded if p.lower().endswith('.parm7')]
    smalls = [p for p in loaded if p.lower().endswith(('.xyz', '.gjf'))]
    known = set(structures + parm_files + smalls)
    unsupported = [p for p in loaded if p not in known]
    if unsupported or len(parm_files) > 1 or (structures and smalls) or (parm_files and IS_CLUSTER):
        reason = ('unsupported file: %s' % ', '.join(os.path.basename(p) for p in unsupported)
                  if unsupported else 'attach one file type at a time')
        input_msg.value = '<div role="alert" style="color:#991b1b">%s; existing files were kept.</div>' % html.escape(reason)
        return False
    parm = parm_files[0] if parm_files else None
    if structures:
        if S.get('inputs') and S.get('mode') not in (None, 'pdb'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current small-molecule files before adding structures.</div>'
            return False
        adding = bool(S.get('inputs'))
        load_pdb(structures, parm=parm, append=adding)
        input_msg.value = '✅ %s <b>%s</b> (%s)' % (
            'appended' if adding else 'loaded', html.escape(', '.join(structures)), html.escape(source))
    elif smalls:
        if S.get('inputs') and S.get('mode') not in (None, 'small'):
            input_msg.value = '<div role="alert" style="color:#991b1b">Remove the current structures before adding small molecules.</div>'
            return False
        adding = bool(S.get('inputs'))
        if adding:
            merged = list(S.get('inputs', [])) + [p for p in smalls if p not in S.get('inputs', [])]
            _invalidate_last_run('Input files changed; validate and run the updated system.')
        else:
            merged = smalls; _clear_structure_bound_state()
        S.update(mode='small', inputs=merged, charge_explicit=False)
        set_subcmd('opt' if len(merged) == 1 else 'all')
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        input_msg.value = '✅ %s <b>%s</b> (%s) — set total charge on Workflow.' % (
            'appended' if adding else 'loaded', html.escape(', '.join(smalls)), html.escape(source))
        _render_input_queue(); refresh()
    elif parm:
        _invalidate_last_run('Amber topology changed; validate and run again.')
        S['parm'] = parm; _render_input_queue(); refresh()
        input_msg.value = '✅ topology <b>%s</b> attached (%s)' % (html.escape(parm), html.escape(source))
    return bool(structures or smalls or parm)

_upload_guard = {'active': False}
def _on_upload(change):
    if _upload_guard['active']: return
    items = change.get('new', upl.value)
    if not items: return
    pairs = list(items.items()) if isinstance(items, dict) else [(f['name'], f) for f in items]
    renamed = []; loaded = []
    try:
        for name, meta in pairs:
            target = _save_upload(name, meta['content']); loaded.append(target)
            if target != os.path.basename(name): renamed.append('%s → %s' % (name, target))
        _ingest_saved_files(loaded, 'file picker')
        if renamed:
            input_msg.value += '<br><small>Existing files kept; uploaded as %s</small>' % html.escape(', '.join(renamed))
    finally:
        _upload_guard['active'] = True
        try: upl.value = ()
        finally: _upload_guard['active'] = False
upl.observe(_on_upload, names='value')

_EX = (['BezA methyltransferase - cluster MEP (R->P)', 'HCN -> HNC - small molecule (fast)']
       if IS_CLUSTER else ['methyltransferase complex - scan mode'])
ex_choice = W.Dropdown(options=_EX, value=_EX[0], description='example',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='400px', max_width='100%'))
ex_btn = W.Button(description='Load example', icon='flask', layout=W.Layout(width='150px'))
def _load_example(_):
    sel = ex_choice.value
    if IS_CLUSTER and sel.startswith('HCN'):
        hcn_r = _runtime_path('examples', 'hcn_r.xyz'); hcn_p = _runtime_path('examples', 'hcn_p.xyz')
        with open(hcn_r, 'w') as fh: fh.write('3\nHCN\nC 0 0 0\nN 0 0 1.16\nH 0 0 -1.07\n')
        with open(hcn_p, 'w') as fh: fh.write('3\nHNC\nC 0 0 0\nN 0 0 1.17\nH 0 0.98 1.60\n')
        _clear_structure_bound_state()
        S.update(mode='small', inputs=[hcn_r, hcn_p], parm=None, center=[], center_ids=[],
                 lcharge={}, scan_atoms=[None, None], scan_stages=[], scan_preset='', charge=0,
                 charge_explicit=True)
        set_subcmd('all')
        _scc = globals().get('_sync_capability_controls')
        if _scc is not None: _scc()
        wq = globals().get('w_q')
        if wq is not None: wq.value = 0
        input_msg.value = 'HCN -> HNC small-molecule MEP (gas-phase, -q 0) - go to Run.'
        refresh(); return
    if IS_CLUSTER:
        try:
            _r, _p = _example_file('1.R.pdb'), _example_file('3.P.pdb')
        except Exception as exc:
            input_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        input_msg.value = '⭐ BezA methyltransferase (R→P MEP).'
        load_pdb([_r, _p], center=['SAM', 'GPP', 'MG'], lcharge={'SAM': 1, 'GPP': -3})
    else:
        try:
            _c = _example_file('methyltransferase/complex.pdb')
            _pm = _example_file('methyltransferase/complex.parm7')
        except Exception as exc:
            input_msg.value = '⚠️ could not fetch the example: %s' % exc; return
        S['scan_target'] = 1.3
        input_msg.value = '⭐ Methyltransferase (scan mode, shipped parm7 → no AmberTools).'
        load_pdb([_c], parm=_pm,
                 center=['SAM', 'PHN'], lcharge={'SAM': 1, 'PHN': -1},
                 scan_preset="[('SAM 359 CS1','PHN 360 C8',1.3)]")
ex_btn.on_click(_load_example)
_input_formats = ('.pdb / .cif / .mmcif / .xyz / .gjf' if IS_CLUSTER else
                  '.pdb / .cif / .mmcif + matching .parm7')
_drop = W.VBox([W.HTML('<div style="text-align:center;color:#567;"><b>Drag and drop</b> '
                       '%s here — or click to browse</div>' % _input_formats), upl])
_drop.add_class('rxdrop')
input_file_rows = W.VBox(layout=W.Layout(width='100%'))
input_order_note = W.HTML()

def _queue_change(path=None, delta=0, remove=False, clear=False, kind='input'):
    if kind == 'parm':
        _invalidate_last_run('Amber topology removed; validate and run again.')
        S['parm'] = None; _render_input_queue(); refresh(); return
    if kind == 'model':
        clear_model = globals().get('_clear_uploaded_model')
        if clear_model is not None: clear_model()
        return
    old_paths = list(S.get('inputs', [])); paths = list(old_paths)
    if clear:
        paths = []; S['parm'] = None; S['model_pdb'] = None
    elif path in paths:
        i = paths.index(path)
        if remove: paths.pop(i)
        else:
            j = max(0, min(len(paths) - 1, i + delta))
            paths[i], paths[j] = paths[j], paths[i]
    if old_paths[:1] != paths[:1]:
        _clear_structure_bound_state()
    else:
        _invalidate_last_run('Input order changed; validate and run again.')
    S['inputs'] = paths
    if not paths:
        S.update(mode=None, _pdb_text='', _pdb_path=None, _atom_meta=[], _atoms={})
    _render_input_queue()
    if paths and S.get('mode') == 'pdb': build_selection()
    else: render_viewer()
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()

def _file_row(path, role, index=None, total=None, kind='input'):
    label = W.HTML('<span style="display:inline-block;overflow:hidden;text-overflow:ellipsis;'
                   'white-space:nowrap;max-width:100%%" title="%s"><b>%s</b> · %s</span>' %
                   (html.escape(path), html.escape(role), html.escape(os.path.basename(path))))
    controls = []
    if kind == 'input':
        up = W.Button(description='↑', tooltip='Move earlier', disabled=(index == 0),
                      layout=W.Layout(width='36px'))
        down = W.Button(description='↓', tooltip='Move later', disabled=(index == total - 1),
                        layout=W.Layout(width='36px'))
        up.on_click(lambda _, p=path: _queue_change(path=p, delta=-1))
        down.on_click(lambda _, p=path: _queue_change(path=p, delta=1))
        controls.extend([up, down])
    close = W.Button(description='×', tooltip='Remove %s' % os.path.basename(path),
                     layout=W.Layout(width='38px'))
    close.on_click(lambda _, p=path, k=kind: _queue_change(path=p, remove=True, kind=k))
    controls.append(close)
    row = W.HBox([label, W.HBox(controls, layout=W.Layout(flex_flow='row nowrap'))],
                 layout=W.Layout(width='100%', flex_flow='row nowrap', align_items='center'))
    row.add_class('rxfile')
    return row

def _render_input_queue():
    paths = list(S.get('inputs', [])); rows = []
    for i, path in enumerate(paths):
        role = ('input' if len(paths) == 1 else ('R' if i == 0 else ('P' if i == len(paths)-1 else 'IM')))
        rows.append(_file_row(path, '%d · %s' % (i + 1, role), i, len(paths)))
    if S.get('parm'): rows.append(_file_row(S['parm'], 'topology', kind='parm'))
    if S.get('model_pdb'): rows.append(_file_row(S['model_pdb'], 'prepared model', kind='model'))
    input_file_rows.children = rows or [W.HTML('<small style="color:#64748b">No files attached yet.</small>')]
    input_order_note.value = ('<small><b>%d structure(s)</b> · first = reactant, last = product%s</small>'
                              % (len(paths), (' · parm7: <code>%s</code>' % os.path.basename(S['parm'])) if S.get('parm') else ''))
    clear_button = globals().get('b_clear_inputs')
    if clear_button is not None: clear_button.disabled = not bool(paths or S.get('parm') or S.get('model_pdb'))
    sync_view = globals().get('_sync_view_input_widget')
    if sync_view is not None: sync_view()
b_clear_inputs = W.Button(description='clear files', icon='eraser', layout=W.Layout(width='120px'))
b_clear_inputs.on_click(lambda _: _queue_change(clear=True))
_render_input_queue()
input_box = W.VBox([
    W.HTML('<b>1. Load structures</b> — separate drops append to this file list.'),
    _drop, W.HTML('<b>2. Confirm reaction order</b> <small>— use ↑/↓ to reorder and × to remove.</small>'),
    input_file_rows, W.HBox([b_clear_inputs]), input_order_note,
    W.HTML('<hr style="margin:6px 0">'), W.HBox([ex_choice, ex_btn]), input_msg,
    W.HTML('<div class="rxcard"><b>Workflow map</b><br><small>'
           '<b>all</b>: prepare → path/scan → optional TS/IRC/freq/DFT · '
           '<b>path-opt/search</b>: endpoints → MEP · <b>scan/2d/3d</b>: coordinate grid · '
           '<b>tsopt → freq → irc</b>: refine and certify a TS · utilities: prepare/inspect/export files.'
           '</small></div>')])

# Colab renders ipywidgets' selection containers (Tab, Accordion) as an empty
# block, so every collapsible below is a Button + VBox that works everywhere.
_HELP_SEQ = {'n': 0}
def _help_markup(tip):
    """Return a hover/focus popover with a native-title fallback."""
    _HELP_SEQ['n'] += 1; ident = 'rxhelp-%d' % _HELP_SEQ['n']
    escaped = html.escape(str(tip), quote=True)
    return ('<span class="rxhelp-trigger" tabindex="0" title="%s" aria-label="Help: %s" aria-describedby="%s">&#9432;'
            '<span id="%s" role="tooltip" class="rxhelp-body">%s</span></span>' %
            (escaped, escaped, ident, ident, html.escape(str(tip))))

def _hdr(content, tip):
    return W.HTML('<div class="rxhelp-row" style="display:flex;align-items:center;gap:4px;">'
                  '<span>%s</span>%s</div>' % (content, _help_markup(tip)))

def _flag_row(widget, tip):
    help_widget = W.HTML(_help_markup(tip), layout=W.Layout(width='22px'))
    row = W.HBox([widget, help_widget], layout=W.Layout(flex_flow='row nowrap', align_items='center'))
    row.layout.display = widget.layout.display or ''
    widget._rx_flag_row = row
    row.add_class('rxflagrow'); return row

def _set_flag_visible(widget, visible):
    display_value = '' if visible else 'none'
    widget.layout.display = display_value
    row = getattr(widget, '_rx_flag_row', None)
    if row is not None: row.layout.display = display_value


def _collapsible(title, child, on_open=None):
    _btn = W.Button(layout=W.Layout(width='auto'))
    _body = W.VBox([child])
    _st = {'open': False}
    def _sync():
        _body.layout.display = '' if _st['open'] else 'none'
        _btn.description = ('Hide ' if _st['open'] else 'Show ') + title
    def _click(_):
        _st['open'] = not _st['open']; _sync()
        if _st['open'] and on_open is not None: on_open()
    _btn.on_click(_click); _sync()
    _box = W.VBox([_btn, _body]); _box.add_class('rxfold')
    return _box

# ============================================================== SELECT tab (3D pick)
viewer_out = W.Output()
viewer_status = W.HTML()
view_input = W.Dropdown(options=[], description='viewing', style={'description_width': 'initial'},
                        layout=W.Layout(width='300px', max_width='100%'))
view_input_note = W.HTML()
_view_input_guard = {'active': False}
center_panel, charge_panel, scan_panel, freeze_panel, measure_panel = (W.VBox() for _ in range(5))
for _p in (center_panel, charge_panel, scan_panel, freeze_panel, measure_panel): _p.add_class('rxcard')
center_ids_html = W.HTML()
center_widget = None


def _center_values():
    """The center picker's options are (label, value) pairs so ligands can be
    flagged in the label; callers that match on residue names need the values."""
    if center_widget is None: return []
    return [o[1] if isinstance(o, tuple) else o for o in center_widget.options]
charge_rows = None
_PICK_ACTIONS = (
    ('Center residue (-c)', 'center', 'center'), ('Ligand charge (-l)', 'ligand', 'center'),
    ('Scan bond · atom A', 'scanA', 'scan'), ('Scan bond · atom B', 'scanB', 'scan'),
    ('Freeze pair · atom A', 'freezeA', 'freeze'), ('Freeze pair · atom B', 'freezeB', 'freeze'),
    ('Freeze atom (Cartesian)', 'freezeatom', 'freeze'),
    ('Measure (dist/angle/dihedral)', 'measure', None),
)
pick_action = W.Dropdown(
    options=[(label, value) for label, value, _panel in _PICK_ACTIONS],
    value='center', description='Click sets:', style={'description_width': 'initial'},
    layout=W.Layout(width='360px', max_width='100%'))
exact_atom = W.Text(value='', description='exact atom',
                    placeholder='1-based index or A:SAM:359:C1',
                    style={'description_width': 'initial'},
                    layout=W.Layout(width='360px', max_width='100%'))
exact_atom_btn = W.Button(description='set current pick', icon='crosshairs',
                          layout=W.Layout(width='165px'),
                          tooltip='Apply “Click sets” to this exact atom without using WebGL.')
exact_atom_msg = W.HTML()

_CLICK_JS = """function(atom,viewer,event,container){
 if(!atom)return;
 ['__rxPickFill','__rxPickFrame','__rxPickAtom','__rxPickHalo'].forEach(function(k){
  if(viewer[k]){viewer.removeShape(viewer[k]);viewer[k]=null;}});
 if(viewer.__rxPickLabel){viewer.removeLabel(viewer.__rxPickLabel);viewer.__rxPickLabel=null;}
 var sel={chain:atom.chain||'',resn:atom.resn,resi:atom.resi};
 if(atom.icode)sel.icode=atom.icode;
 var atoms=viewer.selectedAtoms(sel); if(!atoms.length)atoms=[atom];
 var lo={x:Infinity,y:Infinity,z:Infinity},hi={x:-Infinity,y:-Infinity,z:-Infinity};
 atoms.forEach(function(a){lo.x=Math.min(lo.x,a.x);lo.y=Math.min(lo.y,a.y);lo.z=Math.min(lo.z,a.z);
                           hi.x=Math.max(hi.x,a.x);hi.y=Math.max(hi.y,a.y);hi.z=Math.max(hi.z,a.z);});
 var center={x:(lo.x+hi.x)/2,y:(lo.y+hi.y)/2,z:(lo.z+hi.z)/2};
 var dims={w:Math.max(1.5,hi.x-lo.x+1.3),h:Math.max(1.5,hi.y-lo.y+1.3),d:Math.max(1.5,hi.z-lo.z+1.3)};
 viewer.addStyle(sel,{stick:{color:'#f59e0b',radius:0.42}});
 viewer.__rxPickFill=viewer.addBox({center:center,dimensions:dims,color:'#f59e0b',opacity:0.08});
 viewer.__rxPickFrame=viewer.addShape({});
 var x0=center.x-dims.w/2,x1=center.x+dims.w/2,y0=center.y-dims.h/2,y1=center.y+dims.h/2,z0=center.z-dims.d/2,z1=center.z+dims.d/2;
 [[[x0,y0,z0],[x1,y0,z0]],[[x0,y1,z0],[x1,y1,z0]],[[x0,y0,z1],[x1,y0,z1]],[[x0,y1,z1],[x1,y1,z1]],
  [[x0,y0,z0],[x0,y1,z0]],[[x1,y0,z0],[x1,y1,z0]],[[x0,y0,z1],[x0,y1,z1]],[[x1,y0,z1],[x1,y1,z1]],
  [[x0,y0,z0],[x0,y0,z1]],[[x1,y0,z0],[x1,y0,z1]],[[x0,y1,z0],[x0,y1,z1]],[[x1,y1,z0],[x1,y1,z1]]].forEach(function(e){
   viewer.__rxPickFrame.addCylinder({start:{x:e[0][0],y:e[0][1],z:e[0][2]},end:{x:e[1][0],y:e[1][1],z:e[1][2]},radius:0.07,color:'#f59e0b'});});
 viewer.__rxPickAtom=viewer.addSphere({center:atom,radius:0.70,color:'#ff6b00'});
 viewer.__rxPickHalo=viewer.addSphere({center:atom,radius:1.00,color:'#111827',opacity:0.95,wireframe:true});
 viewer.__rxPickLabel=viewer.addLabel(((atom.chain||'')?atom.chain+':':'')+atom.resn+atom.resi+(atom.icode||'')+' · '+atom.atom,
  {position:atom,backgroundColor:'#111827',backgroundOpacity:0.90,fontColor:'white',fontSize:11,
   borderColor:'#f59e0b',borderThickness:1,inFront:true,showBackground:true});
 viewer.render();
 if(typeof google!=='undefined'&&google.colab&&google.colab.kernel){
  google.colab.kernel.invokeFunction('pdb2reaction_gui.on_click',
   [String(atom.index),atom.resn||'',String(atom.resi),(atom.chain||'').trim(),atom.atom||'',
    String(atom.serial),(atom.icode||'').trim(),viewer.getView()],{});}}"""

_VIEW_CHANGE_JS = """function(view){
 if(typeof window==='undefined'||typeof google==='undefined'||!google.colab||!google.colab.kernel)return;
 clearTimeout(window.__rxViewSyncTimer);
 window.__rxViewSyncTimer=setTimeout(function(){
  google.colab.kernel.invokeFunction('pdb2reaction_gui.on_view_change',[view],{});},180);}"""

_REP = {'cartoon': {'cartoon': {}}, 'sticks': {'stick': {}}, 'spheres': {'sphere': {}},
        'ball+stick': {'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}}, 'line': {'line': {}}}

def _base_style():
    sty = {k: dict(v) for k, v in _REP.get(S.get('rep', 'cartoon'), {'cartoon': {}}).items()}
    col = S.get('color', 'spectrum')
    for k in sty:
        if col == 'spectrum': sty[k]['color'] = 'spectrum'
        elif col == 'chain': sty[k]['colorscheme'] = 'chainHetatm'
        elif col == 'element': sty[k]['colorscheme'] = 'default'
    return sty

def _dash(v, p, q, label, color='orange'):
    if not p or not q: return
    v.addCylinder({'start': {'x': p[0], 'y': p[1], 'z': p[2]}, 'end': {'x': q[0], 'y': q[1], 'z': q[2]},
                   'radius': 0.04, 'dashed': True, 'fromCap': 1, 'toCap': 1, 'color': color})
    v.addLabel(label, {'position': {'x': (p[0]+q[0])/2, 'y': (p[1]+q[1])/2, 'z': (p[2]+q[2])/2},
                       'backgroundColor': 'white', 'fontColor': 'black', 'fontSize': 10})

def _center_indices():
    """Viewer indices selected by residue name or exact chain/residue identity."""
    out = []
    exact = set(S.get('center_ids', []))
    for meta in S.get('_atom_meta', []):
        resn = str(meta.get('resname') or '').upper()
        resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
        rid = ('%s:%s:%s' % (str(meta.get('chain') or ''), resn, resi)) if meta.get('chain') else resi
        if resn in {str(x).upper() for x in S.get('center', [])} or rid in exact or resi in exact:
            out.append(int(meta['index']))
    return sorted(set(out))

def _pick_residue_indices(pick=None):
    """Return the exact residue containing the last clicked atom."""
    pick = pick or S.get('_last_pick')
    if not pick: return []
    try: index = int(pick.get('index', -1))
    except (TypeError, ValueError): index = -1
    metadata = S.get('_atom_meta', [])
    if not (0 <= index < len(metadata)): return []
    target = metadata[index]
    key = (str(target.get('chain') or ''), str(target.get('resname') or '').upper(),
           str(target.get('resseq')), str(target.get('icode') or ''))
    out = []
    for meta in metadata:
        current = (str(meta.get('chain') or ''), str(meta.get('resname') or '').upper(),
                   str(meta.get('resseq')), str(meta.get('icode') or ''))
        if current == key and meta.get('index') is not None:
            out.append(int(meta['index']))
    return sorted(set(out))

def _pick_residue_box(indices):
    """Build a padded box around residue coordinates for a soft highlight."""
    coords = []
    metadata = S.get('_atom_meta', [])
    for index in indices:
        if 0 <= int(index) < len(metadata):
            xyz = metadata[int(index)].get('xyz')
            if xyz is not None and len(xyz) == 3: coords.append(tuple(float(v) for v in xyz))
    if not coords: return None
    lo = [min(p[axis] for p in coords) for axis in range(3)]
    hi = [max(p[axis] for p in coords) for axis in range(3)]
    return {
        'center': {'x': (lo[0] + hi[0]) / 2, 'y': (lo[1] + hi[1]) / 2, 'z': (lo[2] + hi[2]) / 2},
        'dimensions': {'w': max(1.5, hi[0] - lo[0] + 1.3),
                       'h': max(1.5, hi[1] - lo[1] + 1.3),
                       'd': max(1.5, hi[2] - lo[2] + 1.3)},
    }

def _pick_box_edges(box):
    """Return the 12 cylinder edges for a browser-visible residue frame."""
    center, dims = box['center'], box['dimensions']
    x0, x1 = center['x'] - dims['w'] / 2, center['x'] + dims['w'] / 2
    y0, y1 = center['y'] - dims['h'] / 2, center['y'] + dims['h'] / 2
    z0, z1 = center['z'] - dims['d'] / 2, center['z'] + dims['d'] / 2
    return [((x0,y0,z0),(x1,y0,z0)), ((x0,y1,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x1,y0,z1)), ((x0,y1,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y1,z0)), ((x1,y0,z0),(x1,y1,z0)),
            ((x0,y0,z1),(x0,y1,z1)), ((x1,y0,z1),(x1,y1,z1)),
            ((x0,y0,z0),(x0,y0,z1)), ((x1,y0,z0),(x1,y0,z1)),
            ((x0,y1,z0),(x0,y1,z1)), ((x1,y1,z0),(x1,y1,z1))]

def _pick_text(pick=None):
    pick = pick or S.get('_last_pick') or {}
    residue = '%s:%s:%s' % (pick.get('chain'), pick.get('resn'), pick.get('resi')) \
        if pick.get('chain') else '%s:%s' % (pick.get('resn'), pick.get('resi'))
    suffix = ' · atom #%d' % (int(pick['index']) + 1) if pick.get('index') is not None and int(pick['index']) >= 0 else ''
    return '%s · %s%s' % (residue, pick.get('atom') or '?', suffix)

def _draw_last_pick(v):
    """Draw residue context first, then a foreground atom marker and label."""
    pick = S.get('_last_pick')
    if not pick: return
    indices = _pick_residue_indices(pick)
    if indices:
        v.addStyle({'index': indices}, {'stick': {'color': '#f59e0b', 'radius': 0.42}})
        box = _pick_residue_box(indices)
        if box:
            v.addBox(dict(box, color='#f59e0b', opacity=0.08))
            for start, end in _pick_box_edges(box):
                v.addCylinder({'start': {'x': start[0], 'y': start[1], 'z': start[2]},
                               'end': {'x': end[0], 'y': end[1], 'z': end[2]},
                               'radius': 0.07, 'color': '#f59e0b'})
    xyz = pick.get('xyz')
    if xyz is None: return
    center = {'x': float(xyz[0]), 'y': float(xyz[1]), 'z': float(xyz[2])}
    # The opaque atom plus a larger dark wire halo remains visible over the
    # translucent residue box, thick sticks, and optional molecular surface.
    v.addSphere({'center': center, 'radius': 0.70, 'color': '#ff6b00'})
    v.addSphere({'center': center, 'radius': 1.00, 'color': '#111827',
                 'opacity': 0.95, 'wireframe': True})
    v.addLabel(_pick_text(pick), {'position': center, 'backgroundColor': '#111827',
               'backgroundOpacity': 0.90, 'fontColor': 'white', 'fontSize': 11,
               'borderColor': '#f59e0b', 'borderThickness': 1,
               'inFront': True, 'showBackground': True})

def render_viewer():
    if not S.get('_pdb_text'):
        viewer_status.layout.display = ''
        viewer_status.value = (
            '<div role="status" style="min-height:180px;display:flex;align-items:center;'
            'justify-content:center;text-align:center;border:1px dashed #cbd5e1;'
            'border-radius:12px;background:#f8fafc;color:#64748b;padding:16px;">'
            '<div><b>No structure loaded</b><br><small>Load a PDB/mmCIF in the Input tab, '
            'then return here to pick residues and atoms.</small></div></div>')
        with viewer_out: clear_output()
        return
    viewer_status.value = ''
    viewer_status.layout.display = 'none'
    text = S['_pdb_text']; het = S.get('_hetero', [])
    with viewer_out:
        clear_output()
        v = py3Dmol.view(width='100%', height=int(S.get('viewer_height', 320)))
        v.addModel(text, 'pdb', {'keepH': True, 'altLoc': '*'}); v.setStyle({}, _base_style())
        water_sel = {'resn': sorted(_WATER)}
        if not S['show_water']:
            v.setStyle(water_sel, {})
        else:
            v.addStyle(water_sel, {'stick': {'radius': 0.10, 'showNonBonded': True}})
            v.addStyle({'resn': sorted(_WATER), 'elem': 'O'},
                       {'sphere': {'radius': 0.50, 'color': '#38bdf8'}})
        visible_atoms = {} if S['show_water'] else {'not': water_sel}
        if S.get('rep') == 'cartoon':
            v.addStyle(visible_atoms, {'line': {'opacity': 0.30, 'colorscheme': 'default'}})
        if het: v.addStyle({'resn': het}, {'stick': {'colorscheme': 'greenCarbon'}})
        mapping_ok = bool(S.get('_view_mapping_ok', True))
        for rn in (S.get('center', []) if mapping_ok else []):
            v.addStyle({'resn': rn}, {'stick': {'color': 'magenta', 'radius': 0.3}})
        for rid in (S.get('center_ids', []) if mapping_ok else []):
            parts = rid.split(':')
            for meta in S.get('_atom_meta', []):
                same = False
                if len(parts) == 3:
                    meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
                    same = (str(meta.get('chain') or '') == parts[0] and
                            str(meta.get('resname') or '').upper() == parts[1].upper() and
                            meta_resi == parts[2])
                elif len(parts) == 1:
                    meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
                    same = meta_resi == parts[0]
                if same: v.addStyle({'index': meta['index']}, {'stick': {'color': 'magenta', 'radius': 0.3}})
        for s in (S['scan_atoms'] if mapping_ok else []):
            if s: v.addStyle({'index': s.get('index')}, {'sphere': {'color': 'red', 'scale': 0.45}})
        for idx in (S.get('freeze_atoms', []) if mapping_ok else []):
            v.addStyle({'index': int(idx) - 1}, {'sphere': {'color': 'cyan', 'scale': 0.46}})
        for at in (S.get('measure_atoms', []) if mapping_ok else []):
            v.addStyle({'index': at.get('index')}, {'sphere': {'color': 'blue', 'scale': 0.4}})
        sa, sb = S['scan_atoms'] if mapping_ok else (None, None)
        if sa and sb:
            d = scan_distance(); _dash(v, _xyz(sa), _xyz(sb), ('%.2f Å' % d) if d else 'scan', 'red')
        for p in (S.get('freeze_pairs', []) if mapping_ok else []):
            pa, pb = _xyz(p['a']), _xyz(p['b'])
            for at in (p['a'], p['b']):
                if at.get('index') is not None:
                    v.addStyle({'index': at['index']}, {'sphere': {'color': 'cyan', 'scale': 0.42}})
            if pa and pb: _dash(v, pa, pb, '%.2f Å' % math.dist(pa, pb), 'cyan')
        if S.get('surface'):
            v.addSurface(py3Dmol.VDW, {'opacity': 0.12 if S.get('_last_pick') else 0.35,
                                       'color': 'white'}, visible_atoms, visible_atoms)
        _draw_last_pick(v)
        v.setClickable(visible_atoms, True, _CLICK_JS)
        v.setHoverable(visible_atoms, True,
            "function(a,v){if(!a.hl){a.hl=v.addLabel(a.resn+a.resi+' '+a.atom,"
            "{position:a,backgroundColor:'black',fontColor:'white',fontSize:10});}}",
            "function(a,v){if(a.hl){v.removeLabel(a.hl);delete a.hl;}}")
        v.setViewChangeCallback(_VIEW_CHANGE_JS)
        if S.get('spin'): v.spin(True)
        if S.get('_zoompick') and S.get('_last_pick'):
            indices = _pick_residue_indices()
            v.zoomTo({'index': indices} if indices else {'index': S['_last_pick'].get('index')})
            S['_zoompick'] = False
        elif mapping_ok and S.get('_zoomsel') and (S.get('center') or S.get('center_ids')):
            indices = _center_indices()
            v.zoomTo({'index': indices} if indices else {})
            S['_zoomsel'] = False
        elif S.get('_zoomsel'):
            S['_zoomsel'] = False; v.zoomTo()
        elif (isinstance(S.get('_viewer_view'), (list, tuple)) and
              len(S['_viewer_view']) == 8):
            v.setView(list(S['_viewer_view']))
        else:
            v.zoomTo(visible_atoms)
        v.show()

def _render_center_ids():
    ids = S.get('center_ids', [])
    center_ids_html.value = (
        ('<small>+ exact residues: <code>%s</code></small>' % ','.join(ids)) if ids
        else ('<small>none picked yet — choose a name above, or click a residue in the 3D view</small>')
    )

def _resolve_click_meta(atom_index, serial='', resn='', resi='', chain='', atom='', icode=''):
    """Map a browser atom to retained input metadata without trusting 3Dmol index."""
    metadata = S.get('_atom_meta', [])
    norm = lambda value: str(value or '').strip()
    try: serial_int = int(serial)
    except (TypeError, ValueError): serial_int = None
    if serial_int is not None:
        found = [meta for meta in metadata if meta.get('serial') == serial_int]
        if len(found) == 1: return found[0]
    expected = (norm(chain), norm(resn).upper(), norm(resi), norm(icode), norm(atom).upper())
    if any(expected):
        found = [meta for meta in metadata
                 if (norm(meta.get('chain')), norm(meta.get('resname')).upper(),
                     norm(meta.get('resseq')), norm(meta.get('icode')),
                     norm(meta.get('name')).upper()) == expected]
        if len(found) == 1: return found[0]
        return None
    try: return metadata[int(atom_index)]
    except (IndexError, TypeError, ValueError): return None

def on_click(atom_index, resn='', resi='', chain='', atom='', serial='', icode='', view=None):
    meta = _resolve_click_meta(atom_index, serial, resn, resi, chain, atom, icode)
    if meta is None:
        exact_atom_msg.value = ('<small role="alert" style="color:#991b1b">Could not map this viewer atom '
                                'to the input; workflow selections were not changed.</small>')
        return
    try: viewer_index = int(atom_index)
    except (TypeError, ValueError): viewer_index = -1
    index = int(meta['index'])
    resn = str(meta.get('resname') or resn).strip()
    resi = str(meta.get('resseq') if meta.get('resseq') is not None else resi)
    icode = str(meta.get('icode') or icode).strip(); resi += icode
    chain = str(meta.get('chain') or chain).strip()
    atom = str(meta.get('name') or atom).strip()
    serial = meta.get('serial', serial); xyz = meta.get('xyz')
    exact_atom_msg.value = ''
    if isinstance(view, (list, tuple)) and len(view) == 8:
        try:
            values = [float(value) for value in view]
            if all(math.isfinite(value) for value in values): S['_viewer_view'] = values
        except (TypeError, ValueError):
            pass
    act = pick_action.value
    message, tone = '', 'ok'
    if S.get('_view_input_index', 0) and not S.get('_view_mapping_ok', False):
        message, tone = 'view-only: atom identifiers differ from the first input; return to R/input 1 to edit selections', 'warn'
    elif act == 'center':
        rid = ('%s:%s:%s' % (chain, resn, resi)) if chain else str(resi)
        narrowed = False
        if center_widget is not None and resn in set(center_widget.value):
            center_widget.value = tuple(value for value in center_widget.value if value != resn)
            narrowed = True
        elif resn in S.get('center', []):
            S['center'] = [value for value in S['center'] if value != resn]
            narrowed = True
        if rid not in S['center_ids']:
            S['center_ids'].append(rid); _render_center_ids()
            message = ('replaced the %s name selection with this exact center (-c)' % resn
                       if narrowed else 'added as an exact center (-c)')
        else:
            message = 'already selected as an exact center (-c)'
    elif act == 'ligand' and charge_rows is not None and resn in charge_rows:
        row = charge_rows[resn]
        if row.get('auto'):
            message = 'uses the built-in %s charge (%+g)' % (resn, row['val'].value)
        else:
            row['use'].value = True; message = 'enabled its ligand-charge (-l) row'
    elif act == 'ligand':
        message, tone = 'not a detected ligand/cofactor; choose a hetero residue', 'warn'
    elif act in ('scanA', 'scanB'):
        S['scan_atoms'][0 if act == 'scanA' else 1] = {
            'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        message = 'set scan atom %s' % ('A' if act == 'scanA' else 'B')
        if act == 'scanA':
            pick_action.value = 'scanB'; message += '; next click sets atom B'
        _render_scan_panel()
    elif act in ('freezeA', 'freezeB'):
        S['freeze_buf'][0 if act == 'freezeA' else 1] = {
            'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        message = 'set restraint endpoint %s' % ('A' if act == 'freezeA' else 'B')
        if act == 'freezeA':
            pick_action.value = 'freezeB'; message += '; next click sets endpoint B'
        _render_freeze_panel()
    elif act == 'freezeatom':
        idx = index + 1 if index >= 0 else None
        if idx is not None and idx not in S['freeze_atoms']:
            S['freeze_atoms'].append(idx); _render_freeze_panel(); message = 'added frozen atom #%d' % idx
        else:
            message = 'atom is already frozen' if idx is not None else 'could not resolve the atom index'
            tone = 'warn'
    elif act == 'measure':
        S['measure_atoms'].append({'chain': chain, 'resn': resn, 'resi': resi,
                                   'atom': atom, 'xyz': xyz, 'index': index})
        if len(S['measure_atoms']) > 4: S['measure_atoms'].pop(0)
        message = 'added measurement point %d/4' % len(S['measure_atoms'])
        _render_measure_panel()
    S['_last_pick'] = {'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom,
                       'serial': serial, 'icode': icode, 'xyz': xyz, 'index': index,
                       'viewer_index': viewer_index, 'action': act}
    S['_last_pick_message'] = message or 'selected'
    S['_last_pick_tone'] = tone
    _rlp = globals().get('_render_last_pick_status')
    if _rlp is not None: _rlp()
    render_viewer(); refresh()
try:
    from google.colab import output as _co
    _co.register_callback('pdb2reaction_gui.on_click', on_click)
    def _store_view(view):
        if isinstance(view, (list, tuple)) and len(view) == 8:
            try:
                values = [float(value) for value in view]
                if all(math.isfinite(value) for value in values): S['_viewer_view'] = values
            except (TypeError, ValueError):
                pass
    _co.register_callback('pdb2reaction_gui.on_view_change', _store_view)
except Exception:
    _co = None              # click->Python only registers inside Google Colab

def _render_scan_panel():
    a, b = S['scan_atoms']
    def fmt(x): return _aspec(x) if x else '— click to pick —'
    d = scan_distance()
    dist_html = ('<b style="color:%s">current A–B: %.2f Å</b>' % (ACCENT, d)) if d is not None else \
                '<small>current A–B: (pick both atoms)</small>'
    tgt = W.BoundedFloatText(value=float(S['scan_target']), min=0.3, max=6.0, step=0.05,
                             description='target Å', layout=W.Layout(width='160px'))
    def _t(_): S['scan_target'] = tgt.value; refresh()
    tgt.observe(_t, names='value')
    def _addstage(_):
        if a and b: S['scan_stages'].append([{'a': a, 'b': b, 't': S['scan_target']}]); _render_scan_panel(); refresh()
    def _addconc(_):
        if a and b:
            if not S['scan_stages']: S['scan_stages'].append([])
            S['scan_stages'][-1].append({'a': a, 'b': b, 't': S['scan_target']}); _render_scan_panel(); refresh()
    def _clrstages(_): S['scan_stages'] = []; _render_scan_panel(); refresh()
    def _clrcur(_): S['scan_atoms'] = [None, None]; S['scan_preset'] = ''; _render_scan_panel(); render_viewer(); refresh()
    b_ns = W.Button(description='add stage', icon='plus', layout=W.Layout(width='120px'), tooltip='Add the current bond as a new sequential stage (staged scan).')
    b_cc = W.Button(description='add concerted', icon='plus', layout=W.Layout(width='135px'), tooltip='Add the current bond to the last stage (bonds changed together = concerted).')
    b_cs = W.Button(description='clear stages', layout=W.Layout(width='120px'))
    b_cl = W.Button(description='clear bond', icon='eraser', layout=W.Layout(width='110px'))
    b_ns.on_click(_addstage); b_cc.on_click(_addconc); b_cs.on_click(_clrstages); b_cl.on_click(_clrcur)
    summ = '<br>'.join('stage %d: %s' % (i + 1, ' & '.join('%s↔%s→%.3g Å' % (_aspec(x['a']), _aspec(x['b']), x['t']) for x in st))
                       for i, st in enumerate(S['scan_stages'])) or '<i>none (the single bond above is used)</i>'
    sub_now = _wv('dd_subcmd', S['subcmd'])
    if sub_now in ('scan2d', 'scan3d'):                          # 2D/3D grid: per-axis low/high ranges
        need = 2 if sub_now == 'scan2d' else 3
        lo = W.BoundedFloatText(value=1.2, min=0.3, max=6.0, step=0.05, description='low Å', layout=W.Layout(width='148px'))
        hi = W.BoundedFloatText(value=3.0, min=0.3, max=8.0, step=0.05, description='high Å', layout=W.Layout(width='148px'))
        axis_note = W.HTML()
        def _addaxis(_):
            if not (a and b):
                axis_note.value = '<small role="alert" style="color:#991b1b">Pick both atoms first.</small>'; return
            if a.get('index') == b.get('index'):
                axis_note.value = '<small role="alert" style="color:#991b1b">An axis needs two different atoms.</small>'; return
            if lo.value >= hi.value:
                axis_note.value = '<small role="alert" style="color:#991b1b">Low must be smaller than high.</small>'; return
            if len(S['scan_axes']) >= need:
                axis_note.value = '<small role="alert" style="color:#991b1b">This grid already has %d axes.</small>' % need; return
            key = tuple(sorted((a.get('index'), b.get('index'))))
            existing = {tuple(sorted((x['a'].get('index'), x['b'].get('index')))) for x in S['scan_axes']}
            if key in existing:
                axis_note.value = '<small role="alert" style="color:#991b1b">That bond is already an axis.</small>'; return
            S['scan_axes'].append({'a': a, 'b': b, 'lo': lo.value, 'hi': hi.value})
            _render_scan_panel(); refresh()
        def _clraxes(_): S['scan_axes'] = []; _render_scan_panel(); refresh()
        def _popaxis(_):
            if S['scan_axes']: S['scan_axes'].pop(); _render_scan_panel(); refresh()
        b_ax = W.Button(description='add axis', icon='plus', layout=W.Layout(width='120px'),
                        tooltip='Add the current bond as a scan axis (low→high distance range).')
        b_ax.disabled = len(S['scan_axes']) >= need
        b_px = W.Button(description='remove last', icon='minus', layout=W.Layout(width='125px'),
                        disabled=not S['scan_axes'])
        b_cx = W.Button(description='clear axes', layout=W.Layout(width='110px'))
        b_ax.on_click(_addaxis); b_px.on_click(_popaxis); b_cx.on_click(_clraxes)
        axsum = '<br>'.join('axis %d: %s↔%s  [%.3g – %.3g Å]' % (k + 1, _aspec(x['a']), _aspec(x['b']), x['lo'], x['hi'])
                            for k, x in enumerate(S['scan_axes'])) or '<i>none</i>'
        col = '#1f7a3d' if len(S['scan_axes']) == need else '#a60'
        scan_panel.children = [
            W.HTML('<b>%s grid -s</b> <small>pick bond A/B, set low/high, “add axis” — need <b>%d</b> axes</small>' % (sub_now, need)),
            W.HTML('A: <code>%s</code><br>B: <code>%s</code><br>%s' % (fmt(a), fmt(b), dist_html)),
            W.HBox([lo, hi]), W.HBox([b_ax, b_px, b_cx]), axis_note,
            W.HTML('<small style="color:%s"><b>axes (%d/%d):</b><br>%s</small>' % (col, len(S['scan_axes']), need, axsum))]
        return
    note = '' if len(S.get('inputs', [])) == 1 else \
        '<br><small style="color:#a60">(scan applies to single-PDB mode; ignored for multi-PDB MEP)</small>'
    scan_panel.children = [W.HTML('<b>Scan bond -s</b>%s' % note),
                           W.HTML('A: <code>%s</code><br>B: <code>%s</code><br>%s' % (fmt(a), fmt(b), dist_html)),
                           tgt, W.HBox([b_ns, b_cc]), W.HBox([b_cs, b_cl]),
                           W.HTML('<small><b>staged scan:</b><br>%s</small>' % summ)]

def _render_freeze_panel():
    fa, fb = S['freeze_buf']
    def fmt(x): return _aspec(x) if x else '— pick —'
    def _addpair(_):
        if fa and fb:
            S['freeze_pairs'].append({'a': fa, 'b': fb, 't': None}); S['freeze_buf'] = [None, None]
            _render_freeze_panel(); refresh()
    def _clrpairs(_): S['freeze_pairs'] = []; _render_freeze_panel(); refresh()
    def _clratoms(_): S['freeze_atoms'] = []; _render_freeze_panel(); render_viewer(); refresh()
    b_ap = W.Button(description='add freeze pair', icon='plus', layout=W.Layout(width='160px'))
    b_cp = W.Button(description='clear pairs', layout=W.Layout(width='110px'))
    b_ca = W.Button(description='clear atoms', layout=W.Layout(width='110px'))
    b_ap.on_click(_addpair); b_cp.on_click(_clrpairs); b_ca.on_click(_clratoms)
    pairs = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs']) or '(none)'
    atoms = ','.join(str(i) for i in S['freeze_atoms']) or '(none)'
    freeze_panel.children = [
        W.HTML('<b>Distance restraints</b> <small>(<code>--dist-freeze</code>, opt) — set pick mode to '
               '“Freeze pair · A/B”, click two atoms, then add</small>'),
        W.HTML('A: <code>%s</code> &nbsp; B: <code>%s</code>' % (fmt(fa), fmt(fb))),
        W.HBox([b_ap, b_cp]),
        W.HTML('<small>pairs: <code>%s</code></small>' % pairs),
        W.HTML('<hr style="margin:5px 0"><b>Frozen atoms</b> <small>(<code>--freeze-atoms</code>; '
               'this panel appears only for accepting subcommands) — pick mode “Freeze atom”, click atoms</small>'),
        W.HBox([b_ca]),
        W.HTML('<small>1-based indices: <code>%s</code></small>' % atoms)]

def _render_measure_panel():
    pts = [_xyz(a) for a in S['measure_atoms']]
    names = ' , '.join(_aspec(a) for a in S['measure_atoms']) or '(none)'
    rows = []
    if len(pts) >= 2 and pts[0] and pts[1]:
        rows.append('distance(1–2): <b>%.3f Å</b>' % math.dist(pts[0], pts[1]))
    if len(pts) >= 3 and all(pts[:3]):
        ang = _angle(pts[0], pts[1], pts[2])
        if ang is not None: rows.append('angle(1-2-3): <b>%.1f°</b>' % ang)
    if len(pts) >= 4 and all(pts[:4]):
        rows.append('dihedral(1-2-3-4): <b>%.1f°</b>' % _dihedral(pts[0], pts[1], pts[2], pts[3]))
    clr = W.Button(description='clear measure', layout=W.Layout(width='130px'))
    clr.on_click(lambda _: (S.__setitem__('measure_atoms', []), _render_measure_panel(), render_viewer()))
    measure_panel.children = [
        W.HTML('<b>Measure</b> <small>(pick mode “Measure”; 2=dist · 3=angle · 4=dihedral)</small>'),
        W.HTML('<small>atoms: %s</small>' % names),
        W.HTML('<small>%s</small>' % ('<br>'.join(rows) or '—')), clr]

def _view_role(index, total):
    if total <= 1: return 'input'
    if index == 0: return 'R'
    if index == total - 1: return 'P'
    return 'IM%d' % index

def _sync_view_input_widget():
    paths = list(S.get('inputs', []))
    index = max(0, min(int(S.get('_view_input_index', 0)), max(0, len(paths) - 1)))
    S['_view_input_index'] = index
    opts = [('%s · %s' % (_view_role(i, len(paths)), os.path.basename(path)), i)
            for i, path in enumerate(paths)]
    _view_input_guard['active'] = True
    try:
        view_input.options = opts
        if opts: view_input.value = index
    finally:
        _view_input_guard['active'] = False

def _atom_signatures(metadata):
    return [(str(m.get('chain') or ''), str(m.get('resname') or '').upper(),
             str(m.get('resseq')), str(m.get('icode') or ''), str(m.get('name') or '').upper())
            for m in metadata]

def _resolve_atom_query(query):
    query = str(query or '').strip()
    if not query: raise ValueError('Enter a 1-based atom index or CHAIN:RESNAME:RESSEQ:ATOM.')
    metadata = S.get('_atom_meta', [])
    if query.isdigit():
        index = int(query) - 1
        if 0 <= index < len(metadata): return index
        raise ValueError('Atom index %s is outside 1–%d.' % (query, len(metadata)))
    parts = [part.strip() for part in query.split(':')]
    if len(parts) == 4: chain, resn, resi, atom = parts
    elif len(parts) == 3: chain, (resn, resi, atom) = '', parts
    else: raise ValueError('Use CHAIN:RESNAME:RESSEQ[ICODE]:ATOM (or omit CHAIN).')
    found = []
    for meta in metadata:
        meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
        if (str(meta.get('chain') or '') == chain and
                str(meta.get('resname') or '').upper() == resn.upper() and
                meta_resi == resi and str(meta.get('name') or '').upper() == atom.upper()):
            found.append(int(meta['index']))
    if len(found) == 1: return found[0]
    if len(found) > 1: raise ValueError('That atom is ambiguous; include the chain ID.')
    raise ValueError('No retained atom matches %s.' % query)

def _apply_exact_atom(_=None):
    try:
        index = _resolve_atom_query(exact_atom.value)
    except ValueError as exc:
        exact_atom_msg.value = '<small role="alert" style="color:#991b1b">%s</small>' % html.escape(str(exc))
        return
    exact_atom_msg.value = ''
    on_click(str(index), view=S.get('_viewer_view'))
exact_atom_btn.on_click(_apply_exact_atom)

def _view_is_editable():
    return int(S.get('_view_input_index', 0)) == 0 or bool(S.get('_view_mapping_ok'))

def _set_primary_editor_enabled(enabled):
    disabled = not bool(enabled)
    if center_widget is not None: center_widget.disabled = disabled
    if charge_rows is not None:
        for row in charge_rows.values():
            row['use'].disabled = disabled or row.get('auto', False)
            row['val'].disabled = disabled or row.get('auto', False)

def build_selection():
    """Load and commit one view atomically; primary editors remain primary-owned."""
    global center_widget, charge_rows
    if S['mode'] != 'pdb' or not S['inputs']:
        render_viewer()
        return False
    _sync_view_input_widget()
    view_index = int(S.get('_view_input_index', 0))
    path = S['inputs'][view_index]
    try:
        text, metadata, viewer_path = _load_view_structure(path)
        allr, het = parse_residues(text, metadata)
        atoms = parse_atoms(text, metadata)
        signatures = _atom_signatures(metadata)
    except Exception as exc:
        with viewer_out: clear_output(); print('Could not prepare structure for the viewer:', exc)
        input_msg.value = '❌ structure load failed: <code>%s</code>' % html.escape(str(exc))
        return False
    S.update(_pdb_path=viewer_path, _pdb_text=text, _atom_meta=metadata,
             _hetero=het, _atoms=atoms)
    if view_index == 0:
        S['_primary_atom_signatures'] = signatures
        S['_primary_atom_meta'] = [dict(meta) for meta in metadata]
        S['_view_mapping_ok'] = True
        view_input_note.value = ''
    else:
        S['_view_mapping_ok'] = bool(S.get('_primary_atom_signatures')) and signatures == S['_primary_atom_signatures']
        view_input_note.value = (('<small style="color:#166534">Atom identifiers match input 1; '
                                  'picks update the shared workflow selection.</small>')
                                 if S['_view_mapping_ok'] else
                                 ('<small role="alert" style="color:#92400e">View-only: atom identifiers/order '
                                  'differ from input 1. Clicks still highlight atoms and residues, but cannot '
                                  'change the workflow; return to R/input 1 to edit selections.</small>'))
    if view_index == 0:
        initial_center = tuple(x for x in S.get('center', []) if x in het)
        _center_opts = [('%s  ← ligand/cofactor' % r, r) for r in het]
        center_widget = W.SelectMultiple(options=_center_opts, value=initial_center,
                                         rows=min(5, max(3, len(het))), layout=W.Layout(width='230px'))
        charge_rows = {}; rows = []
        for rn in het:
            auto = rn in _ION_CHARGES
            pre = _ION_CHARGES[rn] if auto else S.get('lcharge', {}).get(rn, 0)
            use = W.Checkbox(value=(auto or rn in S.get('lcharge', {})),
                             description=('auto %s' if auto else 'use %s') % rn,
                             disabled=auto, indent=False, layout=W.Layout(width='105px'))
            val = W.BoundedFloatText(value=float(pre), min=-9, max=9, step=1,
                                     description='%s charge' % rn, style={'description_width': 'initial'},
                                     disabled=auto, layout=W.Layout(width='165px'))
            charge_rows[rn] = {'use': use, 'val': val, 'auto': auto}; rows.append(W.HBox([use, val]))
        def _sync_center(_=None):
            if not _view_is_editable(): return
            S['center'] = list(center_widget.value)
            _render_center_ids(); render_viewer(); refresh()
        def _sync_charge_rows(_=None):
            if not _view_is_editable(): return
            S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                            if x['use'].value and not x.get('auto')}
            refresh()
        center_widget.observe(_sync_center, names='value')
        for x in charge_rows.values():
            x['use'].observe(_sync_charge_rows, names='value'); x['val'].observe(_sync_charge_rows, names='value')
        center_panel.children = [
            _hdr('<b>center <code>-c</code></b> <small>— what the active-site model is built around</small>',
                 'Choose a ligand/cofactor name, or click an exact residue in 3D. Exact picks are stored as '
                 'CHAIN:RESNAME:RESSEQ.'),
            W.HTML('<small>Choose a ligand/cofactor name, or click an exact residue in 3D.</small>'),
            center_widget, center_ids_html]
        charge_list = (W.VBox(rows, layout=W.Layout(max_height='140px', overflow='auto', width='100%'))
                       if rows else W.HTML('<i>no hetero ligands</i>'))
        charge_panel.children = [
            W.HTML('<b>ligand charges -l</b> <small>· known ions are automatic</small>'), charge_list]
        _sync_charge_rows()
    _set_primary_editor_enabled(_view_is_editable())
    _render_center_ids(); _render_scan_panel(); _render_freeze_panel(); _render_measure_panel()
    render_viewer()
    return True

def _on_view_input(change):
    if _view_input_guard['active'] or change.get('new') is None: return
    previous = {key: S.get(key) for key in ('_view_input_index', '_last_pick', '_last_pick_message',
                                            '_last_pick_tone', '_viewer_view', '_zoompick', '_zoomsel')}
    previous_note = view_input_note.value
    S.update(_view_input_index=int(change['new']), _last_pick=None,
             _last_pick_message='', _last_pick_tone='ok', _viewer_view=None,
             _zoompick=False, _zoomsel=False)
    if not build_selection():
        S.update(previous); view_input_note.value = previous_note
        _sync_view_input_widget(); render_viewer()
    _render_last_pick_status()
view_input.observe(_on_view_input, names='value')

# PyMOL-like view controls
dd_rep = W.Dropdown(options=['cartoon', 'sticks', 'ball+stick', 'spheres', 'line'], value='cartoon',
                    description='rep', style={'description_width': 'initial'}, layout=W.Layout(width='150px'))
dd_col = W.Dropdown(options=['spectrum', 'chain', 'element'], value='spectrum',
                    description='color', style={'description_width': 'initial'}, layout=W.Layout(width='150px'))
cb_water = W.Checkbox(value=False, description='water', indent=False,
                      layout=W.Layout(width='82px'))
cb_surf  = W.Checkbox(value=False, description='full surface (slow)', indent=False,
                      layout=W.Layout(width='150px'))
cb_spin  = W.Checkbox(value=False, description='spin', indent=False,
                      layout=W.Layout(width='72px'))
dd_height = W.Dropdown(options=[('compact', 320), ('tall', 440), ('extra tall', 560)], value=320,
                       description='height', style={'description_width': 'initial'},
                       layout=W.Layout(width='150px'))
btn_reset = W.Button(description='reset view', icon='refresh', layout=W.Layout(width='120px'),
                     tooltip='Reset the camera and re-render the current selections.')
btn_zoomsel = W.Button(description='zoom to -c', icon='search-plus', layout=W.Layout(width='120px'),
                       tooltip='Zoom to the current center selection.')
btn_zoompick = W.Button(description='zoom to click', icon='crosshairs', layout=W.Layout(width='135px'),
                        tooltip='Zoom to the residue containing the last clicked atom.')
btn_clear_pick = W.Button(description='clear last click', icon='eye-slash', layout=W.Layout(width='145px'),
                          tooltip='Remove only the orange atom/residue highlight; workflow selections stay unchanged.')
def _vopt(_=None):
    S['show_water'] = cb_water.value; S['surface'] = cb_surf.value; S['spin'] = cb_spin.value
    S['rep'] = dd_rep.value; S['color'] = dd_col.value; S['viewer_height'] = dd_height.value
    S['_zoomsel'] = False; S['_zoompick'] = False; render_viewer()
for c in (cb_water, cb_surf, cb_spin, dd_rep, dd_col, dd_height): c.observe(_vopt, names='value')
def _reset_view(_=None):
    S['_viewer_view'] = None; S['_zoomsel'] = False; S['_zoompick'] = False; render_viewer()
btn_reset.on_click(_reset_view)
btn_zoomsel.on_click(lambda _: (S.__setitem__('_zoomsel', True), render_viewer()))
btn_zoompick.on_click(lambda _: (S.__setitem__('_zoompick', True), render_viewer()))
def _clear_last_pick(_=None):
    S.update(_last_pick=None, _last_pick_message='', _last_pick_tone='ok')
    _rlp = globals().get('_render_last_pick_status')
    if _rlp is not None: _rlp()
    render_viewer()
btn_clear_pick.on_click(_clear_last_pick)
view_controls = W.HBox([dd_rep, dd_col, cb_water, dd_height],
                       layout=W.Layout(flex_flow='row wrap'))

_sel_help = ('<small><b>Choose “Click sets”, then click the structure.</b> Exact centers use '
             '<code>CHAIN:RESNAME:RESSEQ</code>; the name list contains ligands/cofactors only.</small>')
summary_html = W.HTML()
def _render_summary():
    cw = center_widget
    cen = ','.join(list(cw.value) if cw is not None else S.get('center', []))
    ids = ','.join(S.get('center_ids', []))
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    auto_lc = ','.join('%s:%+g' % (name, row['val'].value) for name, row in (charge_rows or {}).items()
                       if row.get('auto'))
    sc = ' ; '.join(scan_literals())
    fp = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs'])
    fa = ','.join(str(i) for i in S['freeze_atoms'])
    rows = []
    if cen or ids: rows.append('center -c: <code>%s</code>%s' %
                               (html.escape(cen or '—'), (' · exact <code>%s</code>' % html.escape(ids)) if ids else ''))
    if lc: rows.append('charges -l: <code>%s</code>' % html.escape(lc))
    if auto_lc: rows.append('built-in ion charge: <code>%s</code>' % html.escape(auto_lc))
    if sc: rows.append('scan -s: <code>%s</code>' % html.escape(sc))
    if fp or fa: rows.append('freeze: <code>%s%s</code>' %
                             (html.escape(fp), (' · atoms ' + html.escape(fa)) if fa else ''))
    summary_html.value = (
        '<div style="border:1px solid #cdd;border-radius:8px;padding:5px 7px;background:#f7fbff;">'
        '<b>Selection summary</b><br><small>%s</small></div>' %
        ('<br>'.join(rows) if rows else 'No workflow selections yet.'))

chips_box = W.HBox()
chips_box.add_class('rxchip')
def _render_chips():
    def mk(kind, key):
        def _rm(_):
            if kind == 'center' and center_widget is not None:
                center_widget.value = tuple(x for x in center_widget.value if x != key)
                return
            elif kind == 'id':
                S['center_ids'] = [x for x in S['center_ids'] if x != key]
                _render_center_ids()
            elif kind == 'atom':
                S['freeze_atoms'] = [x for x in S['freeze_atoms'] if x != key]
                _render_freeze_panel()
            _render_chips(); render_viewer(); refresh()
        return _rm
    btns = []
    cur = list(center_widget.value) if center_widget is not None else S.get('center', [])
    for rn in cur:
        b = W.Button(description='%s ✕' % rn, button_style='info', layout=W.Layout(width='auto'))
        b.on_click(mk('center', rn)); btns.append(b)
    for rid in S.get('center_ids', []):
        b = W.Button(description='%s ✕' % rid, button_style='warning', layout=W.Layout(width='auto'))
        b.on_click(mk('id', rid)); btns.append(b)
    for fa in S.get('freeze_atoms', []):
        b = W.Button(description='⚓%d ✕' % fa, layout=W.Layout(width='auto'))
        b.on_click(mk('atom', fa)); btns.append(b)
    chips_box.children = btns or [W.HTML('<small>no center/freeze selection yet — click in 3D</small>')]

freeze_acc = _collapsible('⚙️ Freezing & measurement (advanced)',
                          W.VBox([freeze_panel, W.HTML('<hr style="margin:6px 0">'), measure_panel]))
viewer_out.layout = W.Layout(width='100%')
pick_hint = W.HTML()
last_pick_html = W.HTML()
_PICK_HINT = {
    'center': 'Click a residue to add an exact <code>-c</code> center.',
    'ligand': 'Click a ligand/cofactor to inspect or enable its charge.',
    'scanA': 'Click the first atom of the scan coordinate.', 'scanB': 'Click the second atom of the scan coordinate.',
    'freezeA': 'Click endpoint A of an opt distance restraint.', 'freezeB': 'Click endpoint B of an opt distance restraint.',
    'freezeatom': 'Click atoms to freeze by 1-based Cartesian index.',
    'measure': 'Click 2/3/4 atoms to measure distance/angle/dihedral.',
}
def _render_pick_hint(_=None):
    pick_hint.value = ('<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:9px;'
                       'padding:6px 9px;"><b>Active click:</b> %s</div>' % _PICK_HINT[pick_action.value])
    status_renderer = globals().get('_render_last_pick_status')
    if status_renderer is not None: status_renderer()
pick_action.observe(_render_pick_hint, names='value'); _render_pick_hint()
def _render_last_pick_status():
    pick = S.get('_last_pick')
    if not pick:
        last_pick_html.value = ('<div role="status" aria-live="polite" style="color:#475569">'
                                '<small><b>Click action:</b> %s</small></div>' % _PICK_HINT[pick_action.value])
        return
    tone = '#92400e' if S.get('_last_pick_tone') == 'warn' else '#166534'
    last_pick_html.value = (
        '<div role="status" aria-live="polite" style="border-left:4px solid #f59e0b;'
        'background:#fffbeb;padding:5px 8px;color:%s"><small><b>Last click:</b> '
        '<code>%s</code> — %s. Amber frame/sticks = residue; orange sphere + dark halo = atom. '
        '<b>Next:</b> %s</small></div>' %
        (tone, _pick_text(), S.get('_last_pick_message') or 'selected', _PICK_HINT[pick_action.value]))
_render_last_pick_status()
viewer_legend = W.HTML('<small><b>selection key:</b> '
                       '<span style="color:#b45309">□ residue frame / ◉ last atom</span> · '
                       '<span style="color:#d000d0">● center</span> · '
                       '<span style="color:#d11">● scan</span> · '
                       '<span style="color:#0e7490">● frozen/restraint</span> · '
                       '<span style="color:#2563eb">● measure</span></small>')
viewer_more = _collapsible('More view controls', W.VBox([
    W.HBox([btn_reset, btn_zoomsel, btn_zoompick, btn_clear_pick],
           layout=W.Layout(flex_flow='row wrap')),
    W.HBox([exact_atom, exact_atom_btn], layout=W.Layout(flex_flow='row wrap')),
    exact_atom_msg, W.HBox([cb_surf, cb_spin])]))
viewer_col = W.VBox([
    W.HBox([view_input, pick_action]), view_input_note, view_controls, viewer_more,
    viewer_status, viewer_out, last_pick_html, viewer_legend])
viewer_col.add_class('rxviewer')
# Compute commands operate on the current model. Prepare a cluster here when a
# full protein was loaded; `all` can instead perform the same extraction itself.
extract_msg = W.HTML()
b_extract = W.Button(description='Extract cluster model', icon='scissors',
                     button_style='info', layout=W.Layout(width='220px'),
                     tooltip='Run `extract` with the picked center (-c) and radius (-r), then use its '
                             'output as the input for opt / tsopt / freq / irc / path-opt.')
prep_radius = W.FloatText(value=0.0, description='radius Å',
                          style={'description_width': '58px'},
                          layout=W.Layout(width='115px'),
                          tooltip='Extraction radius; 0 keeps the CLI default.')
def _do_extract(_):
    try:
        if not S['inputs']: raise ValueError('Load a structure in the Input tab first.')
        cen = _center_cli_selectors()
        if not cen: raise ValueError('Pick at least one center residue in the 3D view (-c is required).')
        r = float(prep_radius.value or 0.0)
        model_dir = _runtime_path('prepared_models')
        os.makedirs(model_dir, exist_ok=True)
        outs = [_unique_path(os.path.join(model_dir, '%02d_%s_cluster.pdb' % (i + 1, Path(path).stem)))
                for i, path in enumerate(S['inputs'])]
        cmd = [CLI, 'extract', '-i', *S['inputs'], '-o', *outs, '-c', ','.join(cen)]
        if r and r > 0: cmd += ['-r', str(r)]
        lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
        if lc: cmd += ['-l', lc]
        extract_msg.value = '<small>running: <code>%s</code></small>' % ' '.join(cmd)
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0 or not all(os.path.exists(path) for path in outs):
            raise RuntimeError((p.stdout + p.stderr)[-400:] or 'extract failed')
        previous = {'inputs': list(S['inputs']), 'mode': S['mode'], 'parm': S.get('parm')}
        load_pdb(outs, None, keep_subcmd=True)
        # load_pdb deliberately invalidates an older preparation transaction;
        # install this new transaction only after the new inputs finish loading.
        S['_pre_extract'] = previous
        b_revert.layout.display = ''
        input_msg.value = '✅ <b>%s</b> (extracted cluster model%s)' % (
            ', '.join(outs), 's' if len(outs) != 1 else '')
        extract_msg.value = ('<small style="color:#181">✅ %d model(s) prepared outside the run output; '
                             'they are now the ordered inputs.</small>' % len(outs))
    except Exception as e:
        extract_msg.value = '<small style="color:#a00">extract failed: %s</small>' % e
b_extract.on_click(_do_extract)
b_revert = W.Button(description='Revert', icon='undo', layout=W.Layout(width='110px'),
                    tooltip='Restore the structure that was loaded before the last extraction.')
b_revert.layout.display = 'none'
def _do_revert(_):
    prev = S.get('_pre_extract')
    if not prev: return
    load_pdb(list(prev['inputs']), prev.get('parm'), mode=prev['mode'], keep_subcmd=True)
    input_msg.value = '↩ reverted to <b>%s</b>' % ', '.join(prev['inputs'])
    extract_msg.value = '<small>reverted to the pre-extraction structure</small>'
b_revert.on_click(_do_revert)
extract_panel = W.VBox([
    _hdr('<b>Cluster model</b> <small>— prepare a full protein for this compute</small>',
         'all can do this internally. For a standalone compute, pick the center residues above, '
         'set the extraction radius here, then press Extract. '
         'Revert puts the original structure back.'),
    W.HBox([prep_radius, b_extract, b_revert], layout=W.Layout(flex_flow='row wrap')),
    extract_msg])
extract_panel.add_class('rxcard')
selection_inspector = W.VBox([
    summary_html, chips_box,
    center_panel, charge_panel, scan_panel, extract_panel])
selection_inspector.add_class('rxinspector')
workspace = W.HBox([viewer_col, selection_inspector]); workspace.add_class('rxworkspace')
selection_help = W.HTML(_sel_help)
selection_route = W.HTML()
selection_route.layout.display = 'none'
def _sync_select_availability():
    has_files = bool(S.get('inputs'))
    is_small = S.get('mode') == 'small'
    available = has_files and not is_small
    selection_help.layout.display = '' if available else 'none'
    workspace.layout.display = '' if available else 'none'
    selection_route.layout.display = 'none' if available else ''
    selection_route.value = (
        '<div class="rxcard"><b>Small-molecule input</b><br><small>No residue selection is needed. '
        'Continue to Workflow; use a PDB/mmCIF input when a workflow needs 3D atom picking.</small></div>'
        if is_small else
        '<div class="rxcard"><b>No structure to select</b><br><small>Load a PDB/mmCIF in Input first.</small></div>')
select_box = W.VBox([selection_help, selection_route, workspace, freeze_acc])

# ============================================================== OPTIONS tab
cb_advsub = W.Checkbox(value=False, description='show advanced subcommands (utilities)', indent=False)
dd_subcmd = W.Dropdown(options=BASIC_SUBS, value='all', description='subcommand',
                       style={'description_width': 'initial'}, layout=W.Layout(width='280px'))
subreq = W.HTML(); subcmd_note = W.HTML(); subguide = W.HTML()
def _on_sub(_):
    sub = dd_subcmd.value; S['subcmd'] = sub
    subreq.value = '<small>📥 needs: %s</small>' % SUBREQ.get(sub, '(advanced — complete in the command line)')
    subcmd_note.value = ('' if (sub in COMPUTE or sub == 'extract') else
                         '<small style="color:#166534">utility template filled from the current input; review it below</small>'
                         if sub in AUTOFILL_UTILS else
                         '<small style="color:#7c5c00">utility subcommand — finish it in the command line below</small>')
    goal, read = GUIDE.get(sub, ('Command-line utility', 'Inspect the command help and generated files.'))
    subguide.value = ('<div style="background:#f8fafc;border-left:4px solid #2563eb;padding:7px 10px;">'
                      '<b>%s</b><br><small><b>How to judge it:</b> %s</small></div>' % (goal, read))
    _rsp = globals().get('_render_scan_panel')
    if _rsp is not None: _rsp()                          # scan vs scan2d/3d builder
    _scc = globals().get('_sync_capability_controls')
    if _scc is not None: _scc()
    refresh()
dd_subcmd.observe(_on_sub, names='value')
def _on_advsub(_):
    keep = dd_subcmd.value
    dd_subcmd.options = (SUBS if cb_advsub.value else BASIC_SUBS)
    dd_subcmd.value = keep if keep in dd_subcmd.options else 'all'
cb_advsub.observe(_on_advsub, names='value')
_on_sub(None)   # initialise the requirement hint for the default subcommand
all_mode = W.ToggleButtons(
    options=[('MEP (≥2 files)', 'mep'), ('Scan (1 input)', 'scan'), ('TS-only (1 TS)', 'tsonly')],
    value='mep', description='all mode')
_ALL_MODE_STATE = {'last': 'mep', 'tsopt_before_tsonly': False}
def _on_mode(change):
    old = change.get('old', _ALL_MODE_STATE['last']) if isinstance(change, dict) else _ALL_MODE_STATE['last']
    new = all_mode.value
    if new == 'tsonly' and old != 'tsonly':
        _ALL_MODE_STATE['tsopt_before_tsonly'] = bool(w_ts.value)
        w_ts.value = True
    elif old == 'tsonly' and new != 'tsonly':
        w_ts.value = _ALL_MODE_STATE['tsopt_before_tsonly']
    _ALL_MODE_STATE['last'] = new
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
all_mode.observe(_on_mode, names='value')

_bk0 = BACKEND if BACKEND in MODELS else 'mace'
dd_backend = W.Dropdown(options=['mace', 'uma', 'orb'], value=_bk0,
                        description='-b backend', style={'description_width': 'initial'},
                        layout=W.Layout(width='170px'))
dd_model = W.Dropdown(options=MODELS[_bk0], value=S.get('model', DEFAULT_MODEL[_bk0]),
                      description='model', style={'description_width': 'initial'},
                      layout=W.Layout(width='230px'))
def _bk(_):
    changed = S.get('backend') != dd_backend.value
    S['backend'] = dd_backend.value
    dd_model.options = MODELS[dd_backend.value]
    dd_model.value = DEFAULT_MODEL[dd_backend.value]
    S['model'] = dd_model.value
    if changed: _invalidate_last_run('Backend changed; validate and run again.')
    refresh()
def _mdl(_):
    changed = S.get('model') != dd_model.value
    S['model'] = dd_model.value
    if changed: _invalidate_last_run('Backend model changed; validate and run again.')
    refresh()
dd_backend.observe(_bk, names='value'); dd_model.observe(_mdl, names='value')
w_ts = W.Checkbox(value=False, description='--tsopt (TS + IRC)', indent=False)
w_th = W.Checkbox(value=False, description='--thermo (freq + ΔG)', indent=False)
w_out = W.Text(value='result', description='out dir', layout=W.Layout(width='260px'))
w_reuse = W.Checkbox(value=False, description='reuse non-empty out dir', indent=False,
                     tooltip='Enable only to resume or intentionally add to an existing output directory.')
output_note = W.HTML()
w_q = W.IntText(value=0, description='system charge (-q)', style={'description_width': 'initial'},
                layout=W.Layout(width='220px'))
w_charge_ok = W.Checkbox(value=False, description='charge verified', indent=False,
                         tooltip='Confirm that -q is the net charge of the complete input system.')
def _render_output_note():
    out = _effective_out_dir() if '_effective_out_dir' in globals() else (w_out.value or 'result')
    occupied = os.path.isdir(out) and bool(os.listdir(out))
    output_note.value = ('<small style="color:#a60">⚠ non-empty output exists; enable reuse to run into it</small>'
                         if occupied and not w_reuse.value else '')
def _sync_run(_=None):
    S['tsopt'] = w_ts.value; S['thermo'] = w_th.value
    S['out_dir'] = w_out.value or 'result'; S['charge'] = w_q.value
    sync = globals().get('_sync_capability_controls')
    if sync is not None: sync()
    refresh()
def _sync_charge(_=None):
    S['charge'] = w_q.value; S['charge_explicit'] = True
    if not w_charge_ok.value: w_charge_ok.value = True
    refresh()
def _sync_charge_ok(change):
    S['charge_explicit'] = bool(change['new']); refresh()
for w in (w_ts, w_th, w_out, w_reuse): w.observe(_sync_run, names='value')
w_q.observe(_sync_charge, names='value')
w_charge_ok.observe(_sync_charge_ok, names='value')

# Advanced flags are derived from this repository's live Click command.
adv_mult = W.IntText(value=1, description='-m mult', style={'description_width': 'initial'}, layout=W.Layout(width='160px'))
adv_prec = W.Dropdown(options=['auto', 'fp32', 'fp64'], value='auto', description='--precision', style={'description_width': 'initial'}, layout=W.Layout(width='190px'))
adv_det = W.Checkbox(value=False, description='--deterministic', indent=False)
adv_mep = W.Dropdown(options=['(default)', 'gsm'] + (['dmf'] if DMF_READY else []), value='(default)', description='--mep-mode', style={'description_width': 'initial'}, layout=W.Layout(width='200px'))
adv_dmf = W.Dropdown(options=['(default)', 'gpu', 'cpu'], value='(default)', description='--dmf-backend', style={'description_width': 'initial'}, layout=W.Layout(width='220px'))
adv_thresh = W.Dropdown(options=['(default)', 'gau_loose', 'gau', 'gau_tight', 'gau_vtight', 'baker'], value='(default)', description='--thresh', style={'description_width': 'initial'}, layout=W.Layout(width='230px'))
adv_radius = W.FloatText(value=0.0, description='-r radius Å (0=CLI default)', style={'description_width': 'initial'}, layout=W.Layout(width='270px'))
def _sync_radius_widgets(change):
    target = adv_radius if change['owner'] is prep_radius else prep_radius
    if target.value != change['new']: target.value = change['new']
prep_radius.observe(_sync_radius_widgets, names='value')
adv_radius.observe(_sync_radius_widgets, names='value')
adv_dft = W.Checkbox(value=False, description='--dft (DFT single-point)', indent=False)
adv_dftfb = W.Text(value='', description='--dft-func-basis', placeholder='wb97x-d3,def2-tzvp', style={'description_width': 'initial'}, layout=W.Layout(width='330px'))
adv_flatten = W.Checkbox(value=False, description='--flatten', indent=False,
                         tooltip='Opt-in Cartesian flattening of near-zero modes; helps a stubborn TS converge.')
adv_refine = W.Checkbox(value=False, description='--refine-path', indent=False,
                        tooltip='all only: switch MEP from single-segment path-opt to recursive multi-segment path-search.')
adv_maxcyc = W.IntText(value=0, description='--max-cycles (0=default)', style={'description_width': 'initial'},
                       layout=W.Layout(width='250px'))

adv_search = W.Text(value='', placeholder='filter flags…', description='search',
                    style={'description_width': 'initial'}, layout=W.Layout(width='360px', max_width='100%'))
adv_count = W.HTML()
adv_rows_box = W.VBox(layout=W.Layout(max_height='420px', overflow='auto', width='100%',
                                      border='1px solid #e2e8f0', padding='4px'))

def _option_help(name, sub='all', fallback='See the command help for this option.'):
    for param in _advanced_options(sub):
        if getattr(param, 'name', None) == name and getattr(param, 'help', None): return param.help
    return fallback

def _set_advanced_override(sub, name, value):
    by_sub = S.setdefault('advanced_overrides', {}).setdefault(sub, {})
    if value in (None, ''): by_sub.pop(name, None)
    else: by_sub[name] = value
    refresh()

def _advanced_widget(sub, param):
    flag = _advanced_flag(param); saved = S.setdefault('advanced_overrides', {}).setdefault(sub, {}).get(param.name)
    is_bool = param.is_bool_flag or isinstance(param.type, click.types.BoolParamType)
    if is_bool:
        widget = W.Dropdown(options=[('CLI default', None), ('on', True), ('off', False)], value=saved,
                            description=flag, style={'description_width': 'initial'},
                            layout=W.Layout(width='360px', max_width='92%'))
    elif isinstance(param.type, click.Choice):
        choices = list(param.type.choices)
        widget = W.Dropdown(options=[('CLI default', None)] + [(str(choice), choice) for choice in choices],
                            value=saved, description=flag, style={'description_width': 'initial'},
                            layout=W.Layout(width='420px', max_width='92%'))
    else:
        default = param.default
        placeholder = 'CLI default%s' % ((': %s' % default) if default not in (None, '', ()) else '')
        if param.multiple: placeholder += ' · quote each repeated value'
        widget = W.Text(value='' if saved is None else str(saved), description=flag, placeholder=placeholder,
                        style={'description_width': 'initial'}, layout=W.Layout(width='520px', max_width='92%'))
    widget.observe(lambda change, s=sub, n=param.name: _set_advanced_override(s, n, change['new']), names='value')
    row = _flag_row(widget, param.help or 'No additional help is supplied by this command.')
    row._rx_search = ('%s %s %s' % (flag, param.name, param.help or '')).lower()
    return row

def _render_advanced_rows(_=None):
    sub = _wv('dd_subcmd', S.get('subcmd', 'all')) or 'all'
    query = adv_search.value.strip().lower()
    rows = [_advanced_widget(sub, param) for param in _advanced_options(sub)
            if _advanced_status(sub, param) == 'rendered' and _advanced_semantic_applicable(sub, param.name)]
    shown = [row for row in rows if not query or query in row._rx_search]
    adv_rows_box.children = shown or [W.HTML('<small>No matching advanced flags.</small>')]
    accounted = _advanced_coverage(sub)
    adv_count.value = ('<small><b>%d/%d editable shown</b> · %d CLI options accounted for</small>' %
                       (len(shown), len(rows), len(accounted)))

adv_search.observe(_render_advanced_rows, names='value')

def _sync_capability_controls(_=None):
    sub = dd_subcmd.value
    adv_mep.disabled = sub not in TOOL_CAPABILITIES['mep_mode']
    adv_thresh.disabled = sub not in TOOL_CAPABILITIES['threshold']
    _set_flag_visible(adv_dft, sub == 'all' and DFT_READY)
    adv_radius.disabled = sub not in FLAG_SUBS['adv_radius']
    _set_flag_visible(adv_radius, sub in FLAG_SUBS['adv_radius'])
    # Every surfaced flag is shown only where the CLI accepts it (FLAG_SUBS).
    for _wn, _subs in FLAG_SUBS.items():
        _w = globals().get(_wn)
        if _w is not None and _wn not in ('adv_dft', 'adv_radius'):
            _set_flag_visible(_w, sub in _subs)
    _set_flag_visible(adv_dmf, sub in FLAG_SUBS['adv_dmf'] and adv_mep.value == 'dmf')
    mode = _wv('all_mode', 'mep') if sub == 'all' else None
    if sub == 'all':
        path_active = mode != 'tsonly'
        for _w in (adv_mep, adv_thresh, adv_refine, adv_maxcyc):
            _set_flag_visible(_w, path_active)
        _set_flag_visible(adv_dmf, path_active and adv_mep.value == 'dmf')
        if mode == 'tsonly' and not w_ts.value: w_ts.value = True
        w_ts.disabled = mode == 'tsonly'
        _set_flag_visible(adv_flatten, mode == 'tsonly' or w_ts.value)
    else:
        w_ts.disabled = False
    # "all workflow" mode and the depth switches only exist on `all`; hide them for
    # every other subcommand instead of showing controls the command cannot use.
    for _name in ('all_mode_box', 'depth_box'):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if sub == 'all' else 'none'
    # Select-tab panels follow the same table.
    _panels = SPEC.get(sub, {}).get('panels', ())
    if sub == 'all' and _wv('all_mode', 'mep') != 'scan':
        _panels = tuple(panel for panel in _panels if panel != 'scan')
    _structure_selectable = bool(S.get('inputs')) and S.get('mode') != 'small'
    _prep_active = sub in _PREP_SUBS and _structure_selectable
    _select_panels = set(_panels) if _structure_selectable else set()
    if _prep_active: _select_panels.add('center')
    for _name, _key in (('scan_panel', 'scan'), ('freeze_acc', 'freeze'),
                        ('center_panel', 'center'), ('charge_panel', 'center')):
        _b = globals().get(_name)
        if _b is not None: _b.layout.display = '' if _key in _select_panels else 'none'
    _extract_panel = globals().get('extract_panel')
    if _extract_panel is not None: _extract_panel.layout.display = '' if _prep_active else 'none'
    _sync_select_availability()
    _current_pick = pick_action.value
    _pick_options = [(label, value) for label, value, panel in _PICK_ACTIONS
                     if panel is None or panel in _select_panels]
    pick_action.options = _pick_options
    _valid_picks = {value for _label, value in _pick_options}
    if _current_pick in _valid_picks: pick_action.value = _current_pick
    elif _pick_options: pick_action.value = _pick_options[0][1]
    _kb = globals().get('key_opts_box')
    if _kb is not None:
        _kb.layout.display = '' if any(sub in s for n, s in FLAG_SUBS.items()
                                       if n in ('adv_mep', 'adv_flatten', 'adv_thresh', 'adv_maxcyc')) else 'none'
    _bb = globals().get('backend_box')
    if _bb is not None: _bb.layout.display = '' if sub in MLIP_COMPUTE else 'none'
    _oh = globals().get('outputs_html')
    if _oh is not None:
        _oh.value = ('<small><b>produces:</b> %s</small>'
                     % ' · '.join('<code>%s</code>' % o for o in SPEC.get(sub, {}).get('out', ())))
    adv_dftfb.description = '--func-basis' if sub == 'dft' else '--dft-func-basis'
    _dftfb_applicable = DFT_READY and (sub == 'dft' or (sub == 'all' and adv_dft.value))
    adv_dftfb.disabled = not _dftfb_applicable
    _set_flag_visible(adv_dftfb, _dftfb_applicable)
    label = globals().get('depth_label')
    if label is not None:
        label.value = ('<b>Depth</b> · TS optimization is required in TS-only mode:'
                       if sub == 'all' and mode == 'tsonly' else
                       '<b>Depth</b> <small>· optional TS / thermo / DFT stages</small>')
    render_advanced = globals().get('_render_advanced_rows')
    if render_advanced is not None: render_advanced()

for w in (adv_mult, adv_prec, adv_det, adv_mep, adv_dmf, adv_thresh, adv_radius, adv_dft, adv_dftfb,
          adv_flatten, adv_refine, adv_maxcyc):
    w.observe(refresh, names='value')
adv_dft.observe(_sync_capability_controls, names='value')
adv_mep.observe(_sync_capability_controls, names='value')
_sync_capability_controls()
adv_box = W.VBox([W.HBox([adv_search, adv_count]), adv_rows_box])
adv_acc = _collapsible('⚙️ Additional flags — every CLI option accounted for', adv_box)
all_mode_box = W.VBox([W.HTML('<b>all workflow</b> <small>(MEP / scan with scan-lists / TS-only)</small>'), all_mode])
depth_label = W.HTML('<b>Depth</b> (off = fast MEP only):')
depth_box = W.VBox([depth_label, W.HBox([
    _flag_row(w_ts, _option_help('tsopt', fallback='Run TS optimization and IRC after path generation.')),
    _flag_row(w_th, _option_help('thermo', fallback='Run frequency thermochemistry for the refined structure.')),
    _flag_row(adv_dft, _option_help('dft', fallback='Add a DFT single-point correction.'))])])
outputs_html = W.HTML()
key_opts_box = W.VBox([
    _hdr('<b>Key options</b> <small>(shown only where the subcommand accepts them)</small>',
         'Frequently changed CLI options; the complete selected-command list is below.'),
    W.HBox([_flag_row(adv_mep, _option_help('mep_mode')), _flag_row(adv_dmf, _option_help('dmf_backend')),
            _flag_row(adv_thresh, _option_help('thresh')), _flag_row(adv_maxcyc, _option_help('max_cycles'))]),
    W.HBox([_flag_row(adv_prec, _option_help('precision')), _flag_row(adv_det, _option_help('deterministic')),
            _flag_row(adv_flatten, _option_help('flatten')), _flag_row(adv_refine, _option_help('refine_path'))]),
    W.HBox([_flag_row(adv_mult, _option_help('spin')), _flag_row(adv_radius, _option_help('radius')),
            _flag_row(adv_dftfb, _option_help('dft_func_basis', fallback='DFT functional,basis override.'))])])
_missing_features = []
if not DFT_READY: _missing_features.append('DFT is hidden (rerun Setup with install_dft enabled)')
if not DMF_READY: _missing_features.append('DMF is hidden (pydmf + cyipopt are not installed)')
dependency_note = W.HTML('<small style="color:#92400e">%s</small>' % ' · '.join(_missing_features)
                         if _missing_features else '')
backend_box = W.VBox([
    W.HTML('<b>MLIP backend &amp; model</b> <small>(MACE = no login; UMA needs HF login. '
           'Changing backend here does not install it: rerun Setup first. <code>auto</code> precision '
           'means UMA fp32 and MACE fp64.)</small>'),
    W.HBox([dd_backend, dd_model]), dependency_note])
options_box = W.VBox([
    _hdr('<b>Subcommand</b> <small>(all = end-to-end)</small>',
         'all runs the whole pipeline (extract -> MEP -> optional TS/IRC/thermo/DFT). '
         'The other entries run a single stage on an already-prepared structure.'),
    W.HBox([dd_subcmd, cb_advsub]), subreq, outputs_html, subguide, subcmd_note,
    all_mode_box,
    W.HTML('<hr style="margin:6px 0">'),
    backend_box,
    depth_box,
    W.HTML('<hr style="margin:6px 0">'), key_opts_box,
    W.HTML('<hr style="margin:6px 0">'),
    W.HBox([w_q, w_charge_ok, w_out, w_reuse], layout=W.Layout(flex_flow='row wrap')), output_note,
    W.HTML('<hr style="margin:6px 0">'), adv_acc])
_sync_capability_controls()   # apply all-only visibility now that the boxes exist

# ============================================================== RESULTS tab
res_out = W.Output()
traj_out = W.Output(layout={'width': '100%', 'max_width': '430px', 'min_width': '0', 'flex': '1 1 340px'})
plot_out = W.Output(layout={'width': '100%', 'max_width': '460px', 'min_width': '0', 'flex': '1 1 340px'})
result_context, traj_label, frame_state, trajectory_intro = W.HTML(), W.HTML(), W.HTML(), W.HTML()
artifact_out = W.Output()
artifact_choice = W.Dropdown(options=[], description='artifact preview',
                             style={'description_width': 'initial'},
                             layout=W.Layout(width='620px', max_width='100%'))
traj_choice = W.Dropdown(options=[], description='trajectory',
                         style={'description_width': 'initial'},
                         layout=W.Layout(width='620px', max_width='100%'))
_result_pick_guard = {'active': False}
frame_slider = W.IntSlider(min=0, max=0, value=0, description='frame', continuous_update=False,
                           readout=True, disabled=True,
                           layout=W.Layout(width='100%', max_width='560px'))
_TRAJ = {'frames': [], 'energies': [], 'path': None, 'semantics': {}}

def _parse_trj(path):
    frames, energies = [], []
    lines = open(path).read().splitlines()
    i, n_lines = 0, len(lines)
    while i < n_lines:
        if not lines[i].strip():
            i += 1; continue
        try: n = int(lines[i].strip())
        except ValueError:
            i += 1; continue
        comment = lines[i + 1] if i + 1 < n_lines else ''
        frames.append('\n'.join(lines[i:i + 2 + n]))
        try: energies.append(float(comment.strip().split()[0]))
        except (ValueError, IndexError): energies.append(None)
        i += 2 + n
    return frames, energies

def _rel_kcal():
    es = _TRAJ['energies']
    base = es[0] if (es and es[0] is not None) else next((e for e in es if e is not None), None)
    if base is None: return None
    return [((e - base) * 627.509) if e is not None else None for e in es]

def _trajectory_semantics(sub=None, path=''):
    """Describe a trajectory without assigning reaction semantics to generic runs."""
    sub = str(sub or S.get('_last_subcmd') or S.get('subcmd') or '').lower()
    name = os.path.basename(str(path or '')).lower()
    if 'scan' in name or sub in ('scan', 'scan2d', 'scan3d'):
        return {'title': 'Scan trajectory', 'start': 'scan start', 'end': 'scan end',
                'x': 'scan frame', 'extrema': False}
    if 'irc' in name or sub == 'irc':
        return {'title': 'IRC trajectory', 'start': 'IRC start', 'end': 'IRC end',
                'x': 'IRC frame', 'extrema': False}
    if 'tsopt' in name or sub == 'tsopt':
        return {'title': 'TS-refinement trajectory', 'start': 'initial candidate', 'end': 'refined candidate',
                'x': 'optimization step', 'extrema': False}
    if ('mep' in name or 'path' in name or 'segment' in name or
            sub in ('all', 'path-opt', 'path-search')):
        return {'title': 'Reaction-path trajectory', 'start': 'R', 'end': 'P',
                'x': 'image', 'extrema': True}
    if 'opt' in name or sub == 'opt':
        return {'title': 'Optimization trajectory', 'start': 'initial', 'end': 'optimized',
                'x': 'optimization step', 'extrema': False}
    return {'title': 'Trajectory', 'start': 'initial', 'end': 'final',
            'x': 'frame', 'extrema': False}

def _stationary(ys, semantics=None):
    """Energy-only profile candidates; no frequency or IRC certification is implied."""
    semantics = semantics or _TRAJ.get('semantics') or _trajectory_semantics()
    n = len(ys)
    if not n: return []
    if n < 2: return [(0, semantics['start'])]
    pts = [(0, semantics['start'])]
    if semantics.get('extrema'):
        for k in range(1, n - 1):
            a, b, c = ys[k - 1], ys[k], ys[k + 1]
            if not (a == a and b == b and c == c): continue                   # skip NaN
            if b > a and b >= c and (b - min(a, c)) > 0.3: pts.append((k, 'peak candidate'))
            elif b < a and b <= c and (max(a, c) - b) > 0.3: pts.append((k, 'minimum candidate'))
    pts.append((n - 1, semantics['end']))
    return pts

def _show_frame(i):
    fr = _TRAJ['frames']
    if not fr: return
    i = max(0, min(i, len(fr) - 1))
    with traj_out:
        clear_output()
        v = py3Dmol.view(width='100%', height=320)
        v.addModel(fr[i], 'xyz'); v.setStyle({'stick': {}, 'sphere': {'scale': 0.3}})
        v.zoomTo(); v.show()
    with plot_out:
        clear_output()
        rk = _rel_kcal()
        semantics = _TRAJ.get('semantics') or _trajectory_semantics(path=_TRAJ.get('path'))
        if rk is None:
            frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                                 '<b>Frame %d/%d</b> · energy unavailable</div>' % (i + 1, len(fr)))
            print('(no per-frame energies in the trajectory)'); return
        import matplotlib.pyplot as plt
        plt.rcParams.update({'font.size': 10, 'figure.dpi': 120, 'savefig.dpi': 120,
                             'axes.edgecolor': '#cbd5e1', 'axes.linewidth': 0.9,
                             'xtick.color': '#64748b', 'ytick.color': '#64748b',
                             'axes.labelcolor': '#475569', 'text.color': '#33404d'})
        xs = list(range(len(rk))); ys = [y if y is not None else float('nan') for y in rk]
        stat = _stationary(ys, semantics)
        current_label = dict(stat).get(i, '')
        LINE, HALO, EMPH, MUTE = '#4C72B0', '#F4A259', '#E07B39', '#cfd6dd'
        fig, ax = plt.subplots(figsize=(4.7, 3.2))
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, axis='y', color='#edf0f4', lw=0.9, zorder=0)
        for (si, lab) in stat:                                               # horizontal state levels
            if not (ys[si] == ys[si]): continue
            on = (si == i)
            ax.axhline(ys[si], color=(EMPH if on else MUTE), lw=(2.4 if on else 1.0),
                       ls=('-' if on else (0, (4, 3))), alpha=(0.95 if on else 0.85), zorder=1)
            ax.text(xs[-1], ys[si], ' ' + lab, va='center', ha='left', fontsize=9,
                    color=('#b5601f' if on else '#9aa3ad'), fontweight=('bold' if on else 'normal'))
        ax.plot(xs, ys, '-', color=LINE, lw=2.0, zorder=2)
        ax.plot(xs, ys, 'o', color=LINE, ms=3.6, mec='white', mew=0.5, zorder=3)
        if ys[i] == ys[i]:                                                   # halo + solid center (no harsh dot)
            ax.plot([i], [ys[i]], 'o', ms=22, color=HALO, alpha=0.30, mec='none', zorder=4)
            ax.plot([i], [ys[i]], 'o', ms=9.5, color=EMPH, mec='white', mew=1.0, zorder=5)
        ax.set_xlabel(semantics['x'], fontsize=10); ax.set_ylabel('ΔE  (kcal/mol)', fontsize=10)
        energy_text = ('ΔE = %.1f kcal/mol' % ys[i]) if ys[i] == ys[i] else 'ΔE unavailable'
        label_text = (' · ' + current_label) if current_label else ''
        ax.set_title('frame %d / %d%s   ·   %s' % (i + 1, len(rk), label_text, energy_text),
                     fontsize=10, color='#33404d')
        frame_state.value = ('<div role="status" aria-live="polite" aria-atomic="true">'
                             '<b>Frame %d/%d</b>%s · %s</div>' %
                             (i + 1, len(rk), label_text, energy_text))
        ax.margins(x=0.10, y=0.16)
        fig.tight_layout(); plt.show(); plt.close(fig)

frame_slider.observe(lambda ch: _show_frame(frame_slider.value), names='value')

def _summary_html(summary_path=None):
    if not summary_path or not os.path.exists(summary_path): return ''
    try:
        with open(summary_path) as fh: d = json.load(fh)
    except (OSError, ValueError) as exc:
        return '<div role="alert" style="color:#991b1b">Could not parse <code>%s</code>: %s</div>' % (html.escape(os.path.basename(summary_path)), html.escape(str(exc)))
    def val(x):
        try: return '%.1f' % float(x)
        except (TypeError, ValueError): return '—'
    post = {row.get('index'): row for row in (d.get('post_segments') or [])}
    rows = ''
    for k, seg in enumerate(d.get('segments', []) or []):
        idx = seg.get('index', k + 1); ps = post.get(idx, {})
        mlip = ps.get('mlip') if isinstance(ps.get('mlip'), dict) else {}
        gibbs = ps.get('gibbs_mlip') if isinstance(ps.get('gibbs_mlip'), dict) else {}
        refined = mlip.get('barrier_kcal') is not None
        barrier = mlip.get('barrier_kcal') if refined else seg.get('barrier_kcal')
        delta = mlip.get('delta_kcal') if refined else seg.get('delta_kcal')
        method = 'TSOPT+IRC' if refined else 'MEP band'
        imag = (ps.get('ts_imag') or {}).get('n_imag')
        rows += ('<tr><td>seg %02d</td><td>%s</td><td align=right>%s</td>'
                 '<td align=right>%s</td><td align=right>%s</td><td align=right>%s</td></tr>') % (
                     idx, method, val(barrier), val(gibbs.get('barrier_kcal')), val(delta),
                     '—' if imag is None else str(imag))
    rls = d.get('rate_limiting_step') or {}; rl = rls.get('barrier_kcal')
    table = ('<table style="border-collapse:collapse;" border=1 cellpadding=5>'
         '<caption style="text-align:left;font-weight:600">Current-run segment summary (kcal/mol)</caption>'
         '<tr><th scope="col">segment</th><th scope="col">source</th><th scope="col">ΔE‡</th><th scope="col">ΔG‡</th>'
         '<th scope="col">ΔE</th><th scope="col">n<sub>imag</sub></th></tr>%s</table>' % rows)
    h = '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table
    if rl is not None:
        h += ('<br><b style="font-size:15px;">⚡ rate-limiting barrier: %s kcal/mol</b> '
              '<small>(%s)</small>' % (val(rl), rls.get('method') or 'method not recorded'))
    h += '<br><small>status: %s · images: %s · backend/model: %s / %s</small>' % (
        d.get('status'), d.get('n_images'), d.get('mlip_backend'), d.get('mlip_model'))
    return h

def _current_run_files(out):
    """Return only files changed by the last GUI-launched run for this root."""
    if not S.get('_last_out_dir') or os.path.abspath(out) != os.path.abspath(S['_last_out_dir']):
        return []
    return sorted(path for path in S.get('_last_files', []) if os.path.isfile(path))

def _select_status_json(current, sub):
    """Prefer the aggregate summary for composite workflows, otherwise a leaf result."""
    order = ('summary.json', 'result.json') if sub in ('all', 'path-search') else ('result.json', 'summary.json')
    for name in order:
        candidates = sorted((p for p in current if os.path.basename(p) == name),
                            key=lambda p: (p.count(os.sep), p))
        if candidates: return candidates[0]
    return None

def _result_context_html(out):
    sub = S.get('_last_subcmd') or S.get('subcmd') or 'all'
    goal, read = GUIDE.get(sub, (sub, 'Inspect the terminal status and generated files.'))
    expected = SPEC.get(sub, {}).get('out', ())
    current = _current_run_files(out)
    status_bits = []
    status_json = _select_status_json(current, sub)
    if status_json:
        try:
            with open(status_json) as fh: data = json.load(fh)
            for key in ('execution_status', 'scientific_status', 'status', 'converged',
                        'n_imaginary_modes', 'n_imag'):
                if key in data: status_bits.append('%s=<b>%s</b>' % (key, html.escape(str(data[key]))))
        except Exception:
            status_bits.append('%s could not be parsed' % os.path.basename(status_json))
    manifest = S.get('_last_manifest') or {}
    if manifest:
        status_bits.insert(0, 'run=<b>%s</b>%s' % (
            html.escape(str(manifest.get('status') or 'recorded')),
            (' (exit %s)' % html.escape(str(manifest['exit_code']))) if manifest.get('exit_code') is not None else ''))
    rels = [os.path.relpath(path, out) for path in current]
    shown = rels[:16]
    stdout_only = bool((S.get('_last_manifest') or {}).get('stdout_only'))
    artifacts = ('(standard output; see the run log)' if stdout_only else
                 (' · '.join('<code>%s</code>' % html.escape(p) for p in shown)
                  or '(no current-run files recorded)'))
    if len(rels) > len(shown): artifacts += ' · … +%d files' % (len(rels) - len(shown))
    return ('<div style="border:1px solid #dbeafe;border-radius:11px;padding:9px 11px;background:#f8fbff;">'
            '<b>%s · %s</b><br><small><b>Read it this way:</b> %s<br>'
            '<b>Expected:</b> %s<br><b>Machine status:</b> %s<br>'
            '<b>Current run only:</b> %s</small></div>'
            % (html.escape(sub), html.escape(goal), html.escape(read),
               ' · '.join('<code>%s</code>' % html.escape(x) for x in expected) or 'see command output',
               ' · '.join(status_bits) or 'no current root result/summary status detected', artifacts))

_ARTIFACT_KINDS = {'.png': 'image', '.jpg': 'image', '.jpeg': 'image',
                   '.svg': 'SVG', '.html': 'interactive HTML',
                   '.csv': 'CSV table', '.pdf': 'PDF', '.json': 'JSON',
                   '.yaml': 'YAML', '.yml': 'YAML', '.txt': 'text',
                   '.log': 'text', '.out': 'text', '.md': 'text',
                   '.pdb': 'structure', '.ent': 'structure', '.cif': 'structure',
                   '.mmcif': 'structure'}
_TEXT_PREVIEW_LIMIT = 512 * 1024
_STRUCTURE_PREVIEW_LIMIT = 5 * 1024 * 1024

def _artifact_kind(path):
    path = Path(path)
    if path.suffix.lower() == '.xyz':
        return None if 'trj' in path.name.lower() else 'structure'
    return _ARTIFACT_KINDS.get(path.suffix.lower())

def _csv_preview_html(path, max_rows=50, max_cols=20):
    rows = []
    with open(path, newline='', encoding='utf-8', errors='replace') as fh:
        reader = csv.reader(fh)
        for index, row in enumerate(reader):
            if index > max_rows: break
            rows.append(row[:max_cols])
    if not rows: return '<i>empty CSV</i>'
    head, body = rows[0], rows[1:max_rows + 1]
    table = '<table border="1" cellpadding="4" style="border-collapse:collapse;max-width:100%;">'
    table += '<thead><tr>%s</tr></thead>' % ''.join('<th>%s</th>' % html.escape(cell) for cell in head)
    table += '<tbody>%s</tbody></table>' % ''.join(
        '<tr>%s</tr>' % ''.join('<td>%s</td>' % html.escape(cell) for cell in row) for row in body)
    if len(rows) > max_rows: table += '<small>Preview truncated after %d rows.</small>' % max_rows
    return '<div style="max-width:100%%;overflow-x:auto">%s</div>' % table

def _text_preview_html(path, kind):
    size = os.path.getsize(path)
    with open(path, encoding='utf-8', errors='replace') as fh:
        source = fh.read(_TEXT_PREVIEW_LIMIT + 1)
    truncated = len(source) > _TEXT_PREVIEW_LIMIT or size > _TEXT_PREVIEW_LIMIT
    source = source[:_TEXT_PREVIEW_LIMIT]
    if kind == 'JSON':
        try: source = json.dumps(json.loads(source), indent=2, ensure_ascii=False)
        except (ValueError, TypeError): pass
    note = ('<small>Preview truncated at 512 KiB; download the result for the complete file.</small>'
            if truncated else '')
    return ('<pre style="max-height:520px;max-width:100%%;overflow:auto;white-space:pre-wrap;'
            'word-break:break-word;background:#0f172a;color:#e2e8f0;padding:10px;border-radius:8px;">%s</pre>%s'
            % (html.escape(source), note))

def _structure_preview(path):
    """Show one bounded molecular structure without treating trajectories as files."""
    size = os.path.getsize(path)
    if size > _STRUCTURE_PREVIEW_LIMIT:
        print('Structure preview skipped (%.1f MiB); download the result for the complete file.' %
              (size / 1048576.0))
        return
    suffix = Path(path).suffix.lower()
    fmt = 'pdb' if suffix in ('.pdb', '.ent') else ('cif' if suffix in ('.cif', '.mmcif') else 'xyz')
    with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
    view = py3Dmol.view(width='100%', height=380)
    options = {'keepH': True, 'altLoc': '*'} if fmt == 'pdb' else {}
    view.addModel(source, fmt, options)
    view.setStyle({}, {'cartoon': {'color': 'spectrum'},
                       'line': {'opacity': 0.35, 'colorscheme': 'default'}})
    view.addStyle({'hetflag': True}, {'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo(); view.show()

def _render_artifact(_=None):
    if _result_pick_guard['active']: return
    path = artifact_choice.value
    out = S.get('_last_out_dir') or _effective_result_root()
    with artifact_out:
        clear_output()
        if not path or not os.path.isfile(path): return
        rel = os.path.relpath(path, out)
        display(W.HTML('<div><b>Displayed artifact:</b> <code>%s</code></div>' % html.escape(rel)))
        kind = _artifact_kind(path)
        if kind == 'image':
            display(Image(filename=path)); return
        try:
            if kind == 'structure':
                _structure_preview(path)
            elif kind in ('SVG', 'interactive HTML'):
                with open(path, encoding='utf-8', errors='replace') as fh: source = fh.read()
                title = '%s: %s' % (kind, os.path.basename(path))
                display(HTML('<iframe sandbox="allow-scripts" title="%s" srcdoc="%s" '
                             'style="width:100%%;height:560px;border:1px solid #e6eaf0;'
                             'border-radius:10px;"></iframe>' %
                             (html.escape(title, quote=True), html.escape(source, quote=True))))
            elif kind == 'CSV table':
                display(HTML(_csv_preview_html(path)))
            elif kind in ('JSON', 'YAML', 'text'):
                display(HTML(_text_preview_html(path, kind)))
            elif kind == 'PDF':
                size = os.path.getsize(path)
                if size > 5 * 1024 * 1024:
                    print('PDF preview skipped (%.1f MiB); use the results/diagnostics download.' % (size / 1048576.0))
                else:
                    with open(path, 'rb') as fh: data = base64.b64encode(fh.read()).decode('ascii')
                    display(HTML('<iframe title="PDF result" src="data:application/pdf;base64,%s" '
                                 'style="width:100%%;height:560px;border:1px solid #e6eaf0;'
                                 'border-radius:10px;"></iframe>' % data))
            else:
                print('No inline preview is available for this file type.')
        except OSError as exc:
            print('could not embed artifact:', exc)

def _load_trajectory(path=None, out=None):
    path = path or traj_choice.value
    out = out or S.get('_last_out_dir') or _effective_result_root()
    _TRAJ.update(frames=[], energies=[], path=path, semantics={})
    if path and os.path.isfile(path):
        try:
            _TRAJ['frames'], _TRAJ['energies'] = _parse_trj(path)
        except OSError as exc:
            traj_label.value = '<small role="alert" style="color:#991b1b">Could not read trajectory: %s</small>' % html.escape(str(exc))
    if _TRAJ['frames']:
        semantics = _trajectory_semantics(S.get('_last_subcmd'), path)
        _TRAJ['semantics'] = semantics
        trajectory_intro.value = ('<hr style="margin:8px 0"><b>%s</b> '
                                  '<small>— move the slider to inspect structure and energy together</small>' %
                                  html.escape(semantics['title']))
        traj_label.value = '<small>current-run trajectory: <code>%s</code></small>' % html.escape(os.path.relpath(path, out))
        frame_slider.max = max(0, len(_TRAJ['frames']) - 1); frame_slider.value = 0
        frame_slider.disabled = False
        results_empty.layout.display = 'none'; trajectory_box.layout.display = ''
        _show_frame(0)
    else:
        frame_slider.max = 0; frame_slider.value = 0; frame_slider.disabled = True
        frame_state.value = ''
        trajectory_box.layout.display = 'none'
        with traj_out: clear_output()
        with plot_out: clear_output()

def _on_traj_choice(_=None):
    if not _result_pick_guard['active']: _load_trajectory()
artifact_choice.observe(_render_artifact, names='value')
traj_choice.observe(_on_traj_choice, names='value')

def _results(out=None):
    out = out or S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    result_context.value = _result_context_html(out)
    with res_out:
        clear_output()
        if not current and manifest.get('stdout_only'):
            print('This utility writes to standard output; inspect the run log below.')
        elif not current and manifest.get('status') == 'failed':
            print('Command failed (exit %s). Inspect the run log; no current-run artifact was produced.' % manifest.get('exit_code', '?'))
        elif not current and manifest.get('status') == 'cancelled':
            print('Command was cancelled (exit 130). Inspect the run log; no current-run artifact was produced.')
        elif not current and manifest:
            print('The current run finished without a file artifact. Inspect the run log and command status.')
        elif not current:
            print('No current GUI run is recorded for', out)
            print('Run a workflow first; older files in a reused directory are intentionally not shown.')
        _preview_priority = {'result.json': 0, 'result.yaml': 1, 'result.yml': 1,
                             'thermoanalysis.yaml': 2, 'frequencies_cm-1.txt': 3,
                             'energy_diagram.png': 4, 'summary.json': 20}
        visuals = sorted((p for p in current if _artifact_kind(p)),
                         key=lambda p: (_preview_priority.get(os.path.basename(p).lower(), 10), p))
        summaries = sorted((p for p in current if os.path.basename(p) == 'summary.json'),
                           key=lambda p: p.count(os.sep))
        sh = _summary_html(summaries[0] if summaries else None)
        if sh: display(W.HTML(sh))
        if current and not visuals:
            print('This workflow produced no previewable artifact; inspect status and files above.')
    cand = sorted((p for p in current if Path(p).suffix.lower() == '.xyz' and 'trj' in os.path.basename(p).lower()),
                  key=lambda p: ('mep_trj' not in os.path.basename(p).lower(),
                                 'finished_irc_trj' not in os.path.basename(p).lower(), p))
    _result_pick_guard['active'] = True
    try:
        artifact_choice.options = [('%s · %s' % (_artifact_kind(p), os.path.relpath(p, out)), p)
                                   for p in visuals]
        artifact_choice.disabled = not visuals
        if visuals: artifact_choice.value = visuals[0]
        traj_choice.options = [(os.path.relpath(p, out), p) for p in cand]
        traj_choice.disabled = not cand
        if cand: traj_choice.value = cand[0]
    finally:
        _result_pick_guard['active'] = False
    artifact_box.layout.display = '' if visuals else 'none'
    _render_artifact()
    _load_trajectory(cand[0] if cand else None, out)
    if not cand:
        results_empty.layout.display = ''
        state = manifest.get('status')
        results_empty.value = ('<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">%s</div>' %
                               ('This result has no XYZ trajectory. Inspect its status and files above.' if current else
                                'The command failed before producing a trajectory; inspect the run log.' if state == 'failed' else
                                'The command was cancelled before producing a trajectory.' if state == 'cancelled' else
                                'Run a workflow to inspect its current trajectory here.'))
        traj_label.value = '<small>no XYZ trajectory found for this result</small>'
    res_btn.description = 'Refresh results'
    dl_btn.disabled = not bool(current or manifest or S.get('_last_log'))
res_btn = W.Button(description='Show results', icon='bar-chart', layout=W.Layout(width='200px'))
res_btn.on_click(lambda _: _results(S.get('_last_out_dir') or _effective_result_root()))
dl_btn = W.Button(description='Download results / diagnostics (.zip)', icon='download', disabled=True,
                  tooltip='Current-run files plus the command manifest and captured log; available after failures too.',
                  layout=W.Layout(width='300px', max_width='100%'))
def _download(_):
    out = S.get('_last_out_dir') or _effective_result_root()
    current = _current_run_files(out)
    manifest = S.get('_last_manifest') or {}
    transcript = S.get('_last_log') or ''
    if not (current or manifest or transcript):
        with res_out: clear_output(); print('No current run is available to bundle.'); return
    stem = Path(os.path.abspath(out)).name or 'results'
    z = _unique_path(_runtime_path('downloads', stem + '_current_run.zip'))
    with zipfile.ZipFile(z, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        external_index = 0
        for path in current:
            rel = os.path.relpath(path, out)
            if rel.startswith('..'):
                external_index += 1
                arcname = 'other_outputs/%02d_%s' % (external_index, os.path.basename(path))
            else:
                arcname = rel
            archive.write(path, arcname)
        archive.writestr('colab_run.json', json.dumps(manifest, indent=2))
        if transcript: archive.writestr('colab_run.log', transcript)
    try:
        from google.colab import files as _f; _f.download(z)
    except Exception as e:
        with res_out: print('Download needs Colab; zip saved at', z, '(', e, ')')
dl_btn.on_click(_download)
results_empty = W.HTML(
    '<div role="status" style="padding:10px;border:1px dashed #64748b;border-radius:10px;">'
    'Run a workflow, then choose <b>Show results</b>.</div>')
artifact_box = W.VBox([artifact_choice, artifact_out])
artifact_box.layout.display = 'none'
trajectory_box = W.VBox([
    trajectory_intro, traj_choice, traj_label, frame_state, frame_slider,
    W.HBox([traj_out, plot_out], layout=W.Layout(flex_flow='row wrap'))])
trajectory_box.layout.display = 'none'
trajectory_box.add_class('rxresults')
results_box = W.VBox([
    W.HBox([res_btn, dl_btn], layout=W.Layout(flex_flow='row wrap')),
    result_context, res_out, artifact_box, results_empty, trajectory_box])
results_box.add_class('rxresults')

# ============================================================== command line + run (bottom, PyMOL-style)
logbox = W.Output(layout={'border': '1px solid #ccc', 'max_height': '380px', 'overflow': 'auto'})
def _stream(cmd):
    command_line = '$ ' + ' '.join(shlex.quote(c) for c in cmd) + '\n\n'
    transcript = [command_line]
    with logbox:
        clear_output(); print(command_line, end='')
        try:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                                 start_new_session=(os.name == 'posix'))
        except OSError as exc:
            line = '[launch failed] %s\n[exit 127]\n' % exc
            transcript.append(line); print(line, end='')
            return 127, ''.join(transcript)
        try:
            for line in p.stdout:
                transcript.append(line); print(line, end='')
            p.wait()
        except KeyboardInterrupt:
            line = '\n[interrupt] stopping the child process …\n'
            transcript.append(line); print(line, end='')
            try:
                if os.name == 'posix': os.killpg(os.getpgid(p.pid), signal.SIGTERM)
                else: p.terminate()
            except ProcessLookupError:
                pass
            try: p.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try:
                    if os.name == 'posix': os.killpg(os.getpgid(p.pid), signal.SIGKILL)
                    else: p.kill()
                except ProcessLookupError:
                    pass
                p.wait()
            transcript.append('[exit 130 · cancelled]\n'); print('[exit 130 · cancelled]')
            return 130, ''.join(transcript)
        finally:
            if p.stdout is not None: p.stdout.close()
        trailer = '\n[exit %d]\n' % p.returncode
        transcript.append(trailer); print(trailer, end='')
    return p.returncode, ''.join(transcript)
def _argv():
    line = cmd_box.value.strip()
    if not line or line.startswith('#'):
        with logbox: clear_output(); print('No command yet — load an input (Input tab).')
        return None
    try:
        return shlex.split(line)   # execute exactly what the editable command line shows
    except ValueError as exc:
        with logbox: clear_output(); print('Invalid command line:', exc)
        _set_run_status('✗ invalid command', 'error', 'invalid')
        return None

def _normalized_scope_argv(argv):
    """Apply this root CLI's argv normalization for output classification."""
    args = list(argv or [])
    if len(args) < 2: return args
    try:
        from pdb2reaction.cli.bool_compat import normalize_argv_option_names
        args = [args[0], *normalize_argv_option_names(args[1:])]
    except Exception:
        pass
    first = args[1]
    if first in ('-h', '--help', '--version'): return args
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        if root_cli.get_command(root_ctx, first) is None:
            top_level = set()
            for param in root_cli.params:
                top_level.update(getattr(param, 'opts', ()) or ())
                top_level.update(getattr(param, 'secondary_opts', ()) or ())
            if first.startswith('--'):
                is_top_level = first.split('=', 1)[0] in top_level
            elif first.startswith('-') and len(first) >= 2:
                is_top_level = first[:2] in top_level
            else:
                is_top_level = False
            if first.startswith('-') and not is_top_level:
                args.insert(1, 'all')
        if len(args) > 1 and not args[1].startswith('-'):
            command_name = args[1]
            value_opts, toggle_opts, negative_aliases, single_flags = (
                root_cli._resolve_bool_options(root_ctx, command_name))
            normalized, _legacy = root_cli._normalize_bool_argv(
                args[1:], {command_name: value_opts},
                {command_name: toggle_opts}, {command_name: negative_aliases},
                {command_name: single_flags})
            args = [args[0], *normalized]
    except Exception:
        pass
    return args

def _effective_out_dir(argv=None):
    """Return the last output path in the editable command, with GUI fallback."""
    if argv is None:
        line = cmd_box.value.strip()
        try: argv = shlex.split(line) if line and not line.startswith('#') else []
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    out = None
    flags = ('-o', '--out', '--out-dir', '--output')
    for i, token in enumerate(argv):
        if token in flags and i + 1 < len(argv):
            out = argv[i + 1]
        elif token.startswith('-o') and not token.startswith('--') and token != '-o':
            out = token[2:]
        else:
            for flag_name in flags:
                if token.startswith(flag_name + '='):
                    out = token.split('=', 1)[1]
    return out or _click_output_default(argv) or S.get('out_dir') or 'result'

def _click_subcommand_params(argv):
    """Parse the pinned command without invoking callbacks or validating files."""
    try:
        argv = _normalized_scope_argv(argv)
        import click
        from pdb2reaction.cli import cli as root_cli
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        if command is None: return {}
        sub_ctx = click.Context(command, info_name=argv[1], parent=root_ctx,
                                resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(sub_ctx).parse_args(
            args=list(argv[2:]))
        return parsed
    except Exception:
        return {}

def _click_output_default(argv):
    """Read the selected command's own output-option default from Click."""
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        argv = _normalized_scope_argv(argv)
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, argv[1])
        output_flags = {'-o', '--out', '--out-dir', '--output'}
        for param in command.params if command is not None else ():
            if output_flags.intersection(getattr(param, 'opts', ()) or ()):
                value = getattr(param, 'default', None)
                if isinstance(value, (str, os.PathLike)) and str(value):
                    return str(value)
    except Exception:
        pass
    return None

def _trj2fig_output_targets(argv):
    """Parse every trj2fig output accepted by this pinned CLI release."""
    parsed = _click_subcommand_params(argv)
    outs = parsed.get('outs')
    extra_outs = parsed.get('extra_outs')
    parsed_outputs = (list(outs) if isinstance(outs, (list, tuple)) else [])
    parsed_outputs += (list(extra_outs) if isinstance(extra_outs, (list, tuple)) else [])
    if parsed_outputs:
        return list(dict.fromkeys(str(path) for path in parsed_outputs))
    value_flags = {'-v', '--verbose', '-i', '--input', '--unit', '-r', '--reference',
                   '-q', '--charge', '-m', '--multiplicity', '-b', '--backend',
                   '--solvent', '--solvent-model', '--backend-model', '--precision'}
    output_exts = {'.png', '.jpg', '.jpeg', '.html', '.svg', '.pdf', '.csv'}
    flagged, positional = [], []
    args = list(argv[2:])
    positional_only = False
    i = 0
    while i < len(args):
        token = args[i]
        if token == '--':
            positional_only = True; i += 1; continue
        if not positional_only and token in ('-o', '--out'):
            if i + 1 < len(args): flagged.append(args[i + 1])
            i += 2; continue
        if not positional_only and token.startswith('--out='):
            flagged.append(token.split('=', 1)[1]); i += 1; continue
        if not positional_only and token.startswith('-o') and token != '-o':
            flagged.append(token[2:]); i += 1; continue
        if not positional_only and token in value_flags:
            i += 2; continue
        if not positional_only and any(token.startswith(flag + '=') for flag in value_flags if flag.startswith('--')):
            i += 1; continue
        if positional_only or (not token.startswith('-') and Path(token).suffix.lower() in output_exts):
            positional.append(token)
        i += 1
    found = flagged + positional
    if not found: found = ['energy.png']
    return list(dict.fromkeys(found))

def _flag_enabled(argv, positive, negative):
    enabled = False
    for token in argv:
        if token == positive: enabled = True
        elif token == negative: enabled = False
    return enabled

def _force_dry_run(argv):
    """Append the validating flag before a positional-only ``--`` marker."""
    args = list(argv)
    index = args.index('--') if '--' in args else len(args)
    args.insert(index, '--dry-run')
    return args

def _grouped_option_values(argv, flags, unique=True):
    """Collect repeated or space-grouped values for legacy variadic options."""
    found = []
    i = 2
    while i < len(argv):
        token = argv[i]
        if token in flags:
            i += 1
            while i < len(argv) and not argv[i].startswith('-'):
                found.append(argv[i]); i += 1
            continue
        matched = False
        for flag in flags:
            if flag.startswith('--') and token.startswith(flag + '='):
                found.append(token.split('=', 1)[1]); matched = True; break
        if not matched and '-o' in flags and token.startswith('-o') and token != '-o':
            found.append(token[2:]); matched = True
        i += 1
    return list(dict.fromkeys(found)) if unique else found

def _parsed_path(parsed, key):
    """Return one explicitly parsed CLI path, excluding Click sentinels."""
    value = parsed.get(key)
    return str(value) if isinstance(value, (str, os.PathLike)) and str(value) else None

def _input_needs_cif_companion(inputs):
    """Match the product's mmCIF/PDB-overflow normalization boundary."""
    for value in inputs:
        path = Path(value)
        if path.suffix.lower() in ('.cif', '.mmcif'): return True
        if path.suffix.lower() == '.pdb' and path.is_file():
            try:
                from pdb2reaction.io.structure_formats import pdb_requires_normalization
                if pdb_requires_normalization(path): return True
            except Exception:
                pass
    return False

def _exact_output_scope(outputs, include_json=False, companions=(), expand_user=False):
    """Build a current-run scope from the exact files a command may own."""
    def _absolute(path):
        value = os.path.expanduser(str(path)) if expand_user else str(path)
        return os.path.abspath(value)
    paths = [_absolute(path) for path in outputs]
    paths = list(dict.fromkeys(paths))
    root = os.path.dirname(paths[0]) or '.'
    tracked = list(paths)
    tracked.extend(_absolute(path) for path in companions)
    if include_json:
        tracked.extend((os.path.join(root, 'result.json'),
                        os.path.join(root, 'summary.json')))
    return {'target': paths[0], 'targets': paths, 'root': root,
            'shallow': True, 'prefix': None,
            'exact_targets': list(dict.fromkeys(tracked)),
            'direct_current': True}

def _output_scope(argv=None):
    """Resolve exact file targets or the directory owned by this argv."""
    if argv is None:
        try: argv = shlex.split(cmd_box.value.strip())
        except ValueError: argv = []
    argv = _normalized_scope_argv(argv)
    # Click's eager information flags exit after printing and never own output
    # files. Stop scanning at ``--`` so a positional token is not misread.
    visible = argv[1:argv.index('--')] if '--' in argv else argv[1:]
    if any(token in ('-h', '--help', '--help-advanced', '--version')
           for token in visible):
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    target = _effective_out_dir(argv)
    sub = argv[1] if len(argv) > 1 else S.get('subcmd')
    if sub == 'bond-summary':
        return {'target': 'standard output', 'targets': [], 'root': os.getcwd(),
                'shallow': True, 'prefix': None, 'exact_targets': [],
                'direct_current': True, 'stdout_only': True}
    if sub == 'trj2fig':
        outputs = [os.path.abspath(os.path.expanduser(path))
                   for path in _trj2fig_output_targets(argv)]
        return _exact_output_scope(
            outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
            expand_user=True)
    if sub == 'extract':
        raw_outputs = _grouped_option_values(argv, ('-o', '--output'), unique=False)
        inputs = _grouped_option_values(argv, ('-i', '--input'), unique=False)
        if len(inputs) == 1 and raw_outputs:
            raw_outputs = raw_outputs[:1]
        if not raw_outputs and inputs:
            raw_outputs = (['model.pdb'] if len(inputs) == 1 else
                           ['model_%s.pdb' % Path(path).stem for path in inputs])
        if raw_outputs:
            outputs = list(raw_outputs)
            companions = []
            if len(outputs) == len(inputs) and len(outputs) > 1:
                companions = [Path(output).with_suffix('.cif')
                              for output, input_path in zip(outputs, inputs)
                              if _input_needs_cif_companion([input_path])]
            elif outputs and inputs and _input_needs_cif_companion([inputs[0]]):
                companions = [Path(outputs[0]).with_suffix('.cif')]
            return _exact_output_scope(
                outputs, _flag_enabled(argv, '--out-json', '--no-out-json'),
                companions)
    parsed = _click_subcommand_params(argv)
    if sub == 'energy-diagram':
        output = _parsed_path(parsed, 'output_path') or 'energy_diagram.png'
        if not Path(output).suffix: output += '.png'
        return _exact_output_scope(
            [output], _flag_enabled(argv, '--out-json', '--no-out-json'))
    if sub == 'add-elem-info':
        input_path = _parsed_path(parsed, 'in_pdb')
        output = _parsed_path(parsed, 'out_pdb')
        if input_path and not output:
            source = Path(input_path)
            if _flag_enabled(argv, '--overwrite', '--no-overwrite'):
                output = str(source)
            elif source.name.lower().endswith('.pdb'):
                output = str(source.with_name(source.name[:-4] + '_add_elem.pdb'))
            else:
                output = str(source.with_name(source.name + '_add_elem.pdb'))
        if output: return _exact_output_scope([output])
    if sub == 'fix-altloc':
        input_path = _parsed_path(parsed, 'input_path')
        output = _parsed_path(parsed, 'out')
        inplace = _flag_enabled(argv, '--inplace', '--no-inplace')
        if input_path and os.path.isdir(input_path):
            source = os.path.abspath(input_path)
            owned = source if inplace else (
                os.path.abspath(output) if output else
                str(Path(source).with_name(Path(source).name + '_clean')))
            return {'target': owned, 'targets': [], 'root': owned,
                    'shallow': False, 'prefix': None, 'exact_targets': [],
                    'direct_current': False}
        if input_path:
            source = Path(input_path)
            if inplace:
                return _exact_output_scope(
                    [source, source.with_suffix(source.suffix + '.bak')])
            if output:
                candidate = Path(output)
                output = candidate if candidate.suffix.lower() == '.pdb' else candidate / source.name
            else:
                output = source.with_name(source.stem + '_clean.pdb')
            return _exact_output_scope([output])
    file_output_subs = {'extract', 'fix-altloc', 'add-elem-info', 'energy-diagram',
                        'trj2fig', 'oniom-export', 'oniom-import'}
    if sub in file_output_subs and Path(target).suffix:
        absolute = os.path.abspath(target)
        tracked = [absolute]
        if _flag_enabled(argv, '--out-json', '--no-out-json'):
            parent = os.path.dirname(absolute) or '.'
            tracked.extend((os.path.join(parent, 'result.json'),
                            os.path.join(parent, 'summary.json')))
        return {'target': target, 'targets': [absolute],
                'root': os.path.dirname(absolute) or '.', 'shallow': True,
                'prefix': None, 'exact_targets': tracked, 'direct_current': True}
    return {'target': target, 'targets': [], 'root': target, 'shallow': False,
            'prefix': None, 'exact_targets': [], 'direct_current': False}

def _effective_result_root(argv=None):
    """Resolve the primary directory Results should inspect for this argv."""
    return _output_scope(argv)['root']

def _matches_output_scope(path, scope):
    exact = scope.get('exact_targets') or []
    return not exact or os.path.abspath(path) in set(exact)

def _snapshot_files(root, shallow=False):
    root = os.path.abspath(root)
    if not os.path.isdir(root): return {}
    paths = glob.glob(os.path.join(root, '*')) if shallow else glob.glob(os.path.join(root, '**', '*'), recursive=True)
    snap = {}
    for path in paths:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _snapshot_output_scope(scope):
    if scope.get('stdout_only'): return {}
    exact = scope.get('exact_targets') or []
    if not exact: return _snapshot_files(scope['root'], shallow=scope['shallow'])
    snap = {}
    for path in exact:
        if not os.path.isfile(path): continue
        st = os.stat(path); snap[os.path.abspath(path)] = (st.st_size, st.st_mtime_ns)
    return snap

def _output_scope_collision(scope):
    if scope.get('stdout_only'): return False
    exact = scope.get('exact_targets') or []
    if exact: return any(os.path.lexists(path) for path in exact)
    root = scope['root']
    return os.path.isdir(root) and bool(os.listdir(root))

def _structured_current_paths(root, changed):
    """Use machine-readable claims when present; otherwise use the file delta."""
    root = os.path.abspath(root); claimed = set(); status_json = set(); has_claims = False
    changed_abs = {os.path.abspath(path) for path in changed}
    def _add(value, base=None):
        if not isinstance(value, str) or not value: return
        if os.path.isabs(value): candidates = [value]
        else:
            candidates = []
            if base is not None: candidates.append(os.path.join(base, value))
            candidates.extend((os.path.join(root, value), os.path.abspath(value)))
        candidates = list(dict.fromkeys(os.path.abspath(path) for path in candidates))
        path = next((path for path in candidates if path in changed_abs and os.path.isfile(path)), None)
        if path is None: path = next((path for path in candidates if os.path.isfile(path)), None)
        if path is not None: claimed.add(path)
    for path in list(changed):
        if os.path.basename(path) not in ('result.json', 'summary.json'): continue
        status_json.add(os.path.abspath(path))
        try:
            with open(path) as fh: payload = json.load(fh)
        except Exception:
            continue
        if 'current_output_paths' in payload:
            has_claims = True
            for value in payload.get('current_output_paths') or []: _add(value)
        if 'output_files' in payload:
            has_claims = True
            for value in payload.get('output_files') or []: _add(value)
        if 'files' in payload:
            has_claims = True
            values = payload.get('files') or {}
            values = values.values() if isinstance(values, dict) else values
            for value in values: _add(value)
        if 'key_output_files' in payload:
            has_claims = True
            for key, value in (payload.get('key_output_files') or {}).items():
                if isinstance(value, dict):
                    base = os.path.join(root, 'segments', key) if str(key).startswith('seg_') else root
                    for rel in value.get('files') or []: _add(rel, base)
                else:
                    _add(key)
    return sorted((claimed | status_json) if has_claims else set(changed))

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def _command_input_option_flags(argv):
    """Derive existing-file option arity from pdb2reaction's selected command."""
    fallback_multi = {
        '-i', '--input', '--ref-pdb', '--ref-full-pdb', '-s', '--scan-lists',
    }
    fallback_single = {'--config', '--calc-file', '--ref-mode', '--csv'}
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'):
            return fallback_multi, fallback_single
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None:
            return fallback_multi, fallback_single
        multi, single = set(), set()
        for param in command.params:
            value_type = getattr(param, 'type', None)
            flags = set(getattr(param, 'opts', ()) or ())
            flags.update(getattr(param, 'secondary_opts', ()) or ())
            # scan-lists deliberately accepts either an inline literal or a YAML file,
            # so Click exposes it as STRING; hash only values that resolve to real files.
            if flags.intersection({'-s', '--scan-lists'}):
                multi.update(flags)
                continue
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            if getattr(param, 'multiple', False) or getattr(param, 'nargs', 1) != 1:
                multi.update(flags)
            else:
                single.update(flags)
        return (multi, single) if (multi or single) else (fallback_multi, fallback_single)
    except Exception:
        return fallback_multi, fallback_single

def _click_parsed_input_files(argv):
    """Read option and positional input paths through Click's parser, without invoking."""
    args = _normalized_scope_argv(argv)
    try:
        import click
        from pdb2reaction.cli import cli as root_cli
        if len(args) < 2 or args[1].startswith('-'): return []
        root_ctx = click.Context(root_cli, info_name=CLI, resilient_parsing=True)
        command = root_cli.get_command(root_ctx, args[1])
        if command is None: return []
        command_ctx = click.Context(command, info_name=args[1], parent=root_ctx,
                                    resilient_parsing=True)
        parsed, _remaining, _order = command.make_parser(command_ctx).parse_args(args[2:])
        found = []
        for param in command.params:
            value_type = getattr(param, 'type', None)
            if not (isinstance(value_type, click.Path) and value_type.exists and value_type.file_okay):
                continue
            value = parsed.get(param.name)
            values = value if isinstance(value, (tuple, list)) else (value,)
            for path in values:
                if path is not None and os.path.isfile(os.fspath(path)):
                    found.append(os.path.abspath(os.fspath(path)))
        return found
    except Exception:
        return []

def _command_input_files(argv):
    argv = _normalized_scope_argv(argv)
    multi, single = _command_input_option_flags(argv)
    found = []
    i = 0
    while i < len(argv):
        token = argv[i]
        if token in multi:
            i += 1
            while i < len(argv) and not argv[i].startswith('-'):
                if os.path.isfile(argv[i]): found.append(os.path.abspath(argv[i]))
                i += 1
            continue
        if token in single and i + 1 < len(argv):
            if os.path.isfile(argv[i + 1]): found.append(os.path.abspath(argv[i + 1]))
            i += 2; continue
        for flag in multi | single:
            if token.startswith(flag + '=') and os.path.isfile(token.split('=', 1)[1]):
                found.append(os.path.abspath(token.split('=', 1)[1]))
            elif (flag.startswith('-') and not flag.startswith('--') and token != flag
                  and token.startswith(flag) and os.path.isfile(token[len(flag):])):
                found.append(os.path.abspath(token[len(flag):]))
        i += 1
    found.extend(_click_parsed_input_files(argv))
    return sorted(set(found))

def _validation_fingerprint(argv):
    files = []
    for path in _command_input_files(argv):
        try: digest = _sha256(path)
        except OSError: digest = '<missing>'
        files.append((os.path.abspath(path), digest))
    return (_command_fingerprint(cmd_box.value), tuple(files))
b_rebuild = W.Button(description='rebuild', icon='magic', layout=W.Layout(width='120px'),
                     tooltip='Regenerate the command from the current GUI selections.')
b_rebuild.on_click(lambda _: (_auto.__setitem__('on', True), refresh()))
b_copy = W.Button(description='copy', icon='copy', layout=W.Layout(width='100px'), tooltip='Copy the command line.')
def _copy(_):
    try:
        from google.colab import output as _o
        _o.eval_js('navigator.clipboard.writeText(%s)' % json.dumps(cmd_box.value))
        toast.value = '<small style="color:#1f7a3d">copied ✓</small>'
    except Exception as e:
        toast.value = '<small>copy needs Colab (%s)</small>' % e
b_copy.on_click(_copy)
b_clear = W.Button(description='clear selections', icon='eraser', layout=W.Layout(width='160px'))
def _clear_sel(_):
    if center_widget is not None: center_widget.value = ()
    if charge_rows is not None:
        for x in charge_rows.values(): x['use'].value = bool(x.get('auto'))
    S.update(center_ids=[], scan_atoms=[None, None], scan_preset='', scan_stages=[], scan_axes=[],
             freeze_buf=[None, None], freeze_pairs=[], freeze_atoms=[], measure_atoms=[],
             _last_pick=None, _last_pick_message='', _last_pick_tone='ok')
    _render_center_ids(); _render_scan_panel(); _render_freeze_panel(); _render_measure_panel()
    _render_last_pick_status()
    render_viewer(); refresh()
b_clear.on_click(_clear_sel)
b_validate = W.Button(description='Validate', button_style='info', icon='check', layout=W.Layout(width='130px'),
                      tooltip='Run the displayed compute command with --dry-run appended; no MLIP stage runs.')
b_run = W.Button(description='RUN exact command', button_style='danger', icon='play', layout=W.Layout(width='190px'),
                 tooltip='Execute exactly the command shown above. This may start a costly GPU calculation.')
def _set_running(value):
    for button in (b_validate, b_run, b_rebuild, b_clear): button.disabled = value
    cmd_box.disabled = value
def _do_validate(_):
    _sync_run(); a = _argv()
    if not a: return
    fingerprint = _validation_fingerprint(a)
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    if sub not in COMPUTE:
        with logbox: clear_output(); print('Validate is available for compute subcommands; use the command help for this utility.')
        _set_run_status('validation is for compute workflows', 'warn', 'utility')
        return
    # A final canonical flag wins over legacy ``--dry-run False`` and an
    # explicit ``--no-dry-run``.  Insert before ``--`` so Click parses it.
    a = _force_dry_run(a)
    _set_running(True)
    try:
        _set_run_status('🔎 validating…', 'info', 'validating')
        rc, validation_log = _stream(a)
        _RUN_STATE['validation_log'] = validation_log
        if rc == 0:
            _RUN_STATE['validated_fingerprint'] = fingerprint
            _set_run_status('✓ valid', 'ok', 'valid')
        else:
            _RUN_STATE['validated_fingerprint'] = None
            _set_run_status('✗ validation failed', 'error', 'invalid')
    finally:
        _set_running(False)
b_validate.on_click(_do_validate)
def _do_run(_):
    _sync_run(); a = _argv()
    if not a: return
    effective = _normalized_scope_argv(a)
    sub = effective[1] if len(effective) > 1 else ''
    validated = _RUN_STATE.get('validated_fingerprint')
    if sub in COMPUTE and validated and validated != _validation_fingerprint(a):
        _RUN_STATE['validated_fingerprint'] = None
        _set_run_status('validated inputs changed · validate again', 'warn', 'changed')
        with logbox: clear_output(); print('The command or an input file changed after validation. Validate this exact run again.')
        return
    if sub not in COMPUTE and sub != 'extract' and sub not in AUTOFILL_UTILS and _auto['on']:
        with logbox:
            clear_output(); print('This utility needs command-specific arguments.')
            print('Edit the command line using its --help, then run the exact edited command.')
        _set_run_status('◆ finish utility command', 'warn', 'utility')
        return
    scope = _output_scope(a)
    target = scope['target']; out = scope['root']
    collision = _output_scope_collision(scope)
    if collision and not w_reuse.value:
        _set_run_status('✗ output exists', 'error', 'collision')
        with logbox:
            clear_output(); print('Refusing to overwrite or mix an existing output:', target if scope['shallow'] else out)
            print('Choose a new out dir, or explicitly enable "reuse non-empty out dir" on the Workflow page.')
        _tab_go(1)
        return
    before = _snapshot_output_scope(scope)
    started = time.time()
    input_files = _command_input_files(a)
    real_run = not _flag_enabled(effective, '--dry-run', '--no-dry-run')
    if real_run:
        S.update(_last_out_dir=out, _last_subcmd=(sub or S.get('subcmd')),
                 _last_argv=list(a), _last_files=[], _last_log='',
                 _last_manifest={'tool': TOOL, 'subcommand': sub, 'argv': list(a),
                                 'status': 'running', 'output_root': out,
                                 'stdout_only': bool(scope.get('stdout_only'))})
        dl_btn.disabled = True
    _set_running(True)
    try:
        _RUN_STATE['validated_fingerprint'] = None
        _set_run_status('⏳ running…', 'warn', 'running')
        rc, transcript = _stream(a)
        if real_run: S['_last_log'] = transcript
        else: _RUN_STATE['validation_log'] = transcript
        if rc == 0: _set_run_status('✓ done', 'ok', 'done')
        elif rc == 130: _set_run_status('■ cancelled', 'warn', 'cancelled')
        else: _set_run_status('✗ failed (exit %d)' % rc, 'error', 'failed')
        if real_run:
            after = _snapshot_output_scope(scope)
            changed = sorted(path for path, stat in after.items()
                             if before.get(path) != stat and _matches_output_scope(path, scope))
            current = changed if scope.get('direct_current') else _structured_current_paths(out, changed)
            try:
                from importlib.metadata import version as _version
                package = 'pdb2reaction'
                installed = _version(package)
            except Exception:
                installed = None
            S['_last_out_dir'] = out
            S['_last_subcmd'] = sub or S.get('subcmd')
            S['_last_argv'] = list(a)
            S['_last_files'] = current
            S['_last_manifest'] = {
                'tool': TOOL, 'version': installed, 'subcommand': S['_last_subcmd'],
                'argv': list(a), 'started_unix': started, 'finished_unix': time.time(),
                'output_root': out, 'exit_code': rc,
                'stdout_only': bool(scope.get('stdout_only')),
                'status': 'success' if rc == 0 else ('cancelled' if rc == 130 else 'failed'),
                'inputs': [{'path': path, 'sha256': _sha256(path)} for path in input_files],
                'current_files': [os.path.relpath(path, out) for path in current],
            }
            _results(out); _tab_go(3)       # every real attempt replaces prior Results state
    finally:
        _set_running(False)
b_run.on_click(_do_run)
def _session_dict():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items()
                        if x['use'].value and not x.get('auto')}
    keys = ['tool', 'backend', 'model', 'subcmd', 'inputs', 'parm', 'model_pdb', 'mode', 'center', 'center_ids',
            'lcharge', 'scan_atoms', 'scan_stages', 'scan_axes', 'scan_target', 'scan_preset',
            'freeze_pairs', 'freeze_atoms', 'measure_atoms', 'charge', 'charge_explicit', 'tsopt', 'thermo',
            'out_dir', 'rep', 'color', 'advanced_overrides']
    d = {k: S.get(k) for k in keys}
    d['all_mode'] = _wv('all_mode', 'mep')
    d['advanced'] = {'mult': _wv('adv_mult', 1), 'precision': _wv('adv_prec', 'auto'),
                     'deterministic': _wv('adv_det', False), 'mep_mode': _wv('adv_mep', '(default)'),
                     'dmf_backend': _wv('adv_dmf', '(default)'),
                     'thresh': _wv('adv_thresh', '(default)'), 'radius': _wv('adv_radius', 0.0),
                     'flatten': _wv('adv_flatten', False), 'refine_path': _wv('adv_refine', False),
                     'max_cycles': _wv('adv_maxcyc', 0),
                     'dft': _wv('adv_dft', False), 'dft_func_basis': _wv('adv_dftfb', '')}
    return d
def _save_session(_):
    session_path = _runtime_path('session.json')
    with open(session_path, 'w') as fh: json.dump(_session_dict(), fh, indent=1)
    toast.value = '<small style="color:#1f7a3d">saved session.json</small>'
    try:
        from google.colab import files as _f; _f.download(session_path)
    except Exception:
        toast.value = '<small>saved session.json (working dir)</small>'   # not in Colab
def _apply_session(d):
    _clear_structure_bound_state()
    S.update(_last_pick=None, _last_pick_message='', _last_pick_tone='ok', _viewer_view=None,
             _view_input_index=0, _view_mapping_ok=True)
    for k in ('backend', 'model', 'subcmd', 'out_dir', 'tsopt', 'thermo', 'charge',
              'rep', 'color', 'scan_target', 'scan_preset', 'parm', 'model_pdb', 'mode'):
        if k in d: S[k] = d[k]
    S['inputs'] = d.get('inputs', S['inputs'])
    S['center'] = d.get('center', []); S['center_ids'] = d.get('center_ids', [])
    S['lcharge'] = d.get('lcharge', {}); S['scan_atoms'] = d.get('scan_atoms', [None, None])
    S['scan_stages'] = d.get('scan_stages', []); S['scan_axes'] = d.get('scan_axes', [])
    S['freeze_pairs'] = d.get('freeze_pairs', []); S['freeze_atoms'] = d.get('freeze_atoms', [])
    S['measure_atoms'] = d.get('measure_atoms', [])
    S['advanced_overrides'] = dict(d.get('advanced_overrides', {}))
    S['charge_explicit'] = bool(d.get('charge_explicit', False))
    adv = d.get('advanced', {})
    if S['backend'] in dd_backend.options: dd_backend.value = S['backend']
    dd_model.options = MODELS.get(dd_backend.value, dd_model.options)
    if S.get('model') in dd_model.options: dd_model.value = S['model']
    if S.get('subcmd') in SUBS:
        if S['subcmd'] not in dd_subcmd.options: cb_advsub.value = True
        dd_subcmd.value = S['subcmd']
    w_ts.value = bool(S.get('tsopt')); w_th.value = bool(S.get('thermo'))
    saved_all_mode = d.get('all_mode', 'mep')
    if saved_all_mode in ('mep', 'scan', 'tsonly'):
        all_mode.value = saved_all_mode
    w_out.value = S.get('out_dir', 'result')
    _saved_charge_explicit = S['charge_explicit']
    w_q.value = int(S.get('charge') or 0)
    S['charge_explicit'] = _saved_charge_explicit
    w_charge_ok.value = _saved_charge_explicit
    adv_mult.value = int(adv.get('mult', 1)); adv_prec.value = adv.get('precision', 'auto')
    adv_det.value = bool(adv.get('deterministic', False)); adv_mep.value = adv.get('mep_mode', '(default)')
    adv_dmf.value = adv.get('dmf_backend', '(default)')
    adv_thresh.value = adv.get('thresh', '(default)'); adv_radius.value = float(adv.get('radius', 0.0))
    adv_flatten.value = bool(adv.get('flatten', False)); adv_refine.value = bool(adv.get('refine_path', False))
    adv_maxcyc.value = int(adv.get('max_cycles', 0) or 0)
    adv_dft.value = bool(adv.get('dft', False)); adv_dftfb.value = adv.get('dft_func_basis', '')
    _render_advanced_rows()
    loaded_primary = False
    if S['mode'] == 'pdb' and S['inputs'] and os.path.exists(S['inputs'][0]):
        loaded_primary = build_selection()
    if loaded_primary:
        if center_widget is not None:
            center_widget.value = tuple(v for v in _center_values() if v in set(S['center']))
        if charge_rows is not None:
            for rn, x in charge_rows.items():
                x['use'].value = bool(x.get('auto')) or rn in S['lcharge']
                if x.get('auto'): x['val'].value = float(_ION_CHARGES[rn])
                elif rn in S['lcharge']: x['val'].value = float(S['lcharge'][rn])
    elif not S.get('_pdb_text'):
        render_viewer()
    _render_input_queue(); _sync_view_input_widget(); _render_freeze_panel(); _render_measure_panel(); _render_last_pick_status(); refresh()
    referenced = list(S.get('inputs', [])) + [p for p in (S.get('parm'), S.get('model_pdb')) if p]
    return [p for p in referenced if not os.path.isfile(p)]
up_sess = W.FileUpload(accept='.json', multiple=False, description='load session', layout=W.Layout(width='200px'))
def _on_load_sess(change):
    items = up_sess.value
    pairs = items.items() if isinstance(items, dict) else [(f['name'], f) for f in items]
    for name, meta in pairs:
        c = meta['content']
        try:
            text = c if isinstance(c, str) else bytes(c).decode('utf-8')
            missing = _apply_session(json.loads(text))
            toast.value = ('<small role="alert" style="color:#92400e">loaded settings; re-upload: %s</small>' %
                           html.escape(', '.join(os.path.basename(p) for p in missing)) if missing else
                           '<small style="color:#1f7a3d">loaded %s</small>' % html.escape(name))
        except Exception as e:
            toast.value = '<small style="color:#a00">load failed: %s</small>' % e
up_sess.observe(_on_load_sess, names='value')
b_save = W.Button(description='save settings', icon='save', layout=W.Layout(width='150px'),
                  tooltip='Save GUI settings only; input structures and topology files are not included.'); b_save.on_click(_save_session)
session_row = W.HBox([W.HTML('<small>settings (files excluded):</small>'), b_save, up_sess],
                     layout=W.Layout(flex_flow='row wrap'))
utility_bar = W.HBox([b_rebuild, b_copy, b_clear])
primary_bar = W.HBox([b_validate, b_run]); primary_bar.add_class('rxrun')
run_bar = W.HBox([utility_bar, primary_bar],
                 layout=W.Layout(width='100%', justify_content='space-between'))
interrupt_note = W.HTML('<small>To cancel a long run, use <b>Runtime → Interrupt execution</b>. '
                        'The GUI terminates and reaps the active child process.</small>')
command_more = _collapsible('More: session, cancellation & log',
                            W.VBox([interrupt_note, session_row, logbox]))
cmdline_box = W.VBox([
    W.HBox([W.HTML('<b>Command line</b>'),
            ready_chip, run_status, toast]),
    cmd_box, run_bar, command_more])

# ============================================================== assemble
# Colab's widget frontend renders ipywidgets' Tab as an empty block, so the tab
# strip is built from plain Buttons + a swapping VBox — widgets Colab draws
# reliably. _goto()/_tab_go() drive navigation.
_TAB_PAGES = [('1 Input', input_box), ('2 Workflow', options_box),
              ('3 Select', select_box), ('4 Results', results_box)]
_tab_body = W.VBox([_TAB_PAGES[0][1]])
_tab_status = W.HTML()
_tab_btns = []
def _tab_go(i):
    i = max(0, min(len(_TAB_PAGES) - 1, int(i)))
    _tab_body.children = [_TAB_PAGES[i][1]]
    _tab_status.value = ('<div role="status" aria-live="polite" aria-atomic="true" '
                         'style="color:#475569;font-size:12px;margin:0 2px 5px">'
                         'Step %d of %d · <b>%s</b></div>' %
                         (i + 1, len(_TAB_PAGES), _TAB_PAGES[i][0].split(' ', 1)[1]))
    for _j, _b in enumerate(_tab_btns):
        _b.button_style = 'primary' if _j == i else ''
for _i, (_t, _pane) in enumerate(_TAB_PAGES):
    _btn = W.Button(description=_t, layout=W.Layout(width='150px'))
    _btn.on_click(lambda _c, _k=_i: _tab_go(_k))
    _tab_btns.append(_btn)
_tab_go(0)
_tab_strip = W.HBox(_tab_btns, layout=W.Layout(flex_flow='row wrap'))
_tab_strip.add_class('rxtabs')
app = W.VBox([_tab_strip, _tab_status, _tab_body])
render_viewer()
refresh()
_t = 'pdb2reaction'
header = W.HTML(
    '<div style="background:#0f172a;border-radius:13px;padding:10px 15px;margin-bottom:5px;'
    'box-shadow:0 6px 20px rgba(15,23,42,0.18);">'
    '<div style="display:flex;align-items:center;flex-wrap:wrap;gap:10px;">'
    '<span style="font-size:21px;font-weight:700;letter-spacing:.2px;color:#f8fafc;">%s</span>'
    '<span style="background:#1e293b;color:#93c5fd;padding:3px 11px;border-radius:999px;'
    'font-size:11px;font-weight:600;letter-spacing:.3px;">REACTION-MECHANISM GUI</span></div>'
    '<div style="margin-top:6px;font-size:12.5px;color:#94a3b8;">backend '
    '<b style="color:#e2e8f0;">%s</b> · runs in <b style="color:#e2e8f0;">your own</b> Colab'
    '&nbsp;&nbsp;·&nbsp;&nbsp;<span style="color:#cbd5e1;">Input → Workflow → Select in 3D → Validate → Run</span></div></div>'
    % (_t, BACKEND))
rootbox = W.VBox([header, app, W.HTML('<hr style="margin:5px 0">'), cmdline_box])
rootbox.add_class('rxapp')
display(rootbox)

# --- Drag & drop: wire the .rxdrop zone to load files directly (Colab enhancement) ---
try:
    from google.colab import output as _dnd_out
    import base64 as _dnd_b64
    def _rxgui_drop(files):
        loaded = []; failed = []
        if sum(1 for _f in files if str(_f.get('name', '')).lower().endswith('.parm7')) > 1:
            input_msg.value = '<div role="alert" style="color:#991b1b">Drop one parm7 at a time so its structure pairing is explicit.</div>'
            return {'ok': False, 'n': 0, 'failed': ['multiple parm7 files']}
        for _f in files:
            _name = _f['name']
            if _f.get('error') or 'b64' not in _f:
                failed.append(_name); continue
            try:
                loaded.append(_save_upload(_name, _dnd_b64.b64decode(_f['b64'])))
            except Exception:
                failed.append(_name)
        accepted = _ingest_saved_files(loaded, 'drag & drop')
        if failed:
            input_msg.value += '<br><span style="color:#a00">read failed: %s</span>' % ', '.join(failed)
        return {'ok': bool(accepted) and not failed, 'n': len(loaded), 'failed': failed}
    _dnd_out.register_callback('rxgui.drop', _rxgui_drop)
    display(HTML(r"""<script>
(function(){
  function wire(){
    var box = document.querySelector('.rxdrop');
    if(!box) return false;
    if(box.dataset.rxDnd) return true;
    box.dataset.rxDnd = '1';
    var stop = function(e){ e.preventDefault(); e.stopPropagation(); };
    box.addEventListener('dragover', function(e){ stop(e); box.style.background = '#e0f2fe'; });
    box.addEventListener('dragleave', function(e){ stop(e); box.style.background = ''; });
    box.addEventListener('drop', function(e){
      stop(e); box.style.background = '';
      var fl = e.dataTransfer && e.dataTransfer.files; if(!fl || !fl.length) return;
      var out = new Array(fl.length), left = fl.length;
      for(var i=0;i<fl.length;i++){ (function(file, index){
        var r = new FileReader();
        r.onload = function(){
          out[index] = {name: file.name, b64: (r.result.split(',')[1] || '')};
          if(--left === 0){ google.colab.kernel.invokeFunction('rxgui.drop', [out], {}); }
        };
        r.onerror = function(){
          out[index] = {name: file.name, error: 'read failed'};
          if(--left === 0){ google.colab.kernel.invokeFunction('rxgui.drop', [out], {}); }
        };
        r.readAsDataURL(file);
      })(fl[i], i); }
    });
    return true;
  }
  if(!wire()){ var t = setInterval(function(){ if(wire()) clearInterval(t); }, 400);
               setTimeout(function(){ clearInterval(t); }, 20000); }
})();
</script>"""))
except Exception:
    pass  # drag & drop is a Colab-only enhancement; the upload button works everywhere
